# 35 — A third cluster-disjoint repeat, and the MAE_ONLY arm

**Cold-start summary.** Notebook `34` ran **2 repeats x 5 cluster-disjoint folds** because its
timing gate fired on an unrepresentative AID fold-0 (46.66 min against that arm's eventual 34.82
min mean), projecting 31.69 h against a 30 h threshold. The realised per-fold costs give 26.77 h,
which the same rule would have answered with **3 repeats**. This notebook adds **repeat 2** and
redoes `34`'s Part 4 analysis at **15 folds**.

**Adding a repeat is not re-deciding the gate.** The distinction matters and is the reason this
notebook is allowed to exist at all: a third repeat adds data **uniformly to every arm**, on a fold
assignment frozen before any result was seen (`data/folds/cv_folds_butina_5x5.csv` already carries
`butina_repeat_2`). It is not a search over outcomes, and no arm can be advantaged by it. Re-running
the *gate* on realised costs, by contrast, would be choosing a rule's input after seeing the rule's
output — the selection behaviour this project's `CLAUDE.md` warns against. To make the difference
checkable rather than asserted, **every analysis table below reports the 10-fold and the 15-fold
result side by side**, so a reader can see directly that nothing moved under selection.

**Second arm: MAE_ONLY.** `31` found that DEADZONE changes two things at once relative to PLAIN —
the loss (mse to mae) and the target (point estimate to OOF-clipped) — and isolated the loss-only
component with a MAE_ONLY arm on the random partition. That arm does not exist on this partition,
so `34`'s finding that DEADZONE beats PLAIN on all four isoforms cannot yet be decomposed here.
MAE_ONLY is run across **all three repeats (15 folds)**, paired with DEADZONE on the same units.

**Scope.** Evidence only. No submission is built, validated or sent; no full-data retrain; no
adoption recommended. `notebooks/34_butina_5x5_repeated_cv.ipynb` is never modified or re-executed
and `outputs/34_butina_5x5/` is read-only and protected. No fold file is written or rewritten.

In [1]:
import os

# THREAD ENVIRONMENT -- deliberately NOT set, and asserted unset, exactly as notebook 34 does.
# Notebook 34's Part 3 sanity check FAILED on its first execution (0.0378 max abs ST-RAE against
# notebook 05's stored repeat0_fold0 scores, versus a 0.03 threshold) purely because its setup
# cell had copied notebook 33's own OMP_NUM_THREADS=1. Diagnosed there rather than worked around:
# the training input CSV was byte-identical to 05's, the chemprop argv identical to 31's, and
# val_loss bit-identical at epochs 0 and 1 before diverging at epoch 2. Unsetting these two and
# changing nothing else reproduced 05 EXACTLY (0.00000000, bit-identical trajectory). Restricting
# BLAS/OMP to one thread changes floating-point reduction order and so the training trajectory.
for _v in ["OMP_NUM_THREADS", "KMP_DUPLICATE_LIB_OK"]:
    assert _v not in os.environ, (
        f"{_v} is set in this kernel's environment. It changes Chemprop's training trajectory and "
        f"breaks bit-reproducibility against notebook 05's stored scores -- see the comment above. "
        f"Unset it and restart the kernel before running this notebook.")

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import hashlib
import importlib.metadata
import inspect
import itertools
import json
import subprocess
import time
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import lightgbm as lgb

from src.features import assign_screen_split
from src.chemprop_screen import (
    build_predict_csv,
    build_training_csv,
    run_chemprop_predict,
    run_chemprop_train,
    setup_logging,
    verify_predictions,
)
from src.cv_bootstrap import per_fold_bootstrap_seed
from src.vendor.openadmet_eval.config import ACTIVITY_METRICS, REGRESSION_ENDPOINTS
from src.vendor.openadmet_eval.custom_scoring_functions import rae_soft_threshold_absolute_error
from src.vendor.openadmet_eval.evaluate_predictions import add_macro_endpoint, score_activity_predictions

print(f"python: {sys.version.split()[0]}")
for pkg in ["numpy", "pandas", "scipy", "matplotlib", "rdkit", "chemprop", "torch",
            "scikit-learn", "lightgbm", "xgboost"]:
    try:
        print(f"{pkg}: {importlib.metadata.version(pkg)}")
    except importlib.metadata.PackageNotFoundError:
        pass

METRIC_NAMES = [name for name, _ in ACTIVITY_METRICS]
ISOFORMS = [e.split("_")[0] for e in REGRESSION_ENDPOINTS]
PIC50_COL = {iso: f"{iso}_pIC50_direct_inhibition" for iso in ISOFORMS}
EP_TO_ISO = {f"{iso}_pIC50_direct_inhibition": iso for iso in ISOFORMS}
AUX_COLS = [f"{iso}_AID1851_max_inhibition" for iso in ISOFORMS]
PANEL_TO_ISOFORM = {"p450-cyp1a2": "CYP1A2", "p450-cyp2c9": "CYP2C9",
                    "p450-cyp2d6": "CYP2D6", "p450-cyp3a4": "CYP3A4"}

OUT = REPO_ROOT / "outputs" / "35_butina_repeat3"
FIG_OUT = OUT / "figures"
CHEMPROP_RUNS_DIR = OUT / "chemprop_runs"
PRED_DIR = OUT / "predictions"
SCORE_DIR = OUT / "scores"
OOF_DIR = OUT / "oof_long"
LOG_DIR = REPO_ROOT / "logs" / "35_butina_repeat3"
for d in [OUT, FIG_OUT, CHEMPROP_RUNS_DIR, PRED_DIR, SCORE_DIR, OOF_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# READ-ONLY paths. outputs/34_butina_5x5/ is read here and protected at the end -- notebook 34
# is never modified or re-executed.
FOLDS_PATH = REPO_ROOT / "data" / "folds" / "cv_folds.csv"
BUTINA5X5_PATH = REPO_ROOT / "data" / "folds" / "cv_folds_butina_5x5.csv"
CURATED_PATH = REPO_ROOT / "data" / "processed" / "train_inhibition_curated.csv"
BLIND_PATH = REPO_ROOT / "data" / "processed" / "test_blinded_curated.csv"
AID_CURATED_PATH = REPO_ROOT / "data" / "processed" / "aid1851_curated.csv"
AID_RAW_DIR = REPO_ROOT / "data" / "raw" / "aid1851"
ECFP4_NARROW_PATH = REPO_ROOT / "data" / "processed" / "tabular_baseline_features.csv"
MORDRED_PCA_PATH = REPO_ROOT / "data" / "processed" / "tabular_mordred_pca.csv"
NB05_OUT = REPO_ROOT / "outputs" / "05_cv_comparison"
NB24_OUT = REPO_ROOT / "outputs" / "24_nonnegative_stacking"
NB28_OUT = REPO_ROOT / "outputs" / "28_external_data"
NB29_OUT = REPO_ROOT / "outputs" / "29_cv_confirmation"
NB31_OUT = REPO_ROOT / "outputs" / "31_deadzone"
NB33_OUT = REPO_ROOT / "outputs" / "33_butina_split"
NB34_OUT = REPO_ROOT / "outputs" / "34_butina_5x5"
MANIFEST_PATH = NB05_OUT / "manifest.csv"

VAL_FRACTION = 0.15
EPOCHS = 50
PATIENCE = 5
CHEMELEON_ARCH_ARGS = ["--from-foundation", "CHEMELEON", "--multi-hot-atom-featurizer-mode", "V2"]
WINSOR_LOW_PCT, WINSOR_HIGH_PCT = 1.0, 99.0
XGB_LGBM_EARLY_STOPPING_ROUNDS = 10
N_FOLDS = 5
TOL = 1e-8
SANITY_THRESHOLD = 0.03

TARGET_REPEAT = 2               # the one new repeat of the main design
MAE_ONLY_REPEATS = [0, 1, 2]    # MAE_ONLY runs across all three, to pair with DEADZONE at 15 folds
NB34_REPEATS = [0, 1]           # what notebook 34 already completed, reused read-only
REPEATS_IN_SCOPE = sorted(set([TARGET_REPEAT] + MAE_ONLY_REPEATS))

_env_bin = str(Path(sys.executable).parent)
_parts = os.environ.get("PATH", "").split(os.pathsep)
if _env_bin not in _parts:
    os.environ["PATH"] = os.pathsep.join([_env_bin, *_parts])
print(f"\nchemprop: {subprocess.run(['which','chemprop'],capture_output=True,text=True).stdout.strip()}")
print(f"OUT: {OUT}")

# FRESH campaign timestamp, in this notebook's OWN directory. Notebook 34's campaign.json is
# never read: its wall guard measures from 26 Sep and would skip every unit here.
CAMPAIGN_PATH = OUT / "campaign.json"
assert CAMPAIGN_PATH != NB34_OUT / "campaign.json"
if CAMPAIGN_PATH.exists():
    CAMPAIGN = json.loads(CAMPAIGN_PATH.read_text())
    print(f"[resume] campaign started {CAMPAIGN['started_at_utc']}")
else:
    CAMPAIGN = {"started_at": time.time(),
                "started_at_utc": datetime.now(timezone.utc).isoformat()}
    CAMPAIGN_PATH.write_text(json.dumps(CAMPAIGN, indent=2))
    print(f"[new campaign] started {CAMPAIGN['started_at_utc']}")
RUN_STARTED_AT = float(CAMPAIGN["started_at"])

AUTONOMOUS_DECISIONS = []
ABORT = {"aborted": False, "reason": None, "where": None}


def record_decision(decision, reason, alternatives, authority):
    AUTONOMOUS_DECISIONS.append({"decision": decision, "reason": reason,
                                 "alternatives": alternatives, "authority": authority,
                                 "at_utc": datetime.now(timezone.utc).isoformat()})
    with open(OUT / "autonomous_decisions.json", "w") as fh:
        json.dump(AUTONOMOUS_DECISIONS, fh, indent=2)


def hard_fail(reason, where):
    ABORT.update({"aborted": True, "reason": reason, "where": where})
    with open(OUT / "abort.json", "w") as fh:
        json.dump(ABORT, fh, indent=2)
    print(f"*** RUN ABORTED *** at {where}: {reason}")

python: 3.11.13
numpy: 1.26.4
pandas: 2.3.3
scipy: 1.17.1
matplotlib: 3.11.1
rdkit: 2026.3.3
chemprop: 2.3.1
torch: 2.13.0
scikit-learn: 1.9.0
lightgbm: 4.7.0
xgboost: 3.2.0

chemprop: /Users/codiefreeman/miniconda3-arm64/envs/cyp-admet-v2/bin/chemprop
OUT: /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/35_butina_repeat3
[resume] campaign started 2026-09-30T22:01:34.619317+00:00


## Part 1 — The partition, reused unchanged

**No clustering is recomputed and no fold file is written.**
`data/folds/cv_folds_butina_5x5.csv` already carries all five repeats' cluster-to-fold assignments,
frozen by notebook `34` before any result was seen. Its sha256 is asserted against the value
notebook `34` recorded, which makes the file provably byte-identical to the one `34` ran on — so
`34`'s own verifications carry over rather than needing to be redone: 2,617 clusters from the
organisers' recipe, **0 clusters split across folds in any of the five repeats**, and no
(repeat, fold, isoform) cell below the 15% label-share flag.

Re-running the O(N^2) Butina clustering here would cost a minute and could only either reproduce
that or contradict a hash-identical file. The hash is the stronger check, so it is the one used.

In [2]:
NB34_FOLD_SHA256 = "52f39b1a1cea9200b58dce0a061d4a21ce94d9d58ec77067cf6508b0f4b3e87e"
sha = hashlib.sha256(BUTINA5X5_PATH.read_bytes()).hexdigest()
print(f"data/folds/cv_folds_butina_5x5.csv sha256: {sha}")
print(f"notebook 34 recorded:                      {NB34_FOLD_SHA256}")
assert sha == NB34_FOLD_SHA256, (
    "the cluster-disjoint fold file is NOT byte-identical to the one notebook 34 ran on -- "
    "every cross-repeat comparison below would be invalid. HARD FAILURE.")
print("MATCH -- the partition is provably the same file notebook 34 used.")

cv_folds_butina5 = pd.read_csv(BUTINA5X5_PATH)
REPEAT_COLS = [c for c in cv_folds_butina5.columns if c.startswith("butina_repeat_")]
assert len(REPEAT_COLS) == 5 and f"butina_repeat_{TARGET_REPEAT}" in REPEAT_COLS
print(f"\nrepeat columns present: {REPEAT_COLS}")

curated = pd.read_csv(CURATED_PATH)
blind = pd.read_csv(BLIND_PATH)
assert len(curated) == 4905 and len(blind) == 750
assert set(cv_folds_butina5["inchikey"]) == set(curated["inchikey"])
print(f"curated: {curated.shape}   blind: {blind.shape}")

# Per-fold, per-isoform labelled counts for the repeat this notebook adds -- reported, never
# reshuffled. (Repeats 0 and 1 were reported by notebook 34.)
rows = []
for f in range(N_FOLDS):
    sel = cv_folds_butina5[f"butina_repeat_{TARGET_REPEAT}"] == f
    ik = set(cv_folds_butina5.loc[sel, "inchikey"])
    sub = curated[curated["inchikey"].isin(ik)]
    row = {"repeat": TARGET_REPEAT, "fold": f, "n_compounds": int(sel.sum())}
    for iso in ISOFORMS:
        col = PIC50_COL[iso]
        n = int(sub[col].notna().sum())
        row[iso] = n
        row[f"{iso}_frac"] = n / int(curated[col].notna().sum())
    rows.append(row)
label_counts = pd.DataFrame(rows)
label_counts.to_csv(OUT / "fold_isoform_label_counts_repeat2.csv", index=False)
print(f"\nrepeat {TARGET_REPEAT} per-fold labelled-compound counts:")
print(label_counts[["fold", "n_compounds"] + ISOFORMS].to_string(index=False))
frac_cols = [f"{iso}_frac" for iso in ISOFORMS]
lo, hi = label_counts[frac_cols].to_numpy().min(), label_counts[frac_cols].to_numpy().max()
print(f"\nlabel-share range across (fold, isoform): {lo:.4f} to {hi:.4f}")
FLAG = 0.15
if (label_counts[frac_cols].to_numpy() < FLAG).any():
    print(f"*** DESIGN LIMITATION: a (fold, isoform) cell holds under {FLAG:.0%} of that isoform's "
          f"labels -- reported, not reshuffled.")
else:
    print(f"No (fold, isoform) cell falls below the {FLAG:.0%} flag threshold.")
assert (label_counts["n_compounds"] == 981).all(), "repeat-2 fold sizes are not the expected 981"
print("all five repeat-2 folds hold exactly 981 compounds, as notebook 34 recorded.")

data/folds/cv_folds_butina_5x5.csv sha256: 52f39b1a1cea9200b58dce0a061d4a21ce94d9d58ec77067cf6508b0f4b3e87e
notebook 34 recorded:                      52f39b1a1cea9200b58dce0a061d4a21ce94d9d58ec77067cf6508b0f4b3e87e
MATCH -- the partition is provably the same file notebook 34 used.

repeat columns present: ['butina_repeat_0', 'butina_repeat_1', 'butina_repeat_2', 'butina_repeat_3', 'butina_repeat_4']
curated: (4905, 20)   blind: (750, 4)

repeat 2 per-fold labelled-compound counts:
 fold  n_compounds  CYP1A2  CYP2C9  CYP2D6  CYP3A4
    0          981     281     259     293     472
    1          981     303     240     312     461
    2          981     281     262     303     448
    3          981     279     266     311     459
    4          981     268     258     274     495

label-share range across (fold, isoform): 0.1835 to 0.2146
No (fold, isoform) cell falls below the 15% flag threshold.
all five repeat-2 folds hold exactly 981 compounds, as notebook 34 recorded.


## Part 2 — Arms

The ten arms of notebook `34` (PLAIN, AID, SINGLE, six TREES configs, DEADZONE) plus one new arm:

| arm | what it is | rows | loss | target |
|---|---|---|---|---|
| `plain` | `chemprop_chemeleoninit`, multitask | 4,905 | mse | pIC50 point estimate |
| `aid` | plus notebook 28/29's four AID 1851 auxiliary heads | 21,364 | mse | point estimate |
| `single` | `chemprop_chemeleoninit` trained single-task, one model per isoform | per-isoform | mse | point estimate |
| `trees` | RF / XGBoost / LightGBM on `ecfp4_narrow` and `mordred_pca` | 4,905 | — | point estimate |
| `deadzone` | plain architecture, clipped target | 4,905 | **mae** | `clip(oof_pred, conf_low, conf_high)` |
| **`mae_only`** | **plain architecture, unchanged target** | **4,905** | **mae** | **pIC50 point estimate** |

**Why MAE_ONLY.** DEADZONE differs from PLAIN in two ways at once — the loss function and the
target — so notebook `34`'s finding that it beats PLAIN on all four isoforms cannot say how much of
that is the clipping and how much is simply fitting absolute error instead of squared error.
MAE_ONLY changes **only the loss**, holding the target at the literal curated label. Notebook `31`
ran this decomposition on the random partition and found the split varies sharply by isoform
(CYP2C9's gain almost entirely clipping; CYP2D6's almost entirely the loss switch). That arm does
not exist on this partition, so it is run here across all three repeats.

MAE_ONLY also has a property DEADZONE does not: **its target is the literal label**, so it carries
none of the fold-crosstalk concern that `31` raised and `34` inherited (a DEADZONE training
compound's target comes from an OOF donor model that had itself trained on the fold being held
out). Wherever MAE_ONLY and DEADZONE agree, the crosstalk caveat is not what is driving the result.

Every arm shares the fold assignment and the per-(repeat, fold) seed, so arm-vs-arm differences are
attributable to the arm alone.

In [3]:
# ---- AID table construction, reproduced verbatim from notebook 29 cells 6-7 (via notebook 33
# cell 16, which reproduced the same code). Not reimplemented.
aid_curated = pd.read_csv(AID_CURATED_PATH)
NB28_CACHE = NB28_OUT / "aid1851_max_response.csv"
NB33_CACHE = NB33_OUT / "aid1851_max_response.csv"
LOCAL_CACHE = OUT / "aid1851_max_response.csv"

if NB28_CACHE.exists():
    aid_eff = pd.read_csv(NB28_CACHE)
    print(f"[reusing notebook 28's cache, READ-ONLY] {NB28_CACHE} ({len(aid_eff)} rows)")
elif NB33_CACHE.exists():
    aid_eff = pd.read_csv(NB33_CACHE)
    print(f"[reusing notebook 33's cache, READ-ONLY] {NB33_CACHE} ({len(aid_eff)} rows)")
elif LOCAL_CACHE.exists():
    aid_eff = pd.read_csv(LOCAL_CACHE)
    print(f"[cached locally] {LOCAL_CACHE} ({len(aid_eff)} rows)")
else:
    from src.features import canonicalize_smiles, smiles_to_inchikey
    print("no cache found -- re-deriving Max_Response from the raw batches")
    frames = []
    for f in sorted(AID_RAW_DIR.glob("batch_*.csv")):
        df = pd.read_csv(f, low_memory=False)
        frames.append(df[df["PUBCHEM_SID"].notna()])
    raw = pd.concat(frames, ignore_index=True)
    for col in ["PUBCHEM_SID", "PUBCHEM_CID", "Fit_LogAC50", "Max_Response"]:
        raw[col] = pd.to_numeric(raw[col], errors="coerce")
    raw["isoform"] = raw["Panel Name"].map(PANEL_TO_ISOFORM)
    raw = raw[raw["isoform"].notna()].copy()
    raw = raw.dropna(subset=["PUBCHEM_CID", "PUBCHEM_EXT_DATASOURCE_SMILES"]).copy()
    uniq = raw[["PUBCHEM_CID", "PUBCHEM_EXT_DATASOURCE_SMILES"]].drop_duplicates("PUBCHEM_CID").copy()
    uniq["canonical_smiles"] = uniq["PUBCHEM_EXT_DATASOURCE_SMILES"].apply(canonicalize_smiles)
    uniq["inchikey"] = uniq["PUBCHEM_EXT_DATASOURCE_SMILES"].apply(smiles_to_inchikey)
    raw = raw.merge(uniq[["PUBCHEM_CID", "canonical_smiles", "inchikey"]], on="PUBCHEM_CID", how="left")
    raw = raw.dropna(subset=["inchikey"]).copy()
    aid_eff = raw.groupby(["inchikey", "isoform"], as_index=False).agg(
        max_response=("Max_Response", "mean"), n_replicates_eff=("Max_Response", "size"))
    aid_eff.to_csv(LOCAL_CACHE, index=False)
    print(f"wrote {LOCAL_CACHE}")

aid_eff["max_inhibition"] = -aid_eff["max_response"]

cur_counts = aid_curated["isoform"].value_counts().sort_index()
eff_counts = aid_eff["isoform"].value_counts().sort_index()
assert (cur_counts == eff_counts).all(), "re-derived population does not match aid1851_curated.csv"
assert set(aid_curated["inchikey"]) == set(aid_eff["inchikey"]), "compound sets differ"

aid = aid_curated.merge(aid_eff[["inchikey", "isoform", "max_response", "max_inhibition"]],
                        on=["inchikey", "isoform"], how="inner")
assert len(aid) == len(aid_curated), "join changed row count"

blind_ik = set(blind["inchikey"].dropna())
BLIND_OVERLAP = sorted(set(aid["inchikey"]) & blind_ik)
print(f"\\nAID compounds also in the blind 750 -- EXCLUDED ({len(BLIND_OVERLAP)}): {BLIND_OVERLAP}")
aid = aid[~aid["inchikey"].isin(blind_ik)].copy()
assert len(set(aid["inchikey"]) & blind_ik) == 0

aid_wide = aid.pivot(index="inchikey", columns="isoform", values="max_inhibition")
aid_wide.columns = [f"{c}_AID1851_max_inhibition" for c in aid_wide.columns]
aid_wide = aid_wide.reset_index().merge(
    aid[["inchikey", "canonical_smiles"]].drop_duplicates("inchikey"), on="inchikey", how="left")
aid_wide = aid_wide[["inchikey", "canonical_smiles"] + AUX_COLS]
assert aid_wide["inchikey"].is_unique

curated_ik = set(curated["inchikey"].dropna())
BASE_COLS = ["Molecule_Name", "inchikey", "canonical_smiles"]

curated_part = curated[BASE_COLS + list(REGRESSION_ENDPOINTS)].copy()
n_before = len(curated_part)
curated_part = curated_part.merge(aid_wide[["inchikey"] + AUX_COLS], on="inchikey", how="left")
assert len(curated_part) == n_before
n_attached = int(curated_part[AUX_COLS].notna().any(axis=1).sum())

extra_part = aid_wide[~aid_wide["inchikey"].isin(curated_ik)].copy()
extra_part["Molecule_Name"] = "AID1851_" + extra_part["inchikey"]
for c in REGRESSION_ENDPOINTS:
    extra_part[c] = np.nan
extra_part = extra_part[BASE_COLS + list(REGRESSION_ENDPOINTS) + AUX_COLS]

FULL_POP_RAW = pd.concat([curated_part, extra_part], ignore_index=True)
EXTRA_IK = set(extra_part["inchikey"])
assert FULL_POP_RAW["inchikey"].is_unique and FULL_POP_RAW["Molecule_Name"].is_unique
assert FULL_POP_RAW["canonical_smiles"].notna().all()

NB_ROWS, NB_ATTACHED, NB_EXTRA, NB_AUX_NONNULL = 21364, 98, 16459, 16557
checks = [("FULL_POP row count", len(FULL_POP_RAW), NB_ROWS),
          ("attached overlapping compounds", n_attached, NB_ATTACHED),
          ("AID-only extra compounds", len(extra_part), NB_EXTRA)]
for c in AUX_COLS:
    checks.append((f"non-null {c}", int(FULL_POP_RAW[c].notna().sum()), NB_AUX_NONNULL))
ok = True
for name, got, want in checks:
    good = got == want
    ok &= good
    print(f"  {name:45s} got={got:8d}  want={want:8d}  {'OK' if good else 'MISMATCH'}")
assert ok, "augmented table does not match notebooks 28/29's own row counts -- HARD FAILURE"
print("\\nCONFIRMED: augmented table matches notebooks 28/29/33's own construction exactly.")

[reusing notebook 28's cache, READ-ONLY] /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/28_external_data/aid1851_max_response.csv (66232 rows)
\nAID compounds also in the blind 750 -- EXCLUDED (1): ['RWZAIHHFBKOVIK-UHFFFAOYSA-N']
  FULL_POP row count                            got=   21364  want=   21364  OK
  attached overlapping compounds                got=      98  want=      98  OK
  AID-only extra compounds                      got=   16459  want=   16459  OK
  non-null CYP1A2_AID1851_max_inhibition        got=   16557  want=   16557  OK
  non-null CYP2C9_AID1851_max_inhibition        got=   16557  want=   16557  OK
  non-null CYP2D6_AID1851_max_inhibition        got=   16557  want=   16557  OK
  non-null CYP3A4_AID1851_max_inhibition        got=   16557  want=   16557  OK
\nCONFIRMED: augmented table matches notebooks 28/29/33's own construction exactly.


### Seeds — one per (repeat, fold), shared across arms

Taken from notebook `05`'s `manifest.csv`, mapped onto the Butina (repeat, fold) cells by index —
the same convention notebook `34` used and recorded, stated there and here as a convention rather
than a correspondence (the two partitions' fold indices carry no shared meaning). Because the seed
is held constant across arms within a fold, the mapping cancels out of every arm-vs-arm comparison.
Repeats 0 and 1 are loaded too, since MAE_ONLY runs on those folds as well.

In [4]:
manifest = pd.read_csv(MANIFEST_PATH)
mf = manifest[manifest["config"] == "chemprop_chemeleoninit"].sort_values(["repeat", "fold"])
assert len(mf) == 25, f"expected 25 chemprop_chemeleoninit manifest rows, got {len(mf)}"
FOLD_SEED = {(int(r.repeat), int(r.fold)): int(r.seed) for r in mf.itertuples()}
KNOWN_SEEDS_3 = [2684470948, 4091952314, 233227757]
assert [FOLD_SEED[(0, f)] for f in range(3)] == KNOWN_SEEDS_3, \
    "repeat_0's first three manifest seeds do not match this project's established SEEDS list"
print("confirmed: this project's established SEEDS = "
      f"{KNOWN_SEEDS_3} is exactly repeat_0's first three manifest fold seeds.")

# Cross-check the repeat-0/1 seeds against the ones notebook 34 actually recorded in its own
# run_scores.csv -- if the mapping had drifted, MAE_ONLY would not be paired with 34's arms.
nb34_scores = pd.read_csv(NB34_OUT / "run_scores.csv")
nb34_seed_map = (nb34_scores[["repeat", "fold", "seed"]].drop_duplicates()
                 .set_index(["repeat", "fold"])["seed"].to_dict())
mismatch = [(k, v, FOLD_SEED[k]) for k, v in nb34_seed_map.items() if FOLD_SEED[k] != v]
assert not mismatch, f"seed mapping differs from notebook 34's: {mismatch}"
print(f"confirmed: all {len(nb34_seed_map)} (repeat, fold) seeds match notebook 34's own record exactly.")
print("\n(repeat, fold) -> seed for the repeats this notebook runs:")
for r in sorted(set([TARGET_REPEAT] + MAE_ONLY_REPEATS)):
    print(f"  repeat {r}: " + "  ".join(f"f{f}={FOLD_SEED[(r,f)]}" for f in range(N_FOLDS)))

confirmed: this project's established SEEDS = [2684470948, 4091952314, 233227757] is exactly repeat_0's first three manifest fold seeds.
confirmed: all 10 (repeat, fold) seeds match notebook 34's own record exactly.

(repeat, fold) -> seed for the repeats this notebook runs:
  repeat 0: f0=2684470948  f1=4091952314  f2=233227757  f3=3276785861  f4=3644269654
  repeat 1: f0=1206282609  f1=3543069911  f2=3479688010  f3=877132087  f4=1244265337
  repeat 2: f0=2740186692  f1=3884972972  f2=829139639  f3=2545567356  f4=2718398101


### Per-fold winsorisation of the AID auxiliary columns

Reproduced from notebook `29`'s own correction, exactly as `34` applied it: bounds are recomputed
**per (repeat, fold)** on that fold's own training portion, never on the pooled population, so no
fold's held-out compounds reach its own clip thresholds.

In [5]:
def fold_test_inchikeys(repeat, fold):
    return set(cv_folds_butina5.loc[cv_folds_butina5[REPEAT_COLS[repeat]] == fold, "inchikey"])


def winsorise_for_fold(full_pop_raw, repeat, fold):
    test_ik = fold_test_inchikeys(repeat, fold)
    train_mask = ~full_pop_raw["inchikey"].isin(test_ik)
    pop = full_pop_raw.copy()
    bounds = {}
    for c in AUX_COLS:
        vals = pop.loc[train_mask, c].dropna()
        lo = float(np.percentile(vals, WINSOR_LOW_PCT))
        hi = float(np.percentile(vals, WINSOR_HIGH_PCT))
        bounds[c] = (lo, hi)
        pop[c] = pop[c].clip(lower=lo, upper=hi)
    return pop, bounds, int(train_mask.sum())


bound_rows = []
for r in REPEATS_IN_SCOPE:
    for f in range(N_FOLDS):
        _, bnds, n_train = winsorise_for_fold(FULL_POP_RAW, r, f)
        row = {"repeat": r, "fold": f, "n_train_portion": n_train}
        for c in AUX_COLS:
            row[f"{c.split('_')[0]}_low"] = bnds[c][0]
            row[f"{c.split('_')[0]}_high"] = bnds[c][1]
        bound_rows.append(row)
bounds_df = pd.DataFrame(bound_rows)
bounds_df.to_csv(OUT / "winsor_bounds_by_fold.csv", index=False)
print(f"winsorisation bounds across all {len(bounds_df)} (repeat, fold) cells:")
for iso in ISOFORMS:
    lo, hi = bounds_df[f"{iso}_low"], bounds_df[f"{iso}_high"]
    print(f"  {iso:9s} low [{lo.min():.3f}, {lo.max():.3f}] range={lo.max()-lo.min():.3f}   "
          f"high [{hi.min():.3f}, {hi.max():.3f}] range={hi.max()-hi.min():.3f}")

# Cross-check against notebook 34's own saved bounds for the repeats it ran. These are a pure
# function of the fold assignment and the AID table, both asserted identical above, so they must
# agree to floating-point tolerance -- a direct check that this notebook reconstructed 34's
# population rather than merely claiming to.
# Notebook 34 saved bounds for all five repeats of the partition (its loop covered the whole
# fold file); this notebook computes only the three it runs. Compare on the overlap.
nb34_bounds = pd.read_csv(NB34_OUT / "winsor_bounds_by_fold.csv")
nb34_in_scope = nb34_bounds[nb34_bounds["repeat"].isin(REPEATS_IN_SCOPE)]
shared = bounds_df.merge(nb34_in_scope, on=["repeat", "fold"], suffixes=("_new", "_nb34"))
assert len(shared) == len(nb34_in_scope) == len(bounds_df), (
    f"overlap mismatch: {len(shared)} shared vs {len(nb34_in_scope)} nb34 in scope "
    f"vs {len(bounds_df)} computed here")
worst = 0.0
for iso in ISOFORMS:
    for side in ["low", "high"]:
        d = float((shared[f"{iso}_{side}_new"] - shared[f"{iso}_{side}_nb34"]).abs().max())
        worst = max(worst, d)
print(f"\ncross-check vs notebook 34's own winsorisation bounds over the {len(shared)} shared cells: "
      f"max abs difference {worst:.2e}")
assert worst < TOL, "winsorisation bounds do not reproduce notebook 34's -- the population differs"
print("MATCH -- this notebook reconstructs notebook 34's AID population exactly.")

winsorisation bounds across all 15 (repeat, fold) cells:
  CYP1A2    low [-13.205, -13.190] range=0.015   high [116.239, 116.297] range=0.057
  CYP2C9    low [-184.353, -184.295] range=0.058   high [121.180, 121.184] range=0.004
  CYP2D6    low [-29.332, -29.328] range=0.003   high [100.335, 100.357] range=0.022
  CYP3A4    low [-75.063, -74.493] range=0.570   high [108.115, 108.135] range=0.020

cross-check vs notebook 34's own winsorisation bounds over the 15 shared cells: max abs difference 1.42e-14
MATCH -- this notebook reconstructs notebook 34's AID population exactly.


### Building the population per (repeat, fold, arm)

`assign_screen_split` (`src/features.py`, unmodified — nothing under `src/` is touched by this
notebook) is reused directly with `repeat_col="butina_repeat_{r}"`.

In [6]:
CHEMPROP_ARMS = ["plain", "aid", "single", "deadzone", "mae_only"]
TREE_ARMS = [f"{feat}__{algo}" for feat in ["ecfp4_narrow", "mordred_pca"]
             for algo in ["rf", "xgboost", "lightgbm"]]
# Arm order for reporting: notebook 34's ten, then the new MAE_ONLY arm.
NB34_ARMS = ["plain"] + TREE_ARMS + ["single", "aid", "deadzone"]
ALL_ARMS = NB34_ARMS + ["mae_only"]
print(f"notebook 34's arms ({len(NB34_ARMS)}): {NB34_ARMS}")
print(f"new here: ['mae_only'] -- {len(ALL_ARMS)} arms total")

LOSS_ARGS = {"plain": ["--loss-function", "mse"],
             "aid": ["--loss-function", "mse"],
             "single": ["--loss-function", "mse"],
             "deadzone": ["--loss-function", "mae"],
             "mae_only": ["--loss-function", "mae"]}
ARCH_ARGS = {a: CHEMELEON_ARCH_ARGS + LOSS_ARGS[a] for a in LOSS_ARGS}

# Confirm chemprop's mae loss really is plain absolute error against the installed API, rather
# than assuming it from the flag name -- notebook 31's own check, repeated here because MAE_ONLY
# depends on it entirely.
import chemprop.nn as cp_nn
_mae_src = inspect.getsource(cp_nn.LossFunctionRegistry["mae"]).replace(" ", "")
assert "(preds-targets).abs()" in _mae_src, "unexpected mae loss implementation -- stopping"
assert cp_nn.PredictorRegistry["regression"]._T_default_criterion is cp_nn.LossFunctionRegistry["mse"]
print("\nCONFIRMED against the installed chemprop: 'mae' computes (preds - targets).abs(); the")
print("regression default is mse. Both are passed explicitly rather than relied on implicitly.")


def build_population(full_pop, repeat, fold, seed, arm, curated_source=None):
    # `curated_source` supplies the 4 scored target columns; None means the real curated labels.
    # DEADZONE passes its own clipped-target table. MAE_ONLY passes None -- its target IS the
    # literal label, which is the whole point of the arm. The SPLIT never depends on it.
    split_df = assign_screen_split(cv_folds_butina5, repeat_col=REPEAT_COLS[repeat],
                                   test_fold=fold, val_fraction=VAL_FRACTION, seed=seed)
    use_aux = arm == "aid"
    cols = BASE_COLS + list(REGRESSION_ENDPOINTS) + (AUX_COLS if use_aux else [])
    src_pop = full_pop if use_aux else full_pop[full_pop["inchikey"].isin(curated_ik)]
    pop = src_pop[cols].copy()
    if curated_source is not None:
        pop = pop.drop(columns=list(REGRESSION_ENDPOINTS)).merge(
            curated_source[["inchikey"] + list(REGRESSION_ENDPOINTS)], on="inchikey", how="left")
        assert len(pop) == len(src_pop), "target-substitution merge changed row count"
    pop = pop.merge(split_df[["inchikey", "screen_split"]], on="inchikey", how="left")
    assert len(pop) == len(src_pop), "split merge changed row count"
    pop.loc[pop["screen_split"].isna(), "screen_split"] = "screen_inner_train"
    assert pop["screen_split"].isin(["screen_test", "screen_inner_train", "screen_inner_val"]).all()
    return pop


ARM_TARGET_COLS = {"plain": list(REGRESSION_ENDPOINTS),
                   "aid": list(REGRESSION_ENDPOINTS) + AUX_COLS,
                   "deadzone": list(REGRESSION_ENDPOINTS),
                   "mae_only": list(REGRESSION_ENDPOINTS)}

print(f"\nstructural checks on the split construction (repeat {TARGET_REPEAT}, folds 0/2/4):")
for f in [0, 2, 4]:
    seed = FOLD_SEED[(TARGET_REPEAT, f)]
    wp, _, _ = winsorise_for_fold(FULL_POP_RAW, TARGET_REPEAT, f)
    pa = build_population(wp, TARGET_REPEAT, f, seed, "aid")
    pb = build_population(FULL_POP_RAW, TARGET_REPEAT, f, seed, "plain")
    pm = build_population(FULL_POP_RAW, TARGET_REPEAT, f, seed, "mae_only")
    ta = set(pa.loc[pa["screen_split"] == "screen_test", "Molecule_Name"])
    tb = set(pb.loc[pb["screen_split"] == "screen_test", "Molecule_Name"])
    tm = set(pm.loc[pm["screen_split"] == "screen_test", "Molecule_Name"])
    assert ta == tb == tm, f"held-out sets differ between arms at fold {f}"
    held_ik = set(pa.loc[pa["screen_split"] == "screen_test", "inchikey"])
    va = set(pa.loc[pa["screen_split"] == "screen_inner_val", "inchikey"])
    assert held_ik.isdisjoint(EXTRA_IK), f"an AID-only compound was held out at fold {f}"
    assert va.isdisjoint(EXTRA_IK), f"an AID-only compound is in the validation set at fold {f}"
    assert held_ik == fold_test_inchikeys(TARGET_REPEAT, f), \
        "held-out set is not this fold's own cluster block"
    # MAE_ONLY must be PLAIN's population exactly -- same rows, same target values.
    assert pm[list(REGRESSION_ENDPOINTS)].equals(pb[list(REGRESSION_ENDPOINTS)]), \
        f"mae_only target values differ from plain at fold {f} -- it must differ only in loss"
    print(f"  fold {f} (seed {seed}): held-out identical across arms ({len(ta)} compounds); "
          f"mae_only targets bit-identical to plain")
print("\nCONFIRMED: held-out sets identical across arms; AID-only extras never held out or "
      "validated on; MAE_ONLY differs from PLAIN in loss function only.")

notebook 34's arms (10): ['plain', 'ecfp4_narrow__rf', 'ecfp4_narrow__xgboost', 'ecfp4_narrow__lightgbm', 'mordred_pca__rf', 'mordred_pca__xgboost', 'mordred_pca__lightgbm', 'single', 'aid', 'deadzone']
new here: ['mae_only'] -- 11 arms total



CONFIRMED against the installed chemprop: 'mae' computes (preds - targets).abs(); the
regression default is mse. Both are passed explicitly rather than relied on implicitly.

structural checks on the split construction (repeat 2, folds 0/2/4):
  fold 0 (seed 2740186692): held-out identical across arms (981 compounds); mae_only targets bit-identical to plain
  fold 2 (seed 829139639): held-out identical across arms (981 compounds); mae_only targets bit-identical to plain
  fold 4 (seed 2718398101): held-out identical across arms (981 compounds); mae_only targets bit-identical to plain

CONFIRMED: held-out sets identical across arms; AID-only extras never held out or validated on; MAE_ONLY differs from PLAIN in loss function only.


### TREES — feature matrices

`load_feature_matrix` reproduced verbatim from `scripts/run_5x5_cv_comparison.py` (the same reuse
notebook `05c` made), so features, the `split == "train"` filter and index alignment are this
project's own, unchanged. CheMeleon embeddings are deliberately not loaded — notebook `33`
established a head downstream of the same representation cannot decorrelate from Chemprop, which
is the question TREES exists to test.

In [7]:
FEATURE_FILES = {"ecfp4_narrow": ECFP4_NARROW_PATH, "mordred_pca": MORDRED_PCA_PATH}


def load_feature_matrix(name):
    # Reproduced verbatim from scripts/run_5x5_cv_comparison.py's own load_feature_matrix
    # (CSV branch only -- the .npy/CheMeleon branch is deliberately unused here).
    path = FEATURE_FILES[name]
    df = pd.read_csv(path)
    if "split" in df.columns:
        df = df[df["split"] == "train"].reset_index(drop=True)
    index_df = df[["Molecule_Name", "inchikey"]].reset_index(drop=True)
    drop_cols = [c for c in ["Molecule_Name", "inchikey", "split"] if c in df.columns]
    X = df.drop(columns=drop_cols).to_numpy(dtype=np.float64)
    return index_df, X


FEATURES = {}
for name in FEATURE_FILES:
    idx, X = load_feature_matrix(name)
    FEATURES[name] = (idx, X)
    assert len(idx) == 4905, f"{name}: expected 4905 training rows, got {len(idx)}"
    assert set(idx["inchikey"]) == curated_ik, f"{name}: compound set does not match curated"
    print(f"{name:14s} X={X.shape}  finite={np.isfinite(X).all()}")


def run_trees_one(config, repeat, fold, seed):
    # Reproduced from scripts/run_5x5_cv_comparison.py's run_tabular_or_naive: RF fits on the
    # pooled inner train+val (no early stopping concept); XGBoost/LightGBM fit on inner-train
    # with inner-val as the early-stopping eval set. One fit per endpoint, as there.
    feature_name, algo = config.split("__")
    feat_index, X = FEATURES[feature_name]
    population = build_population(FULL_POP_RAW, repeat, fold, seed, "plain")
    pop_pos = feat_index.reset_index().merge(population, on=["Molecule_Name", "inchikey"], how="left")
    assert pop_pos["screen_split"].notna().all(), f"{config}: feature index did not align 1:1"

    pool_mask = pop_pos["screen_split"].isin(["screen_inner_train", "screen_inner_val"])
    inner_train_mask = pop_pos["screen_split"] == "screen_inner_train"
    inner_val_mask = pop_pos["screen_split"] == "screen_inner_val"
    test_sel = pop_pos["screen_split"] == "screen_test"
    test_positions = pop_pos.loc[test_sel, "index"].to_numpy()

    pred_df = pop_pos.loc[test_sel, ["Molecule_Name", "inchikey"]].reset_index(drop=True)
    for endpoint in REGRESSION_ENDPOINTS:
        has_label = pop_pos[endpoint].notna()
        if algo == "rf":
            idx = pop_pos.loc[pool_mask & has_label, "index"].to_numpy()
            y = pop_pos.loc[pool_mask & has_label, endpoint].to_numpy()
            model = RandomForestRegressor(random_state=seed)
            model.fit(X[idx], y)
        else:
            tr_idx = pop_pos.loc[inner_train_mask & has_label, "index"].to_numpy()
            va_idx = pop_pos.loc[inner_val_mask & has_label, "index"].to_numpy()
            y_tr = pop_pos.loc[inner_train_mask & has_label, endpoint].to_numpy()
            y_va = pop_pos.loc[inner_val_mask & has_label, endpoint].to_numpy()
            if algo == "xgboost":
                model = XGBRegressor(early_stopping_rounds=XGB_LGBM_EARLY_STOPPING_ROUNDS,
                                     random_state=seed)
                model.fit(X[tr_idx], y_tr, eval_set=[(X[va_idx], y_va)], verbose=False)
            else:
                model = LGBMRegressor(random_state=seed, verbosity=-1)
                model.fit(X[tr_idx], y_tr, eval_set=[(X[va_idx], y_va)],
                          callbacks=[lgb.early_stopping(stopping_rounds=XGB_LGBM_EARLY_STOPPING_ROUNDS,
                                                        verbose=False)])
        pred_df[endpoint] = np.asarray(model.predict(X[test_positions])).reshape(-1)
    return pred_df

ecfp4_narrow   X=(4905, 2057)  finite=True
mordred_pca    X=(4905, 132)  finite=True


## Part 3 — Execution

**Order: the new repeat's main design first, MAE_ONLY second.** Within repeat 2 the arm order
matches notebook `34` exactly — PLAIN, TREES, SINGLE, AID, DEADZONE — with DEADZONE last because
its target must be built from PLAIN's completed repeat-2 out-of-fold predictions **on this
partition**. If PLAIN's repeat 2 is incomplete the DEADZONE arm is skipped and recorded; falling
back to `outputs/11_caruana_prep/`, which lives on the random partition, is never done.

MAE_ONLY runs afterwards across all three repeats. Putting it second is deliberate: if the wall
guard bites, it costs the decomposition arm rather than leaving the third repeat half-finished,
and repeat 2 either completes as a whole or does not.

**One run per cell**, each with an on-disk cache-skip plus content check, so a per-cell timeout
loses at most one unit and re-executing resumes rather than restarts. Scores are appended after
every single run. `epochs=50`, `patience=5` set explicitly; `src/chemprop_screen.py` reused
unmodified.

In [8]:
RUN_SCORES_PATH = OUT / "run_scores.csv"
TIMINGS_PATH = OUT / "run_timings.csv"

# NO TIMING GATE HERE. Notebook 34 needed one because its design size was the open question;
# this notebook's design is fixed and small -- exactly one new repeat of the main arms (25 units)
# plus MAE_ONLY across three repeats (15 units) -- and is projected from notebook 34's own
# REALISED per-fold means rather than a single fold: plain 7.23, trees 1.15, single 11.35,
# aid 34.82, deadzone 9.70 min/fold, so repeat 2 costs about 5.35 h, and MAE_ONLY about 8 min a
# fold for another 2 h. Roughly 7.4 h against the 12 h guard below.
DESIGN = {"target_repeat": TARGET_REPEAT, "mae_only_repeats": MAE_ONLY_REPEATS,
          "n_folds": N_FOLDS, "started_at_utc": CAMPAIGN["started_at_utc"]}
(OUT / "design.json").write_text(json.dumps(DESIGN, indent=2))
print(f"design: repeat {TARGET_REPEAT} of the main arms + MAE_ONLY on repeats {MAE_ONLY_REPEATS}")

# Wall-clock guard measured from THIS campaign's own start (never notebook 34's, whose timestamp
# is from 26 Sep and would skip every unit). Skipped units are recorded, never silently dropped.
# Ordering puts the main design's repeat 2 first and MAE_ONLY second, so if the guard bites it
# costs the decomposition arm rather than leaving the third repeat half-finished.
WALL_DEADLINE_H = 12.0
DEADLINE_SKIPS = []


def _deadline_exceeded():
    return (time.time() - RUN_STARTED_AT) / 3600.0 > WALL_DEADLINE_H


def _last_epoch(tag):
    m = CHEMPROP_RUNS_DIR / tag / "model_0" / "trainer_logs" / "version_0" / "metrics.csv"
    if not m.exists():
        return None
    eps = [int(r.split(",")[0]) for r in m.read_text().splitlines()[1:]
           if r and r.split(",")[0].isdigit()]
    return max(eps) if eps else None


def _append_rows(path, new_rows, key_cols):
    new_df = pd.DataFrame(new_rows)
    if path.exists():
        old = pd.read_csv(path)
        keys = set(map(tuple, new_df[key_cols].values.tolist()))
        mask = ~old[key_cols].apply(lambda r: tuple(r) in keys, axis=1)
        combined = pd.concat([old[mask], new_df], ignore_index=True)
    else:
        combined = new_df
    combined = combined.sort_values(key_cols).reset_index(drop=True)
    combined.to_csv(path, index=False)
    return combined


def score_and_append(repeat, fold, arm, seed, pred_df):
    ground_truth = curated[curated["inchikey"].isin(pred_df["inchikey"])].copy()
    with per_fold_bootstrap_seed(seed):
        scored = score_activity_predictions(pred_df, ground_truth, list(REGRESSION_ENDPOINTS))
        scored = add_macro_endpoint(scored, list(REGRESSION_ENDPOINTS), ACTIVITY_METRICS)
    scored.to_csv(SCORE_DIR / f"{arm}__r{repeat}f{fold}.csv", index=False)
    pe = scored.groupby("Endpoint")[METRIC_NAMES].mean()
    rows = []
    for endpoint in pe.index:
        row = {"repeat": repeat, "fold": fold, "arm": arm, "seed": seed, "Endpoint": endpoint}
        row.update(pe.loc[endpoint].to_dict())
        rows.append(row)
    _append_rows(RUN_SCORES_PATH, rows, ["repeat", "fold", "arm", "Endpoint"])
    return pe


def save_oof_long(repeat, fold, arm, pred_df):
    # One file per run, overwritten idempotently (never appended to), so a resume can never
    # duplicate rows. The per-isoform WIDE tables matching
    # outputs/11_caruana_prep/oof_long_{iso}.csv's own column shape are assembled from these
    # in Part 4 -- see the assembly cell there.
    frames = []
    for iso in ISOFORMS:
        ep = PIC50_COL[iso]
        m = pred_df[["inchikey", ep]].rename(columns={ep: "y_pred"}).copy()
        m["isoform"] = iso
        frames.append(m[["inchikey", "isoform", "y_pred"]])
    out = pd.concat(frames, ignore_index=True)
    out.insert(0, "fold", fold)
    out.insert(0, "repeat", repeat)
    out.insert(0, "arm", arm)
    out.to_csv(OOF_DIR / f"{arm}__r{repeat}f{fold}.csv", index=False)


def _cached(arm, repeat, fold, expected_names):
    p = PRED_DIR / f"{arm}__r{repeat}f{fold}.csv"
    if not p.exists():
        return None
    try:
        df = pd.read_csv(p)
    except Exception:
        return None
    if len(df) != len(expected_names) or set(df["Molecule_Name"]) != expected_names:
        print(f"  [cache MISMATCH] {arm}__r{repeat}f{fold}: content check failed -- will retrain")
        return None
    if df[list(REGRESSION_ENDPOINTS)].isna().to_numpy().any():
        print(f"  [cache MISMATCH] {arm}__r{repeat}f{fold}: NaN in a predicted column -- will retrain")
        return None
    return df

design: repeat 2 of the main arms + MAE_ONLY on repeats [0, 1, 2]


In [9]:
def run_chemprop_arm(repeat, fold, arm, curated_source=None):
    # PLAIN / AID / DEADZONE: one multitask chemprop run.
    seed = FOLD_SEED[(repeat, fold)]
    tag = f"{arm}__r{repeat}f{fold}"
    wp, _, _ = winsorise_for_fold(FULL_POP_RAW, repeat, fold) if arm == "aid" else (FULL_POP_RAW, None, None)
    population = build_population(wp, repeat, fold, seed, arm, curated_source=curated_source)
    target_cols = ARM_TARGET_COLS[arm]
    expected_names = set(population.loc[population["screen_split"] == "screen_test", "Molecule_Name"])

    hit = _cached(arm, repeat, fold, expected_names)
    if hit is not None:
        print(f"  [cache hit] {tag}")
        score_and_append(repeat, fold, arm, seed, hit)
        save_oof_long(repeat, fold, arm, hit)
        return hit, 0.0, _last_epoch(tag)

    run_dir = CHEMPROP_RUNS_DIR / tag
    run_dir.mkdir(parents=True, exist_ok=True)
    logger = setup_logging(LOG_DIR / f"{tag}.log", f"nb34_{tag}")
    logger.info(f"=== {tag}: repeat={repeat} fold={fold} arm={arm} seed={seed} ===")

    t0 = time.time()
    train_csv = run_dir / "train_input.csv"
    build_training_csv(population, target_cols, train_csv, logger, require_all_targets=False)
    predict_csv = build_predict_csv(population, run_dir, logger)
    run_chemprop_train(train_csv, target_cols, run_dir, logger, ARCH_ARGS[arm], EPOCHS, PATIENCE, seed)
    raw_pred_csv = run_dir / "raw_predictions.csv"
    run_chemprop_predict(run_dir / "model_0", predict_csv, raw_pred_csv, logger)
    verify_predictions(raw_pred_csv, expected_names, list(REGRESSION_ENDPOINTS), logger)
    elapsed = (time.time() - t0) / 60.0

    pred_df = pd.read_csv(raw_pred_csv)
    assert "inchikey" in pred_df.columns, "raw_predictions.csv lacks inchikey -- stopping"
    pred_df.to_csv(PRED_DIR / f"{tag}.csv", index=False)
    score_and_append(repeat, fold, arm, seed, pred_df)
    save_oof_long(repeat, fold, arm, pred_df)
    return pred_df, elapsed, _last_epoch(tag)


def run_single_arm(repeat, fold):
    # SINGLE: four independent single-task chemprop runs, one per isoform, assembled into one
    # prediction frame before scoring. require_all_targets=True drops rows lacking that one
    # isoform's label -- src/chemprop_screen.py's own contract for single-task models.
    seed = FOLD_SEED[(repeat, fold)]
    tag = f"single__r{repeat}f{fold}"
    population = build_population(FULL_POP_RAW, repeat, fold, seed, "plain")
    expected_names = set(population.loc[population["screen_split"] == "screen_test", "Molecule_Name"])

    hit = _cached("single", repeat, fold, expected_names)
    if hit is not None:
        print(f"  [cache hit] {tag}")
        score_and_append(repeat, fold, "single", seed, hit)
        save_oof_long(repeat, fold, "single", hit)
        return hit, 0.0, None

    logger = setup_logging(LOG_DIR / f"{tag}.log", f"nb34_{tag}")
    t0 = time.time()
    merged = None
    last_eps = []
    for endpoint in REGRESSION_ENDPOINTS:
        sub_tag = f"{tag}__{endpoint.split('_')[0]}"
        run_dir = CHEMPROP_RUNS_DIR / sub_tag
        run_dir.mkdir(parents=True, exist_ok=True)
        sub_pred = run_dir / "raw_predictions.csv"
        sub_ok = False
        if sub_pred.exists():
            try:
                _c = pd.read_csv(sub_pred)
                sub_ok = (len(_c) == len(expected_names)
                          and set(_c["Molecule_Name"]) == expected_names
                          and not _c[endpoint].isna().any())
            except Exception:
                sub_ok = False
            if not sub_ok:
                print(f"  [cache MISMATCH] {sub_tag}: content check failed -- retraining")
        if not sub_ok:
            train_csv = run_dir / "train_input.csv"
            build_training_csv(population, [endpoint], train_csv, logger, require_all_targets=True)
            predict_csv = build_predict_csv(population, run_dir, logger)
            run_chemprop_train(train_csv, [endpoint], run_dir, logger,
                               ARCH_ARGS["single"], EPOCHS, PATIENCE, seed)
            run_chemprop_predict(run_dir / "model_0", predict_csv, sub_pred, logger)
            verify_predictions(sub_pred, expected_names, [endpoint], logger)
        else:
            logger.info(f"[cache hit] {sub_tag}")
        last_eps.append(_last_epoch(sub_tag))
        part = pd.read_csv(sub_pred)
        keep = ["Molecule_Name", "inchikey", "canonical_smiles", endpoint]
        keep = [c for c in keep if c in part.columns]
        part = part[keep]
        merged = part if merged is None else merged.merge(
            part[["inchikey", endpoint]], on="inchikey", how="inner")
    elapsed = (time.time() - t0) / 60.0
    assert len(merged) == len(expected_names), "single-task assembly changed row count"
    assert set(merged["Molecule_Name"]) == expected_names
    assert not merged[list(REGRESSION_ENDPOINTS)].isna().to_numpy().any()
    merged.to_csv(PRED_DIR / f"{tag}.csv", index=False)
    score_and_append(repeat, fold, "single", seed, merged)
    save_oof_long(repeat, fold, "single", merged)
    return merged, elapsed, last_eps


def run_trees_arm(repeat, fold):
    seed = FOLD_SEED[(repeat, fold)]
    population = build_population(FULL_POP_RAW, repeat, fold, seed, "plain")
    expected_names = set(population.loc[population["screen_split"] == "screen_test", "Molecule_Name"])
    total = 0.0
    for config in TREE_ARMS:
        hit = _cached(config, repeat, fold, expected_names)
        if hit is not None:
            print(f"  [cache hit] {config}__r{repeat}f{fold}")
            score_and_append(repeat, fold, config, seed, hit)
            save_oof_long(repeat, fold, config, hit)
            continue
        t0 = time.time()
        pred_df = run_trees_one(config, repeat, fold, seed)
        el = (time.time() - t0) / 60.0
        total += el
        assert set(pred_df["Molecule_Name"]) == expected_names
        pred_df.to_csv(PRED_DIR / f"{config}__r{repeat}f{fold}.csv", index=False)
        score_and_append(repeat, fold, config, seed, pred_df)
        save_oof_long(repeat, fold, config, pred_df)
        print(f"  {config}__r{repeat}f{fold}: {el:.2f} min")
    return None, total, None


def run_unit(repeat, fold, arm_kind):
    # The single entry point every run cell calls. Handles abort state, the reduced design,
    # the wall-clock guard, timing, and the per-epoch degradation flag.
    label = f"{arm_kind}__r{repeat}f{fold}"
    if ABORT["aborted"]:
        print(f"[skipped -- RUN ABORTED earlier: {ABORT['reason']} at {ABORT['where']}] {label}")
        return None
    if not _in_design(repeat, arm_kind):
        print(f"[outside this notebook's design] {label}")
        return None
    if arm_kind == "deadzone" and not DEADZONE_READY.get(repeat, False):
        print(f"[skipped -- DEADZONE target unavailable for repeat {repeat}] {label}")
        return None
    if _deadline_exceeded():
        DEADLINE_SKIPS.append(label)
        print(f"[skipped -- {WALL_DEADLINE_H}h wall-clock guard exceeded] {label}")
        return None

    t0 = time.time()
    try:
        if arm_kind == "trees":
            _, el, ep = run_trees_arm(repeat, fold)
        elif arm_kind == "single":
            _, el, ep = run_single_arm(repeat, fold)
        elif arm_kind == "deadzone":
            _, el, ep = run_chemprop_arm(repeat, fold, "deadzone",
                                         curated_source=DEADZONE_TARGETS[repeat])
        else:
            # plain and mae_only both take the literal curated labels; they differ only in the
            # loss function ARCH_ARGS supplies.
            _, el, ep = run_chemprop_arm(repeat, fold, arm_kind)
    except Exception as exc:
        # One retry, then a clean abort -- never improvise around a hard failure.
        print(f"  *** {label} FAILED: {type(exc).__name__}: {exc}")
        print(f"  *** retrying once ***")
        try:
            if arm_kind == "trees":
                _, el, ep = run_trees_arm(repeat, fold)
            elif arm_kind == "single":
                _, el, ep = run_single_arm(repeat, fold)
            elif arm_kind == "deadzone":
                _, el, ep = run_chemprop_arm(repeat, fold, "deadzone",
                                             curated_source=DEADZONE_TARGETS[repeat])
            else:
                _, el, ep = run_chemprop_arm(repeat, fold, arm_kind)
        except Exception as exc2:
            hard_fail(f"{type(exc2).__name__}: {exc2} (failed twice)", label)
            return None

    wall = (time.time() - t0) / 60.0
    row = {"repeat": repeat, "fold": fold, "arm_kind": arm_kind, "wall_min": wall,
           "last_epoch": (max(e for e in ep if e is not None) if isinstance(ep, list) and any(e is not None for e in ep) else ep)}
    row["min_per_epoch"] = (wall / row["last_epoch"]) if row.get("last_epoch") else np.nan

    # A CACHE HIT MUST NOT OVERWRITE A COMPLETED RUN'S TIMING RECORD. Re-executing this
    # notebook after training finishes (which this notebook did, to apply corrections to the
    # Part 4 analysis) replays every run cell as a cache hit taking a few hundredths of a
    # minute. Appending those would silently destroy the only record of what the training
    # actually cost -- the same class of loss the brief's "never delete or overwrite a
    # completed run's cached predictions" rule exists to prevent, applied to the timing file.
    # A unit whose wall time is implausibly short for real training is therefore recorded only
    # if nothing is on file for it yet.
    CACHE_HIT_MAX_MIN = 0.5
    is_cache_hit = wall < CACHE_HIT_MAX_MIN
    already = False
    if TIMINGS_PATH.exists():
        _t = pd.read_csv(TIMINGS_PATH)
        already = bool(((_t["repeat"] == repeat) & (_t["fold"] == fold)
                        & (_t["arm_kind"] == arm_kind)).any())
    if is_cache_hit and already:
        print(f"[cache hit] {label}: keeping the existing timing record, not overwriting it")
    else:
        _append_rows(TIMINGS_PATH, [row], ["repeat", "fold", "arm_kind"])
    print(f"{label}: {wall:.2f} min  (last_epoch={row['last_epoch']}, "
          f"min/epoch={row['min_per_epoch'] if not pd.isna(row['min_per_epoch']) else float('nan'):.3f})")

    # Per-epoch degradation flag: notebook 28 saw 10.6x on an unchanged workload; notebook 29
    # did not. Flagged beyond 3x against this arm's own fastest observed epoch -- never aborted.
    tm = pd.read_csv(TIMINGS_PATH)
    same = tm[(tm["arm_kind"] == arm_kind) & tm["min_per_epoch"].notna()]
    if len(same) > 1 and not pd.isna(row["min_per_epoch"]):
        floor = float(same["min_per_epoch"].min())
        if floor > 0 and row["min_per_epoch"] / floor > 3.0:
            print(f"  *** PER-EPOCH DEGRADATION FLAG: {row['min_per_epoch']:.3f} min/epoch is "
                  f"{row['min_per_epoch']/floor:.1f}x this arm's own fastest ({floor:.3f}) -- "
                  f"logged, NOT aborted (notebook 28's documented failure mode).")
    return True


DEADZONE_READY = {}
DEADZONE_TARGETS = {}


def _in_design(repeat, arm_kind):
    # The one place this notebook's scope is defined. Main arms: the new repeat only (repeats 0
    # and 1 are notebook 34's and are never retrained). MAE_ONLY: all three repeats, so it pairs
    # with DEADZONE on all 15 units.
    if arm_kind == "mae_only":
        return repeat in MAE_ONLY_REPEATS
    return repeat == TARGET_REPEAT


print("run harness ready.")
print("in-design check:", {f"repeat{r}": [k for k in ["plain", "trees", "single", "aid",
                                                      "deadzone", "mae_only"]
                                          if _in_design(r, k)] for r in [0, 1, 2]})

run harness ready.
in-design check: {'repeat0': ['mae_only'], 'repeat1': ['mae_only'], 'repeat2': ['plain', 'trees', 'single', 'aid', 'deadzone', 'mae_only']}


### SANITY CHECK — before the main run

**This check earned its place on the first execution: it failed, and it was right to.** The first
run of this notebook copied notebook 33's setup cell verbatim, including
`os.environ.setdefault("OMP_NUM_THREADS", "1")`, and the sanity check came back at **0.0378** max
abs ST-RAE difference against a 0.03 threshold — so the run aborted cleanly at this cell, before
any of the 125 training units started. The diagnosis, and why it was not worked around: the
training input CSV was **byte-identical** to notebook 05's own (sha256 `66a1b380…`), the chemprop
CLI argv was **identical** to notebook 31's own baseline invocation, and val_loss was
**bit-identical for epochs 0 and 1** before diverging at epoch 2. Data and initialisation were
right; the arithmetic was not. Re-running with `OMP_NUM_THREADS`/`KMP_DUPLICATE_LIB_OK` unset and
nothing else changed reproduced notebook 05 **exactly** (0.00000000 on all four isoforms, with a
bit-identical val_loss trajectory). Notebooks 29 and 31 — the two that reproduce notebook 05 to
0.0000 — do not set those variables. The setup cell now asserts they are unset.

**A consequence for notebook 33, stated rather than left implicit.** Notebook 33's Chemprop runs
were produced under `OMP_NUM_THREADS=1`, and it never ran this check (it noted, correctly, that no
stored external comparison exists for a brand-new partition). Its own arm-vs-arm comparisons remain
internally consistent, since both its arms ran under the same setting. But its *absolute* Butina
numbers are not on the same numerical footing as notebook 05/29/31's random-partition numbers, which
is exactly the comparison it drew — a second reason, independent of fold count, to treat its
cross-partition conclusion as provisional.

PLAIN is retrained on the **RANDOM** partition's `repeat_0`/`fold_0` and compared against
`outputs/05_cv_comparison/scores/chemprop_chemeleoninit__repeat0_fold0.csv` (READ-ONLY).
Notebooks 18, 20, 22, 28, 29 and 31 all matched to 0.0000 max abs ST-RAE difference. The threshold
here is **0.03**; a mismatch beyond that is a **HARD FAILURE** and aborts the run, because it would
mean the pipeline has drifted and nothing downstream is interpretable.

In [10]:
from src.chemprop_screen import load_screen_population

SANITY_TAG = "sanity_plain__random_r0f0"
sanity_seed = FOLD_SEED[(0, 0)]
logger = setup_logging(LOG_DIR / f"{SANITY_TAG}.log", f"nb34_{SANITY_TAG}")
sanity_pred_path = PRED_DIR / f"{SANITY_TAG}.csv"

# Exactly scripts/run_5x5_cv_comparison.py's own run_chemprop path, on the frozen random
# partition, read-only.
sanity_pop = load_screen_population(FOLDS_PATH, CURATED_PATH, "repeat_0", 0,
                                    VAL_FRACTION, sanity_seed, logger)
sanity_expected = set(sanity_pop.loc[sanity_pop["screen_split"] == "screen_test", "Molecule_Name"])

if sanity_pred_path.exists() and set(pd.read_csv(sanity_pred_path)["Molecule_Name"]) == sanity_expected:
    print(f"[cache hit] {SANITY_TAG}")
    sanity_pred = pd.read_csv(sanity_pred_path)
else:
    run_dir = CHEMPROP_RUNS_DIR / SANITY_TAG
    run_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    train_csv = run_dir / "train_input.csv"
    build_training_csv(sanity_pop, list(REGRESSION_ENDPOINTS), train_csv, logger,
                       require_all_targets=False)
    predict_csv = build_predict_csv(sanity_pop, run_dir, logger)
    run_chemprop_train(train_csv, list(REGRESSION_ENDPOINTS), run_dir, logger,
                       ARCH_ARGS["plain"], EPOCHS, PATIENCE, sanity_seed)
    raw_pred_csv = run_dir / "raw_predictions.csv"
    run_chemprop_predict(run_dir / "model_0", predict_csv, raw_pred_csv, logger)
    verify_predictions(raw_pred_csv, sanity_expected, list(REGRESSION_ENDPOINTS), logger)
    sanity_pred = pd.read_csv(raw_pred_csv)
    sanity_pred.to_csv(sanity_pred_path, index=False)
    print(f"\\nsanity run wall time: {(time.time()-t0)/60:.2f} min")

sanity_gt = curated[curated["inchikey"].isin(sanity_pred["inchikey"])].copy()
with per_fold_bootstrap_seed(sanity_seed):
    sanity_scored = score_activity_predictions(sanity_pred, sanity_gt, list(REGRESSION_ENDPOINTS))
mine = sanity_scored.groupby("Endpoint")["ST-RAE"].mean()

nb05 = pd.read_csv(NB05_OUT / "scores" / "chemprop_chemeleoninit__repeat0_fold0.csv")
theirs = nb05.groupby("Endpoint")["ST-RAE"].mean()

print(f"\\nSANITY CHECK -- PLAIN retrained on random repeat_0/fold_0 vs. notebook 05's stored scores")
print(f"{'endpoint':42s} {'this run':>10s} {'nb05':>10s} {'abs diff':>10s}")
max_diff = 0.0
for ep in sorted(set(mine.index) & set(theirs.index)):
    d = abs(float(mine[ep]) - float(theirs[ep]))
    max_diff = max(max_diff, d)
    print(f"{ep:42s} {mine[ep]:10.4f} {theirs[ep]:10.4f} {d:10.4f}")
print(f"\\nmax abs ST-RAE difference: {max_diff:.6f}   threshold: {SANITY_THRESHOLD}")

pd.DataFrame({"Endpoint": list(mine.index), "this_run": mine.to_numpy(),
              "nb05": [float(theirs.get(e, np.nan)) for e in mine.index]}
             ).to_csv(OUT / "sanity_check.csv", index=False)

if max_diff > SANITY_THRESHOLD:
    hard_fail(f"sanity check FAILED: max abs ST-RAE difference {max_diff:.6f} exceeds "
              f"{SANITY_THRESHOLD} against notebook 05's stored chemprop_chemeleoninit "
              f"repeat0_fold0 scores -- the pipeline has drifted and nothing downstream is "
              f"interpretable.", "Part 3 sanity check")
else:
    print("PASS -- the pipeline reproduces notebook 05's own stored numbers within threshold.")

2026-10-01 06:48:06,546 [INFO] loaded cv_folds.csv: (4905, 7)


2026-10-01 06:48:06,546 [INFO] loaded train_inhibition_curated.csv: (4905, 20)


2026-10-01 06:48:06,549 [INFO] assign_screen_split(repeat_col='repeat_0', test_fold=0, val_fraction=0.15, seed=2684470948) split counts: {'screen_inner_train': 3335, 'screen_test': 981, 'screen_inner_val': 589}


2026-10-01 06:48:06,551 [INFO] rows after SMILES/target join: 4905 (missing: 0)


2026-10-01 06:48:06,552 [INFO]   CYP1A2_pIC50_direct_inhibition: labeled-compound counts per split = {'screen_inner_train': 975, 'screen_inner_val': 160, 'screen_test': 277}


...

2026-10-01 06:48:07.925 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 686 compound(s) with no ground truth for endpoint CYP2D6_pIC50_direct_inhibition


2026-10-01 06:48:08.614 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP3A4_pIC50_direct_inhibition


2026-10-01 06:48:08.615 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 531 compound(s) with no ground truth for endpoint CYP3A4_pIC50_direct_inhibition


2026-10-01 06:48:09.361 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:169 - Completed scoring activity predictions


\nSANITY CHECK -- PLAIN retrained on random repeat_0/fold_0 vs. notebook 05's stored scores
endpoint                                     this run       nb05   abs diff
CYP1A2_pIC50_direct_inhibition                 0.7884     0.7884     0.0000
CYP2C9_pIC50_direct_inhibition                 0.7113     0.7113     0.0000
CYP2D6_pIC50_direct_inhibition                 0.9580     0.9580     0.0000
CYP3A4_pIC50_direct_inhibition                 0.4910     0.4910     0.0000
\nmax abs ST-RAE difference: 0.000000   threshold: 0.03
PASS -- the pipeline reproduces notebook 05's own stored numbers within threshold.


### Repeat 2 — the new repeat of the main design

In [11]:
# PLAIN -- repeat 2, fold 0
run_unit(2, 0, 'plain')

2026-10-01 06:48:09.379 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:48:09.381 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:48:09.381 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:48:09.381 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 700 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] plain__r2f0


...

2026-10-01 06:48:12.180 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:48:12.181 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:48:12.181 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] plain__r2f0: keeping the existing timing record, not overwriting it
plain__r2f0: 0.05 min  (last_epoch=9, min/epoch=0.005)


True

In [12]:
# PLAIN -- repeat 2, fold 1
run_unit(2, 1, 'plain')

2026-10-01 06:48:12.227 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:48:12.228 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:48:12.228 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:48:12.229 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 678 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] plain__r2f1


...

2026-10-01 06:48:15.009 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:48:15.009 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:48:15.009 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] plain__r2f1: keeping the existing timing record, not overwriting it
plain__r2f1: 0.05 min  (last_epoch=9, min/epoch=0.005)


True

In [13]:
# PLAIN -- repeat 2, fold 2
run_unit(2, 2, 'plain')

2026-10-01 06:48:15.055 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:48:15.056 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:48:15.056 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:48:15.056 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 700 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] plain__r2f2


...

2026-10-01 06:48:17.825 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:48:17.825 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:48:17.825 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] plain__r2f2: keeping the existing timing record, not overwriting it
plain__r2f2: 0.05 min  (last_epoch=19, min/epoch=0.002)


True

In [14]:
# PLAIN -- repeat 2, fold 3
run_unit(2, 3, 'plain')

2026-10-01 06:48:17.870 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:48:17.871 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:48:17.871 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:48:17.871 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 702 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] plain__r2f3


...

2026-10-01 06:48:20.653 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:48:20.653 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:48:20.654 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] plain__r2f3: keeping the existing timing record, not overwriting it
plain__r2f3: 0.05 min  (last_epoch=7, min/epoch=0.007)


True

In [15]:
# PLAIN -- repeat 2, fold 4
run_unit(2, 4, 'plain')

2026-10-01 06:48:20.699 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:48:20.700 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:48:20.700 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:48:20.701 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 713 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] plain__r2f4


...

2026-10-01 06:48:23.472 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:48:23.472 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:48:23.472 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] plain__r2f4: keeping the existing timing record, not overwriting it
plain__r2f4: 0.05 min  (last_epoch=6, min/epoch=0.008)


True

In [16]:
# TREES (6 configs) -- repeat 2, fold 0
run_unit(2, 0, 'trees')

2026-10-01 06:48:23.517 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:48:23.518 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 25 columns.


2026-10-01 06:48:23.518 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:48:23.518 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 700 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] ecfp4_narrow__rf__r2f0


...

2026-10-01 06:48:40.426 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:48:40.426 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:48:40.427 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] trees__r2f0: keeping the existing timing record, not overwriting it
trees__r2f0: 0.28 min  (last_epoch=None, min/epoch=nan)


True

In [17]:
# TREES (6 configs) -- repeat 2, fold 1
run_unit(2, 1, 'trees')

2026-10-01 06:48:40.472 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:48:40.474 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 25 columns.


2026-10-01 06:48:40.474 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:48:40.474 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 678 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] ecfp4_narrow__rf__r2f1


...

2026-10-01 06:48:57.384 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:48:57.385 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:48:57.385 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] trees__r2f1: keeping the existing timing record, not overwriting it
trees__r2f1: 0.28 min  (last_epoch=None, min/epoch=nan)


True

In [18]:
# TREES (6 configs) -- repeat 2, fold 2
run_unit(2, 2, 'trees')

2026-10-01 06:48:57.431 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:48:57.432 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 25 columns.


2026-10-01 06:48:57.432 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:48:57.432 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 700 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] ecfp4_narrow__rf__r2f2


...

2026-10-01 06:49:14.334 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:49:14.335 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:49:14.335 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] trees__r2f2: keeping the existing timing record, not overwriting it
trees__r2f2: 0.28 min  (last_epoch=None, min/epoch=nan)


True

In [19]:
# TREES (6 configs) -- repeat 2, fold 3
run_unit(2, 3, 'trees')

2026-10-01 06:49:14.381 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:49:14.382 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 25 columns.


2026-10-01 06:49:14.382 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:49:14.382 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 702 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] ecfp4_narrow__rf__r2f3


...

2026-10-01 06:49:31.272 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:49:31.272 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:49:31.273 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] trees__r2f3: keeping the existing timing record, not overwriting it
trees__r2f3: 0.28 min  (last_epoch=None, min/epoch=nan)


True

In [20]:
# TREES (6 configs) -- repeat 2, fold 4
run_unit(2, 4, 'trees')

2026-10-01 06:49:31.319 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:49:31.320 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 25 columns.


2026-10-01 06:49:31.320 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:49:31.320 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 713 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] ecfp4_narrow__rf__r2f4


...

2026-10-01 06:49:48.161 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:49:48.161 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:49:48.162 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] trees__r2f4: keeping the existing timing record, not overwriting it
trees__r2f4: 0.28 min  (last_epoch=None, min/epoch=nan)


True

In [21]:
# SINGLE (4 single-task models) -- repeat 2, fold 0
run_unit(2, 0, 'single')

2026-10-01 06:49:48.207 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:49:48.209 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:49:48.209 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:49:48.209 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 700 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] single__r2f0


...

2026-10-01 06:49:50.994 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:49:50.994 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:49:50.995 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] single__r2f0: keeping the existing timing record, not overwriting it
single__r2f0: 0.05 min  (last_epoch=None, min/epoch=nan)


True

In [22]:
# SINGLE (4 single-task models) -- repeat 2, fold 1
run_unit(2, 1, 'single')

2026-10-01 06:49:51.038 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:49:51.040 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:49:51.040 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:49:51.040 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 678 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] single__r2f1


...

2026-10-01 06:49:53.815 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:49:53.815 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:49:53.815 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] single__r2f1: keeping the existing timing record, not overwriting it
single__r2f1: 0.05 min  (last_epoch=None, min/epoch=nan)


True

In [23]:
# SINGLE (4 single-task models) -- repeat 2, fold 2
run_unit(2, 2, 'single')

2026-10-01 06:49:53.860 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:49:53.861 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:49:53.861 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:49:53.861 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 700 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] single__r2f2


...

2026-10-01 06:49:56.645 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:49:56.645 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:49:56.646 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] single__r2f2: keeping the existing timing record, not overwriting it
single__r2f2: 0.05 min  (last_epoch=None, min/epoch=nan)


True

In [24]:
# SINGLE (4 single-task models) -- repeat 2, fold 3
run_unit(2, 3, 'single')

2026-10-01 06:49:56.691 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:49:56.692 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:49:56.692 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:49:56.692 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 702 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] single__r2f3


...

2026-10-01 06:49:59.511 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:49:59.512 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:49:59.512 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] single__r2f3: keeping the existing timing record, not overwriting it
single__r2f3: 0.05 min  (last_epoch=None, min/epoch=nan)


True

In [25]:
# SINGLE (4 single-task models) -- repeat 2, fold 4
run_unit(2, 4, 'single')

2026-10-01 06:49:59.556 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:49:59.558 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:49:59.558 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:49:59.558 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 713 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] single__r2f4


...

2026-10-01 06:50:02.339 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:50:02.340 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:50:02.340 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] single__r2f4: keeping the existing timing record, not overwriting it
single__r2f4: 0.05 min  (last_epoch=None, min/epoch=nan)


True

In [26]:
# AID -- repeat 2, fold 0
run_unit(2, 0, 'aid')

2026-10-01 06:50:02.392 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:50:02.393 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 30 columns.


2026-10-01 06:50:02.393 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:50:02.393 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 700 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] aid__r2f0


...

2026-10-01 06:50:05.166 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:50:05.166 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:50:05.167 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] aid__r2f0: keeping the existing timing record, not overwriting it
aid__r2f0: 0.05 min  (last_epoch=10, min/epoch=0.005)


True

In [27]:
# AID -- repeat 2, fold 1
run_unit(2, 1, 'aid')

2026-10-01 06:50:05.218 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:50:05.219 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 30 columns.


2026-10-01 06:50:05.219 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:50:05.219 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 678 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] aid__r2f1


...

2026-10-01 06:50:07.975 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:50:07.975 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:50:07.976 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] aid__r2f1: keeping the existing timing record, not overwriting it
aid__r2f1: 0.05 min  (last_epoch=11, min/epoch=0.004)


True

In [28]:
# AID -- repeat 2, fold 2
run_unit(2, 2, 'aid')

2026-10-01 06:50:08.028 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:50:08.029 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 30 columns.


2026-10-01 06:50:08.029 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:50:08.029 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 700 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] aid__r2f2


...

2026-10-01 06:50:10.841 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:50:10.842 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:50:10.842 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] aid__r2f2: keeping the existing timing record, not overwriting it
aid__r2f2: 0.05 min  (last_epoch=9, min/epoch=0.005)


True

In [29]:
# AID -- repeat 2, fold 3
run_unit(2, 3, 'aid')

2026-10-01 06:50:10.894 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:50:10.895 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 30 columns.


2026-10-01 06:50:10.895 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:50:10.896 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 702 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] aid__r2f3


...

2026-10-01 06:50:13.671 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:50:13.671 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:50:13.672 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] aid__r2f3: keeping the existing timing record, not overwriting it
aid__r2f3: 0.05 min  (last_epoch=13, min/epoch=0.004)


True

In [30]:
# AID -- repeat 2, fold 4
run_unit(2, 4, 'aid')

2026-10-01 06:50:13.724 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:50:13.725 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 30 columns.


2026-10-01 06:50:13.726 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:50:13.726 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 713 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] aid__r2f4


...

2026-10-01 06:50:16.499 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:50:16.499 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:50:16.500 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] aid__r2f4: keeping the existing timing record, not overwriting it
aid__r2f4: 0.05 min  (last_epoch=7, min/epoch=0.007)


True

**DEADZONE target for repeat 2** — built from PLAIN's completed repeat-2 out-of-fold predictions on
this partition. Each compound is held out exactly once per repeat, so PLAIN's five folds give one
out-of-fold prediction per compound. Two guards are applied before any DEADZONE training, matching
notebook `31`'s own: a materiality check (the clipped target must differ from the raw label on a
material fraction of rows) and, the one doing the real work, a band-hit-rate ceiling — an overfit
or in-fold donor would push the hit rate toward 1.0.

In [31]:
# ---- DEADZONE target for repeat 2. Built from PLAIN's own out-of-fold predictions ON THIS
# PARTITION, never from outputs/11_caruana_prep/, which lives on the random partition.
R = TARGET_REPEAT
DEADZONE_READY[R] = False
if ABORT["aborted"]:
    print(f"[skipped] the run has aborted")
else:
    plain_files = [PRED_DIR / f"plain__r{R}f{f}.csv" for f in range(N_FOLDS)]
    missing = [p.name for p in plain_files if not p.exists()]
    if missing:
        msg = (f"PLAIN's repeat {R} is incomplete ({len(missing)} of {N_FOLDS} fold predictions "
               f"missing: {missing}). DEADZONE for repeat {R} is SKIPPED -- falling back to the "
               f"random-partition OOF is explicitly ruled out.")
        print(f"*** {msg}")
        record_decision(decision=f"Skipped the DEADZONE arm for repeat {R}", reason=msg,
                        alternatives="Use outputs/11_caruana_prep/'s random-partition OOF as the target (explicitly forbidden).",
                        authority="the brief's own DEADZONE dependency rule")
    else:
        oof = pd.concat([pd.read_csv(p)[["inchikey"] + list(REGRESSION_ENDPOINTS)]
                         for p in plain_files], ignore_index=True)
        assert len(oof) == len(curated), f"PLAIN repeat {R} OOF has {len(oof)} rows, expected {len(curated)}"
        assert oof["inchikey"].is_unique, "a compound appears in more than one fold's held-out set"
        assert set(oof["inchikey"]) == curated_ik

        curated_dz = curated.copy()
        report = []
        for iso in ISOFORMS:
            ep = PIC50_COL[iso]
            m = curated[["inchikey", ep, f"{ep}_conf_low", f"{ep}_conf_high"]].merge(
                oof[["inchikey", ep]].rename(columns={ep: "_oof"}), on="inchikey", how="left")
            assert m["inchikey"].tolist() == curated["inchikey"].tolist(), \
                "left merge did not preserve curated row order -- positional assignment would misalign"
            assert m["_oof"].notna().all(), f"{iso}: a compound has no PLAIN OOF prediction"
            has_label = m[ep].notna()
            clipped = m["_oof"].clip(lower=m[f"{ep}_conf_low"], upper=m[f"{ep}_conf_high"])
            target = np.where(has_label, clipped, np.nan)
            curated_dz[ep] = target

            differs = (np.abs(target - m[ep].to_numpy()) > 1e-6) & has_label.to_numpy()
            frac_differs = float(differs.sum() / has_label.sum())
            inside = ((m["_oof"] >= m[f"{ep}_conf_low"]) & (m["_oof"] <= m[f"{ep}_conf_high"]))
            hit_rate = float(inside[has_label].mean())
            report.append({"repeat": R, "isoform": iso, "n_labelled": int(has_label.sum()),
                           "frac_target_differs_from_label": frac_differs,
                           "band_hit_rate": hit_rate})
        rep = pd.DataFrame(report)
        print(rep.to_string(index=False))
        _append_rows(OUT / "deadzone_target_checks.csv", report, ["repeat", "isoform"])

        # Cross-check against notebook 34's own repeat-0/1 checks: the band hit rate is a
        # property of PLAIN's fit on this partition and should land in the same range (34
        # recorded 0.178-0.426), not jump. Reported, not asserted -- a different repeat is a
        # different fold assignment and some movement is expected.
        nb34_checks = pd.read_csv(NB34_OUT / "deadzone_target_checks.csv")
        print(f"\nnotebook 34's repeat-0/1 band hit rates: "
              f"{nb34_checks['band_hit_rate'].min():.3f}-{nb34_checks['band_hit_rate'].max():.3f}")
        print(f"this repeat's:                            "
              f"{rep['band_hit_rate'].min():.3f}-{rep['band_hit_rate'].max():.3f}")

        bad = rep[(rep["frac_target_differs_from_label"] < 0.50) | (rep["band_hit_rate"] >= 0.90)]
        if len(bad):
            msg = (f"DEADZONE target check failed for repeat {R}:\n{bad.to_string(index=False)}\n"
                   f"The clipped target has silently become the identity, or the OOF join leaked "
                   f"in-fold predictions. Arm SKIPPED for this repeat.")
            print(f"*** {msg}")
            record_decision(decision=f"Skipped the DEADZONE arm for repeat {R}", reason=msg,
                            alternatives="Train the arm anyway on a target that is not the intended intervention.",
                            authority="the brief's own pre-training assertion on the DEADZONE target")
        else:
            dz_path = OUT / f"curated_deadzone_target_r{R}.csv"
            curated_dz.to_csv(dz_path, index=False)
            DEADZONE_TARGETS[R] = curated_dz
            DEADZONE_READY[R] = True
            print(f"\nDEADZONE target for repeat {R} PASSED both checks -> {dz_path.name}")

 repeat isoform  n_labelled  frac_target_differs_from_label  band_hit_rate
      2  CYP1A2        1412                             1.0       0.223796
      2  CYP2C9        1285                             1.0       0.412451
      2  CYP2D6        1493                             1.0       0.186872
      2  CYP3A4        2335                             1.0       0.362313

notebook 34's repeat-0/1 band hit rates: 0.178-0.426
this repeat's:                            0.187-0.412

DEADZONE target for repeat 2 PASSED both checks -> curated_deadzone_target_r2.csv


In [32]:
# DEADZONE -- repeat 2, fold 0
run_unit(2, 0, 'deadzone')

2026-10-01 06:50:16.602 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:50:16.604 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:50:16.604 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:50:16.604 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 700 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] deadzone__r2f0


...

2026-10-01 06:50:19.386 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:50:19.386 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:50:19.387 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] deadzone__r2f0: keeping the existing timing record, not overwriting it
deadzone__r2f0: 0.05 min  (last_epoch=12, min/epoch=0.004)


True

In [33]:
# DEADZONE -- repeat 2, fold 1
run_unit(2, 1, 'deadzone')

2026-10-01 06:50:19.433 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:50:19.434 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:50:19.434 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:50:19.434 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 678 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] deadzone__r2f1


...

2026-10-01 06:50:22.212 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:50:22.212 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:50:22.213 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] deadzone__r2f1: keeping the existing timing record, not overwriting it
deadzone__r2f1: 0.05 min  (last_epoch=11, min/epoch=0.004)


True

In [34]:
# DEADZONE -- repeat 2, fold 2
run_unit(2, 2, 'deadzone')

2026-10-01 06:50:22.259 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:50:22.260 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:50:22.260 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:50:22.260 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 700 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] deadzone__r2f2


...

2026-10-01 06:50:25.027 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:50:25.027 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:50:25.028 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] deadzone__r2f2: keeping the existing timing record, not overwriting it
deadzone__r2f2: 0.05 min  (last_epoch=8, min/epoch=0.006)


True

In [35]:
# DEADZONE -- repeat 2, fold 3
run_unit(2, 3, 'deadzone')

2026-10-01 06:50:25.073 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:50:25.074 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:50:25.074 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:50:25.074 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 702 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] deadzone__r2f3


...

2026-10-01 06:50:27.878 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:50:27.879 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:50:27.879 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] deadzone__r2f3: keeping the existing timing record, not overwriting it
deadzone__r2f3: 0.05 min  (last_epoch=8, min/epoch=0.006)


True

In [36]:
# DEADZONE -- repeat 2, fold 4
run_unit(2, 4, 'deadzone')

2026-10-01 06:50:27.926 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:50:27.928 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:50:27.928 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:50:27.928 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 713 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] deadzone__r2f4


...

2026-10-01 06:50:30.711 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:50:30.712 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:50:30.712 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] deadzone__r2f4: keeping the existing timing record, not overwriting it
deadzone__r2f4: 0.05 min  (last_epoch=14, min/epoch=0.003)


True

In [37]:
# ---- Progress snapshot after repeat 2's main design
if RUN_SCORES_PATH.exists():
    _rs = pd.read_csv(RUN_SCORES_PATH)
    _done = _rs[["repeat", "fold", "arm"]].drop_duplicates()
    print(f"completed (repeat, fold, arm) units so far: {len(_done)}")
    print(_done.groupby(["repeat", "arm"]).size().unstack(fill_value=0).to_string())
_el = (time.time() - RUN_STARTED_AT) / 3600.0
print(f"\nelapsed since campaign start: {_el:.2f} h  (wall guard {WALL_DEADLINE_H} h)")
if DEADLINE_SKIPS:
    print(f"units skipped by the wall guard so far: {len(DEADLINE_SKIPS)}")

completed (repeat, fold, arm) units so far: 65
arm     aid  deadzone  ecfp4_narrow__lightgbm  ecfp4_narrow__rf  ecfp4_narrow__xgboost  mae_only  mordred_pca__lightgbm  mordred_pca__rf  mordred_pca__xgboost  plain  single
repeat                                                                                                                                                                       
0         0         0                       0                 0                      0         5                      0                0                     0      0       0
1         0         0                       0                 0                      0         5                      0                0                     0      0       0
2         5         5                       5                 5                      5         5                      5                5                     5      5       5

elapsed since campaign start: 7.82 h  (wall guard 12.0 h)


### MAE_ONLY — all three repeats

The loss-function-only arm, run across repeats 0, 1 and 2 so it is paired with DEADZONE on all 15
units. Repeats 0 and 1 reuse notebook `34`'s own fold assignment and seeds (asserted identical in
Part 2); nothing of notebook `34`'s is retrained or overwritten.

In [38]:
# MAE_ONLY -- repeat 0, fold 0
run_unit(0, 0, 'mae_only')

2026-10-01 06:50:30.763 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:50:30.764 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:50:30.764 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:50:30.764 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 684 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] mae_only__r0f0


...

2026-10-01 06:50:33.548 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:50:33.548 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:50:33.549 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] mae_only__r0f0: keeping the existing timing record, not overwriting it
mae_only__r0f0: 0.05 min  (last_epoch=11, min/epoch=0.004)


True

In [39]:
# MAE_ONLY -- repeat 0, fold 1
run_unit(0, 1, 'mae_only')

2026-10-01 06:50:33.593 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:50:33.595 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:50:33.595 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:50:33.595 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 723 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] mae_only__r0f1


...

2026-10-01 06:50:36.371 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:50:36.372 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:50:36.372 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] mae_only__r0f1: keeping the existing timing record, not overwriting it
mae_only__r0f1: 0.05 min  (last_epoch=17, min/epoch=0.003)


True

In [40]:
# MAE_ONLY -- repeat 0, fold 2
run_unit(0, 2, 'mae_only')

2026-10-01 06:50:36.416 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:50:36.418 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:50:36.418 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:50:36.418 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 675 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] mae_only__r0f2


...

2026-10-01 06:50:39.222 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:50:39.222 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:50:39.223 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] mae_only__r0f2: keeping the existing timing record, not overwriting it
mae_only__r0f2: 0.05 min  (last_epoch=11, min/epoch=0.004)


True

In [41]:
# MAE_ONLY -- repeat 0, fold 3
run_unit(0, 3, 'mae_only')

2026-10-01 06:50:39.270 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:50:39.271 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:50:39.271 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:50:39.271 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 701 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] mae_only__r0f3


...

2026-10-01 06:50:42.058 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:50:42.058 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:50:42.059 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] mae_only__r0f3: keeping the existing timing record, not overwriting it
mae_only__r0f3: 0.05 min  (last_epoch=15, min/epoch=0.003)


True

In [42]:
# MAE_ONLY -- repeat 0, fold 4
run_unit(0, 4, 'mae_only')

2026-10-01 06:50:42.106 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:50:42.107 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:50:42.108 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:50:42.108 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 710 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] mae_only__r0f4


...

2026-10-01 06:50:44.883 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:50:44.883 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:50:44.884 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] mae_only__r0f4: keeping the existing timing record, not overwriting it
mae_only__r0f4: 0.05 min  (last_epoch=11, min/epoch=0.004)


True

In [43]:
# MAE_ONLY -- repeat 1, fold 0
run_unit(1, 0, 'mae_only')

2026-10-01 06:50:44.928 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:50:44.929 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:50:44.929 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:50:44.930 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 704 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] mae_only__r1f0


...

2026-10-01 06:50:47.708 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:50:47.708 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:50:47.709 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] mae_only__r1f0: keeping the existing timing record, not overwriting it
mae_only__r1f0: 0.05 min  (last_epoch=10, min/epoch=0.005)


True

In [44]:
# MAE_ONLY -- repeat 1, fold 1
run_unit(1, 1, 'mae_only')

2026-10-01 06:50:47.754 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:50:47.755 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:50:47.755 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:50:47.756 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 715 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] mae_only__r1f1


...

2026-10-01 06:50:50.539 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:50:50.539 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:50:50.540 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] mae_only__r1f1: keeping the existing timing record, not overwriting it
mae_only__r1f1: 0.05 min  (last_epoch=9, min/epoch=0.005)


True

In [45]:
# MAE_ONLY -- repeat 1, fold 2
run_unit(1, 2, 'mae_only')

2026-10-01 06:50:50.584 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:50:50.585 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:50:50.585 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:50:50.585 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 672 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] mae_only__r1f2


...

2026-10-01 06:50:53.370 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:50:53.370 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:50:53.371 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] mae_only__r1f2: keeping the existing timing record, not overwriting it
mae_only__r1f2: 0.05 min  (last_epoch=12, min/epoch=0.004)


True

In [46]:
# MAE_ONLY -- repeat 1, fold 3
run_unit(1, 3, 'mae_only')

2026-10-01 06:50:53.416 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:50:53.417 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:50:53.417 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:50:53.417 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 706 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] mae_only__r1f3


...

2026-10-01 06:50:56.190 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:50:56.191 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:50:56.191 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] mae_only__r1f3: keeping the existing timing record, not overwriting it
mae_only__r1f3: 0.05 min  (last_epoch=9, min/epoch=0.005)


True

In [47]:
# MAE_ONLY -- repeat 1, fold 4
run_unit(1, 4, 'mae_only')

2026-10-01 06:50:56.236 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:50:56.237 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:50:56.238 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:50:56.238 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 696 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] mae_only__r1f4


...

2026-10-01 06:50:59.045 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:50:59.045 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:50:59.046 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] mae_only__r1f4: keeping the existing timing record, not overwriting it
mae_only__r1f4: 0.05 min  (last_epoch=8, min/epoch=0.006)


True

In [48]:
# MAE_ONLY -- repeat 2, fold 0
run_unit(2, 0, 'mae_only')

2026-10-01 06:50:59.094 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:50:59.095 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:50:59.095 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:50:59.096 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 700 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] mae_only__r2f0


...

2026-10-01 06:51:01.884 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:51:01.885 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:51:01.885 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] mae_only__r2f0: keeping the existing timing record, not overwriting it
mae_only__r2f0: 0.05 min  (last_epoch=12, min/epoch=0.004)


True

In [49]:
# MAE_ONLY -- repeat 2, fold 1
run_unit(2, 1, 'mae_only')

2026-10-01 06:51:01.930 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:51:01.931 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:51:01.931 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:51:01.931 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 678 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] mae_only__r2f1


...

2026-10-01 06:51:04.719 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:51:04.719 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:51:04.720 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] mae_only__r2f1: keeping the existing timing record, not overwriting it
mae_only__r2f1: 0.05 min  (last_epoch=15, min/epoch=0.003)


True

In [50]:
# MAE_ONLY -- repeat 2, fold 2
run_unit(2, 2, 'mae_only')

2026-10-01 06:51:04.766 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:51:04.767 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:51:04.768 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:51:04.768 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 700 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] mae_only__r2f2


...

2026-10-01 06:51:07.539 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:51:07.539 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:51:07.540 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] mae_only__r2f2: keeping the existing timing record, not overwriting it
mae_only__r2f2: 0.05 min  (last_epoch=10, min/epoch=0.005)


True

In [51]:
# MAE_ONLY -- repeat 2, fold 3
run_unit(2, 3, 'mae_only')

2026-10-01 06:51:07.585 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:51:07.586 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:51:07.586 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:51:07.587 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 702 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] mae_only__r2f3


...

2026-10-01 06:51:10.408 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:51:10.408 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:51:10.409 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] mae_only__r2f3: keeping the existing timing record, not overwriting it
mae_only__r2f3: 0.05 min  (last_epoch=15, min/epoch=0.003)


True

In [52]:
# MAE_ONLY -- repeat 2, fold 4
run_unit(2, 4, 'mae_only')

2026-10-01 06:51:10.454 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-10-01 06:51:10.455 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 26 columns.


2026-10-01 06:51:10.455 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-10-01 06:51:10.455 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 713 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] mae_only__r2f4


...

2026-10-01 06:51:13.236 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-10-01 06:51:13.236 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-10-01 06:51:13.237 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] mae_only__r2f4: keeping the existing timing record, not overwriting it
mae_only__r2f4: 0.05 min  (last_epoch=9, min/epoch=0.005)


True

In [53]:
# ---- Progress snapshot after MAE_ONLY
if RUN_SCORES_PATH.exists():
    _rs = pd.read_csv(RUN_SCORES_PATH)
    _done = _rs[["repeat", "fold", "arm"]].drop_duplicates()
    print(f"completed (repeat, fold, arm) units in THIS notebook: {len(_done)}")
    print(_done.groupby(["repeat", "arm"]).size().unstack(fill_value=0).to_string())
_el = (time.time() - RUN_STARTED_AT) / 3600.0
print(f"\nelapsed since campaign start: {_el:.2f} h  (wall guard {WALL_DEADLINE_H} h)")
if DEADLINE_SKIPS:
    print(f"*** units skipped by the wall guard: {len(DEADLINE_SKIPS)}")
    for s in DEADLINE_SKIPS:
        print(f"    {s}")
if ABORT["aborted"]:
    print(f"*** RUN ABORTED at {ABORT['where']}: {ABORT['reason']}")

completed (repeat, fold, arm) units in THIS notebook: 65
arm     aid  deadzone  ecfp4_narrow__lightgbm  ecfp4_narrow__rf  ecfp4_narrow__xgboost  mae_only  mordred_pca__lightgbm  mordred_pca__rf  mordred_pca__xgboost  plain  single
repeat                                                                                                                                                                       
0         0         0                       0                 0                      0         5                      0                0                     0      0       0
1         0         0                       0                 0                      0         5                      0                0                     0      0       0
2         5         5                       5                 5                      5         5                      5                5                     5      5       5

elapsed since campaign start: 7.83 h  (wall guard 12.0 h)


## Part 4 — Analysis at 15 folds

Read-only from here on: no model is trained below this point. Notebook `34`'s scores and
out-of-fold predictions are read from `outputs/34_butina_5x5/` and never written to.

**Every table reports the 10-fold and the 15-fold result side by side.** That is the check that
adding a repeat was not a selection: if the 15-fold answer simply is the 10-fold answer with
tighter intervals, nothing moved under the extra data. Where a verdict does change, it is named
explicitly rather than quietly replacing the old one.

In [54]:
# ---- Assemble the combined score table: notebook 34's two repeats (READ-ONLY) plus this
# notebook's repeat 2 and MAE_ONLY runs.
nb34_rs = pd.read_csv(NB34_OUT / "run_scores.csv")
assert sorted(nb34_rs["repeat"].unique()) == NB34_REPEATS, "notebook 34's repeats are not [0, 1]"
assert "mae_only" not in set(nb34_rs["arm"]), "notebook 34 unexpectedly contains a mae_only arm"
print(f"notebook 34 (read-only): {len(nb34_rs)} rows, "
      f"{nb34_rs[['repeat','fold','arm']].drop_duplicates().shape[0]} units, "
      f"{nb34_rs['arm'].nunique()} arms, repeats {sorted(nb34_rs['repeat'].unique())}")

assert RUN_SCORES_PATH.exists(), "no run_scores.csv from this notebook -- nothing was trained"
new_rs = pd.read_csv(RUN_SCORES_PATH)
print(f"this notebook:           {len(new_rs)} rows, "
      f"{new_rs[['repeat','fold','arm']].drop_duplicates().shape[0]} units, "
      f"{new_rs['arm'].nunique()} arms, repeats {sorted(new_rs['repeat'].unique())}")

# No (repeat, fold, arm) unit may appear in both sources -- that would mean something of
# notebook 34's was retrained here, which this notebook is not allowed to do.
k34 = set(map(tuple, nb34_rs[["repeat", "fold", "arm"]].drop_duplicates().values.tolist()))
knew = set(map(tuple, new_rs[["repeat", "fold", "arm"]].drop_duplicates().values.tolist()))
overlap = sorted(k34 & knew)
assert not overlap, f"these units exist in BOTH notebook 34 and this notebook: {overlap[:10]}"
print(f"\nno overlap between the two sources ({len(k34)} + {len(knew)} = {len(k34 | knew)} units)")

rs = pd.concat([nb34_rs, new_rs], ignore_index=True)
units = rs[["repeat", "fold", "arm"]].drop_duplicates()
COMPLETED = {arm: sorted(map(tuple, units.loc[units["arm"] == arm, ["repeat", "fold"]].values.tolist()))
             for arm in units["arm"].unique()}
ARMS_PRESENT = [a for a in ALL_ARMS if len(COMPLETED.get(a, [])) > 0]
PLAIN_UNITS = set(COMPLETED.get("plain", []))
UNITS_10 = {(r, f) for (r, f) in PLAIN_UNITS if r in NB34_REPEATS}   # notebook 34's own folds
UNITS_15 = set(PLAIN_UNITS)

print("\ncompleted (repeat, fold) units per arm:")
for arm in ALL_ARMS:
    u = COMPLETED.get(arm, [])
    print(f"  {arm:24s} {len(u):3d} units  repeats {sorted({r for r, _ in u})}")
print(f"\n10-fold set (notebook 34's repeats {NB34_REPEATS}): {len(UNITS_10)} folds")
print(f"15-fold set (adding repeat {TARGET_REPEAT}):        {len(UNITS_15)} folds")
if ABORT["aborted"]:
    print(f"\n*** NOTE: the run aborted at {ABORT['where']}: {ABORT['reason']}")
if DEADLINE_SKIPS:
    print(f"*** NOTE: {len(DEADLINE_SKIPS)} units skipped by the {WALL_DEADLINE_H}h wall guard")

notebook 34 (read-only): 500 rows, 100 units, 10 arms, repeats [0, 1]
this notebook:           325 rows, 65 units, 11 arms, repeats [0, 1, 2]

no overlap between the two sources (100 + 65 = 165 units)

completed (repeat, fold) units per arm:
  plain                     15 units  repeats [0, 1, 2]
  ecfp4_narrow__rf          15 units  repeats [0, 1, 2]
  ecfp4_narrow__xgboost     15 units  repeats [0, 1, 2]
  ecfp4_narrow__lightgbm    15 units  repeats [0, 1, 2]
  mordred_pca__rf           15 units  repeats [0, 1, 2]
  mordred_pca__xgboost      15 units  repeats [0, 1, 2]
  mordred_pca__lightgbm     15 units  repeats [0, 1, 2]
  single                    15 units  repeats [0, 1, 2]
  aid                       15 units  repeats [0, 1, 2]
  deadzone                  15 units  repeats [0, 1, 2]
  mae_only                  15 units  repeats [0, 1, 2]

10-fold set (notebook 34's repeats [0, 1]): 10 folds
15-fold set (adding repeat 2):        15 folds


In [55]:
# ---- Assemble the per-isoform WIDE out-of-fold tables at 15 folds.
# Notebook 34's own wide tables already hold its ten arms for repeats 0/1 in exactly this shape
# (and are tracked in git); this notebook's per-run long files supply repeat 2 for those arms
# plus MAE_ONLY for all three repeats. Reusing 34's assembled tables rather than re-deriving
# them from its per-run files keeps this side of the join byte-identical to what 34 reported.
new_parts = []
for arm in ARMS_PRESENT:
    for (r, f) in COMPLETED[arm]:
        p = OOF_DIR / f"{arm}__r{r}f{f}.csv"
        if p.exists():
            new_parts.append(pd.read_csv(p))
new_oof = pd.concat(new_parts, ignore_index=True) if new_parts else pd.DataFrame()
print(f"this notebook's per-run OOF rows: {len(new_oof)}")

OOF_WIDE = {}
for iso in ISOFORMS:
    ep = PIC50_COL[iso]
    nb34_wide = pd.read_csv(NB34_OUT / "oof_long" / f"oof_long_{iso}.csv")
    assert sorted(nb34_wide["repeat"].unique()) == NB34_REPEATS

    sub = new_oof[new_oof["isoform"] == iso] if len(new_oof) else new_oof
    if len(sub):
        new_wide = sub.pivot_table(index=["repeat", "fold", "inchikey"], columns="arm",
                                   values="y_pred", aggfunc="first").reset_index()
        new_wide.columns.name = None
    else:
        new_wide = pd.DataFrame(columns=["repeat", "fold", "inchikey"])

    # Repeat 2 rows are new; MAE_ONLY for repeats 0/1 is a new COLUMN on existing rows.
    new_r2 = new_wide[new_wide["repeat"] == TARGET_REPEAT] if len(new_wide) else new_wide
    new_r01 = new_wide[new_wide["repeat"].isin(NB34_REPEATS)] if len(new_wide) else new_wide
    # Only columns notebook 34 does NOT already have, and only those actually carrying data for
    # these repeats -- the pivot spans every arm that ran anywhere, so the rest are all-NaN
    # placeholders whose names would collide with notebook 34's own.
    r01_cols = [c for c in new_r01.columns
                if c not in ("repeat", "fold", "inchikey")
                and c not in nb34_wide.columns
                and new_r01[c].notna().any()]
    merged01 = nb34_wide.copy()
    if len(new_r01) and r01_cols:
        merged01 = merged01.merge(new_r01[["repeat", "fold", "inchikey"] + r01_cols],
                                  on=["repeat", "fold", "inchikey"], how="left")
        assert len(merged01) == len(nb34_wide), "the MAE_ONLY join changed notebook 34's row count"

    truth = curated[["inchikey", "Molecule_Name", ep, f"{ep}_conf_low", f"{ep}_conf_high"]].rename(
        columns={ep: "y_true", f"{ep}_conf_low": "conf_low", f"{ep}_conf_high": "conf_high"})
    if len(new_r2):
        r2 = new_r2.merge(truth, on="inchikey", how="left")
        r2 = r2[r2["y_true"].notna()]
        wide = pd.concat([merged01, r2], ignore_index=True)
    else:
        wide = merged01

    arm_cols = [c for c in ARMS_PRESENT if c in wide.columns]
    wide = wide[["repeat", "fold", "inchikey", "Molecule_Name"] + arm_cols
                + ["y_true", "conf_low", "conf_high"]]
    wide = wide[wide["y_true"].notna()].reset_index(drop=True)
    OOF_WIDE[iso] = wide
    wide.to_csv(OOF_DIR / f"oof_long_{iso}_15fold.csv", index=False)
    nulls = {a: int(wide[a].isna().sum()) for a in arm_cols}
    print(f"{iso}: {len(wide)} labelled rows, repeats {sorted(wide['repeat'].unique())}, "
          f"{len(arm_cols)} arms; nulls {nulls}")
    # Every arm ran every completed fold, so a populated column is the only correct outcome.
    # A silent NaN here empties the blend pool and the across-repeat floor without any error.
    assert all(c in wide.columns for c in ARMS_PRESENT), (
        f"{iso}: arm columns missing after the join -- {sorted(set(ARMS_PRESENT) - set(wide.columns))}")
    bad_nulls = {a: n for a, n in nulls.items() if n}
    assert not bad_nulls, (
        f"{iso}: arm columns carry NaN after assembly ({bad_nulls}). Every arm completed every "
        f"fold, so this means the join or the concat misaligned -- stopping rather than silently "
        f"analysing a hole.")
    _dupe_msg = f"{iso}: duplicate (repeat, fold, inchikey) rows after assembly"
    assert len(wide) == len(wide.drop_duplicates(["repeat", "fold", "inchikey"])), _dupe_msg

print(f"\nwrote 15-fold wide tables to {OOF_DIR} (notebook 34's own remain untouched)")

this notebook's per-run OOF rows: 255060


CYP1A2: 4236 labelled rows, repeats [0, 1, 2], 11 arms; nulls {'plain': 0, 'ecfp4_narrow__rf': 0, 'ecfp4_narrow__xgboost': 0, 'ecfp4_narrow__lightgbm': 0, 'mordred_pca__rf': 0, 'mordred_pca__xgboost': 0, 'mordred_pca__lightgbm': 0, 'single': 0, 'aid': 0, 'deadzone': 0, 'mae_only': 0}
CYP2C9: 3855 labelled rows, repeats [0, 1, 2], 11 arms; nulls {'plain': 0, 'ecfp4_narrow__rf': 0, 'ecfp4_narrow__xgboost': 0, 'ecfp4_narrow__lightgbm': 0, 'mordred_pca__rf': 0, 'mordred_pca__xgboost': 0, 'mordred_pca__lightgbm': 0, 'single': 0, 'aid': 0, 'deadzone': 0, 'mae_only': 0}


CYP2D6: 4479 labelled rows, repeats [0, 1, 2], 11 arms; nulls {'plain': 0, 'ecfp4_narrow__rf': 0, 'ecfp4_narrow__xgboost': 0, 'ecfp4_narrow__lightgbm': 0, 'mordred_pca__rf': 0, 'mordred_pca__xgboost': 0, 'mordred_pca__lightgbm': 0, 'single': 0, 'aid': 0, 'deadzone': 0, 'mae_only': 0}


CYP3A4: 7005 labelled rows, repeats [0, 1, 2], 11 arms; nulls {'plain': 0, 'ecfp4_narrow__rf': 0, 'ecfp4_narrow__xgboost': 0, 'ecfp4_narrow__lightgbm': 0, 'mordred_pca__rf': 0, 'mordred_pca__xgboost': 0, 'mordred_pca__lightgbm': 0, 'single': 0, 'aid': 0, 'deadzone': 0, 'mae_only': 0}

wrote 15-fold wide tables to /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/35_butina_repeat3/oof_long (notebook 34's own remain untouched)


### 4.1 Model comparison — 10 folds beside 15

Per isoform, per arm: out-of-fold ST-RAE as mean and across-fold spread, and a paired test against
PLAIN following this project's protocol — Levene's homogeneity branch **checked rather than
assumed** (Brown–Forsythe, median-centred), paired *t* when variances are homogeneous and Wilcoxon
signed-rank when they are not, Benjamini–Hochberg corrected across arms within each isoform.

One asymmetry to keep in view when reading the 10-fold column: for notebook `34`'s ten arms it is
literally `34`'s own published numbers, unchanged. For **MAE_ONLY** it is this notebook's own
repeats 0 and 1 — that arm did not exist on this partition before, so its 10-fold column is new
data rather than a reproduction.

In [56]:
def paired_test(x, y, label):
    # x, y are the same (repeat, fold) units in the same order. Returns the branch actually
    # taken, not a pre-chosen test.
    d = np.asarray(y, float) - np.asarray(x, float)
    n = len(d)
    if n < 3:
        return {"test": "none (n<3)", "stat": np.nan, "p": np.nan, "mean_diff": float(d.mean()) if n else np.nan,
                "levene_p": np.nan, "n": n, "se": np.nan, "sign_resolved": False,
                "ci_low": np.nan, "ci_high": np.nan, "dz": np.nan,
                "n_improving": int((d < 0).sum())}
    lev_stat, lev_p = stats.levene(np.asarray(x, float), np.asarray(y, float), center="median")
    if lev_p > 0.05:
        test = "paired t"
        stat, p = stats.ttest_rel(np.asarray(y, float), np.asarray(x, float))
    else:
        test = "Wilcoxon signed-rank"
        try:
            stat, p = stats.wilcoxon(np.asarray(y, float), np.asarray(x, float))
        except ValueError:
            stat, p = np.nan, 1.0
    sd = d.std(ddof=1)
    se = sd / np.sqrt(n)
    tcrit = stats.t.ppf(0.975, n - 1)
    return {"test": test, "stat": float(stat), "p": float(p), "mean_diff": float(d.mean()),
            "levene_p": float(lev_p), "n": n, "se": float(se),
            "sign_resolved": bool(abs(d.mean()) > se),
            "ci_low": float(d.mean() - tcrit * se),
            "ci_high": float(d.mean() + tcrit * se),
            "dz": float(d.mean() / sd) if sd > 0 else np.nan,
            "n_improving": int((d < 0).sum())}


In [57]:
METRICS_4_1 = ["ST-RAE", "MAE", "R2", "Spearman_R"]
rs_ep = rs[rs["Endpoint"].isin(REGRESSION_ENDPOINTS)].copy()
rs_ep["isoform"] = rs_ep["Endpoint"].map(EP_TO_ISO)

FOLD_SETS = {"10": UNITS_10, "15": UNITS_15}


def summarise(unit_set):
    out = []
    for iso in ISOFORMS:
        for arm in ARMS_PRESENT:
            sub = rs_ep[(rs_ep["isoform"] == iso) & (rs_ep["arm"] == arm)]
            sub = sub[[ (r, f) in unit_set for r, f in zip(sub["repeat"], sub["fold"]) ]]
            if not len(sub):
                continue
            row = {"isoform": iso, "arm": arm, "n_folds": len(sub)}
            for m in METRICS_4_1:
                row[f"{m}_mean"] = float(sub[m].mean())
                row[f"{m}_std"] = float(sub[m].std(ddof=1)) if len(sub) > 1 else np.nan
            out.append(row)
    return pd.DataFrame(out)


arm_summary_by_set = {k: summarise(v) for k, v in FOLD_SETS.items()}
arm_summary = arm_summary_by_set["15"]
for k, df in arm_summary_by_set.items():
    df.to_csv(OUT / f"arm_summary_{k}fold.csv", index=False)

cmp = arm_summary_by_set["10"][["isoform", "arm", "ST-RAE_mean", "ST-RAE_std", "n_folds"]].merge(
    arm_summary_by_set["15"][["isoform", "arm", "ST-RAE_mean", "ST-RAE_std", "n_folds"]],
    on=["isoform", "arm"], how="outer", suffixes=("_10", "_15"))
cmp["delta"] = cmp["ST-RAE_mean_15"] - cmp["ST-RAE_mean_10"]
cmp.to_csv(OUT / "arm_summary_10_vs_15.csv", index=False)

print("out-of-fold ST-RAE by arm: 10 folds (notebook 34) beside 15 folds (this notebook)")
for iso in ISOFORMS:
    print(f"\n{iso}")
    sub = cmp[cmp["isoform"] == iso].sort_values("ST-RAE_mean_15")
    print(f"  {'arm':24s} {'10-fold':>16s} {'15-fold':>16s} {'delta':>8s}")
    for _, r_ in sub.iterrows():
        m10 = (f"{r_['ST-RAE_mean_10']:.4f}+/-{r_['ST-RAE_std_10']:.4f}"
               if pd.notna(r_["ST-RAE_mean_10"]) else "n/a")
        print(f"  {r_['arm']:24s} {m10:>16s} "
              f"{r_['ST-RAE_mean_15']:.4f}+/-{r_['ST-RAE_std_15']:.4f} {r_['delta']:+8.4f}")
print(f"\nlargest |delta| between the two fold counts: {cmp['delta'].abs().max():.4f} ST-RAE")

out-of-fold ST-RAE by arm: 10 folds (notebook 34) beside 15 folds (this notebook)

CYP1A2
  arm                               10-fold          15-fold    delta
  deadzone                  0.7717+/-0.0493 0.7718+/-0.0476  +0.0001
  mae_only                  0.8250+/-0.0540 0.8172+/-0.0498  -0.0078
  aid                       0.8136+/-0.0556 0.8207+/-0.0586  +0.0071
  plain                     0.8544+/-0.0634 0.8546+/-0.0610  +0.0002
  ecfp4_narrow__rf          0.8671+/-0.0488 0.8689+/-0.0459  +0.0018
  ecfp4_narrow__lightgbm    0.8792+/-0.0447 0.8802+/-0.0489  +0.0010
  single                    0.8841+/-0.0640 0.8835+/-0.0706  -0.0006
  ecfp4_narrow__xgboost     0.9024+/-0.0545 0.9064+/-0.0536  +0.0040
  mordred_pca__rf           0.9411+/-0.0611 0.9407+/-0.0588  -0.0005
  mordred_pca__lightgbm     0.9520+/-0.0494 0.9502+/-0.0509  -0.0018
  mordred_pca__xgboost      0.9878+/-0.0417 0.9898+/-0.0498  +0.0019

CYP2C9
  arm                               10-fold          15-fold    delta
  d

In [58]:
def bh(pvals):
    p = np.asarray(pvals, float)
    ok = ~np.isnan(p)
    out = np.full_like(p, np.nan)
    if ok.sum() == 0:
        return out
    q = p[ok]
    order = np.argsort(q)
    m = len(q)
    adj = np.empty(m)
    prev = 1.0
    for rank in range(m - 1, -1, -1):
        val = q[order[rank]] * m / (rank + 1)
        prev = min(prev, val)
        adj[order[rank]] = prev
    out[ok] = np.clip(adj, 0, 1)
    return out


def run_paired(unit_set, metrics=("ST-RAE", "MAE")):
    rows = []
    for iso in ISOFORMS:
        for metric in metrics:
            for arm in ARMS_PRESENT:
                if arm == "plain":
                    continue
                shared = sorted((set(COMPLETED[arm]) & set(COMPLETED["plain"])) & unit_set)
                if len(shared) < 3:
                    continue
                key = lambda a: (rs_ep[(rs_ep["isoform"] == iso) & (rs_ep["arm"] == a)]
                                 .set_index(["repeat", "fold"])[metric])
                sp, sa = key("plain"), key(arm)
                x = [float(sp.loc[u]) for u in shared]
                y = [float(sa.loc[u]) for u in shared]
                res = paired_test(x, y, f"{iso}/{arm}")
                res.update({"isoform": iso, "metric": metric, "arm": arm,
                            "mean_plain": float(np.mean(x)), "mean_arm": float(np.mean(y))})
                rows.append(res)
    out = pd.DataFrame(rows)
    out["p_BH"] = np.nan
    for (iso, metric), grp in out.groupby(["isoform", "metric"]):
        out.loc[grp.index, "p_BH"] = bh(grp["p"].to_numpy())
    out["verdict"] = np.where(out["p_BH"] >= 0.05, "NOT DIFFERENT",
                     np.where(out["mean_diff"] < 0, "SIGNIFICANTLY BETTER", "SIGNIFICANTLY WORSE"))
    return out


paired_by_set = {k: run_paired(v) for k, v in FOLD_SETS.items()}
paired = paired_by_set["15"]
for k, df in paired_by_set.items():
    df.to_csv(OUT / f"paired_tests_vs_plain_{k}fold.csv", index=False)

st15 = paired_by_set["15"][paired_by_set["15"]["metric"] == "ST-RAE"]
st10 = paired_by_set["10"][paired_by_set["10"]["metric"] == "ST-RAE"]
vcmp = st10[["isoform", "arm", "mean_diff", "p_BH", "verdict"]].merge(
    st15[["isoform", "arm", "mean_diff", "ci_low", "ci_high", "p_BH", "verdict", "n"]],
    on=["isoform", "arm"], how="outer", suffixes=("_10", "_15"))
vcmp["verdict_changed"] = vcmp["verdict_10"] != vcmp["verdict_15"]
vcmp.to_csv(OUT / "verdict_10_vs_15.csv", index=False)

print("=== paired tests vs PLAIN, ST-RAE: verdict at 10 folds vs 15 folds ===")
print("(BH-corrected across arms within isoform, at each fold count separately)")
cols = ["isoform", "arm", "mean_diff_10", "verdict_10", "mean_diff_15", "ci_low", "ci_high",
        "p_BH_15", "verdict_15", "verdict_changed"]
print(vcmp[cols].round(4).to_string(index=False))

changed = vcmp[vcmp["verdict_changed"] & vcmp["verdict_10"].notna()]
print(f"\nverdicts that CHANGED between 10 and 15 folds: {len(changed)} of "
      f"{int(vcmp['verdict_10'].notna().sum())} comparable (arm, isoform) cells")
if len(changed):
    changed = changed.copy()
    changed["sign_flipped"] = np.sign(changed["mean_diff_10"]) != np.sign(changed["mean_diff_15"])
    changed["shrank_toward_zero"] = changed["mean_diff_15"].abs() < changed["mean_diff_10"].abs()
    print(changed[["isoform", "arm", "mean_diff_10", "verdict_10", "mean_diff_15", "verdict_15",
                   "p_BH_10", "p_BH_15", "sign_flipped"]].round(4).to_string(index=False))
    n_rev = int(changed["sign_flipped"].sum())
    print("")
    if n_rev == 0:
        print(f"NONE of these {len(changed)} is a reversal: every one keeps the SAME sign of")
        print("mean_diff and merely crosses the p<0.05 line. What changed is statistical")
        print("resolution, not direction -- which is what adding folds uniformly should do.")
        gained = changed[changed["verdict_10"] == "NOT DIFFERENT"]
        lost = changed[changed["verdict_15"] == "NOT DIFFERENT"]
        print(f"  gained significance at 15 folds: {len(gained)} "
              f"({', '.join(gained['isoform'] + '/' + gained['arm']) if len(gained) else 'none'})")
        print(f"  lost significance at 15 folds:   {len(lost)} "
              f"({', '.join(lost['isoform'] + '/' + lost['arm']) if len(lost) else 'none'})")
        if len(lost):
            print("  A lost verdict is NOT a refutation: the effect is still pointing the same way,")
            print("  it is simply no longer resolved against this pool's BH correction. Where that")
            print("  happens to one of notebook 34's own headline findings it is named here rather")
            print("  than left for the reader to notice.")
    else:
        print(f"*** {n_rev} of these {len(changed)} genuinely REVERSED sign between 10 and 15 folds.")
        print(changed[changed["sign_flipped"]][["isoform", "arm", "mean_diff_10",
                                                "mean_diff_15"]].round(4).to_string(index=False))
        print("A sign reversal on a uniformly-added repeat is a real instability in the estimate,")
        print("not a power change, and should be read as such.")
else:
    print("None. Adding the third repeat tightened the estimates without moving any verdict.")

=== paired tests vs PLAIN, ST-RAE: verdict at 10 folds vs 15 folds ===
(BH-corrected across arms within isoform, at each fold count separately)
isoform                    arm  mean_diff_10           verdict_10  mean_diff_15  ci_low  ci_high  p_BH_15           verdict_15  verdict_changed
 CYP1A2                    aid       -0.0407        NOT DIFFERENT       -0.0339 -0.0590  -0.0087   0.0198 SIGNIFICANTLY BETTER             True
 CYP1A2               deadzone       -0.0827 SIGNIFICANTLY BETTER       -0.0828 -0.1080  -0.0575   0.0000 SIGNIFICANTLY BETTER            False
 CYP1A2 ecfp4_narrow__lightgbm        0.0249        NOT DIFFERENT        0.0257 -0.0048   0.0561   0.1152        NOT DIFFERENT            False
 CYP1A2       ecfp4_narrow__rf        0.0127        NOT DIFFERENT        0.0143 -0.0188   0.0475   0.3696        NOT DIFFERENT            False
 CYP1A2  ecfp4_narrow__xgboost        0.0481        NOT DIFFERENT        0.0519  0.0109   0.0928   0.0240  SIGNIFICANTLY WORSE          

### 4.1b The MAE_ONLY decomposition — what is DEADZONE's gain actually made of?

Notebook `34` found DEADZONE significantly better than PLAIN on all four isoforms, but DEADZONE
changes the loss **and** the target at once. MAE_ONLY changes only the loss. On the same 15 folds:

- **loss-alone share** = (PLAIN − MAE_ONLY) / (PLAIN − DEADZONE)
- **clipping-on-top share** = (MAE_ONLY − DEADZONE) / (PLAIN − DEADZONE)

A share near 1.0 for loss-alone means the clipping is doing nothing and the whole effect is
absolute-error fitting; near 0 means the clipping is carrying it. Shares are only meaningful where
DEADZONE actually beats PLAIN, so the denominator's sign is checked and reported rather than
assumed. Notebook `31` found this split varies sharply by isoform on the random partition
(CYP2C9 almost entirely clipping, CYP2D6 almost entirely the loss switch) — recomputed here on
this project's own figures rather than quoted.

In [59]:
rows = []
for iso in ISOFORMS:
    def mean_of(arm, unit_set):
        sub = rs_ep[(rs_ep["isoform"] == iso) & (rs_ep["arm"] == arm)]
        sub = sub[[ (r, f) in unit_set for r, f in zip(sub["repeat"], sub["fold"]) ]]
        return float(sub["ST-RAE"].mean()) if len(sub) else np.nan
    for label, uset in FOLD_SETS.items():
        p_, m_, d_ = mean_of("plain", uset), mean_of("mae_only", uset), mean_of("deadzone", uset)
        total = p_ - d_
        rows.append({"isoform": iso, "fold_set": f"{label}-fold", "plain": p_, "mae_only": m_,
                     "deadzone": d_, "total_gain_plain_minus_deadzone": total,
                     "loss_alone": p_ - m_, "clipping_on_top": m_ - d_,
                     "loss_alone_share": (p_ - m_) / total if total and total > 0 else np.nan,
                     "clipping_share": (m_ - d_) / total if total and total > 0 else np.nan,
                     "deadzone_beats_plain": bool(total > 0) if pd.notna(total) else None})
decomp = pd.DataFrame(rows)
decomp.to_csv(OUT / "mae_only_decomposition.csv", index=False)
print(decomp.round(4).to_string(index=False))

print("\n--- reading, per isoform, on the 15-fold set ---")
for iso in ISOFORMS:
    r_ = decomp[(decomp["isoform"] == iso) & (decomp["fold_set"] == "15-fold")].iloc[0]
    if not r_["deadzone_beats_plain"]:
        print(f"  {iso}: DEADZONE does not beat PLAIN here (total {r_['total_gain_plain_minus_deadzone']:+.4f}) "
              f"-- shares are undefined and not reported.")
        continue
    ls, cs = r_["loss_alone_share"], r_["clipping_share"]
    if ls > 0.7:
        verdict = "mostly the LOSS FUNCTION -- clipping adds little"
    elif ls < 0.3:
        verdict = "mostly the CLIPPING -- the loss switch alone does little"
    else:
        verdict = "genuinely split between the two"
    extra = ""
    if ls < 0:
        extra = " (loss-alone is NEGATIVE: mae_only is worse than plain, so clipping does more than the whole gain)"
    print(f"  {iso}: total {r_['total_gain_plain_minus_deadzone']:+.4f} = loss-alone {r_['loss_alone']:+.4f} "
          f"({ls:.0%}) + clipping {r_['clipping_on_top']:+.4f} ({cs:.0%}) -> {verdict}{extra}")

# MAE_ONLY carries none of DEADZONE's fold-crosstalk exposure (its target is the literal label),
# so this is also a check on how much of 34's DEADZONE result could be crosstalk-inflated.
mo = st15[st15["arm"] == "mae_only"][["isoform", "mean_diff", "p_BH", "verdict", "n"]]
print("\nMAE_ONLY vs PLAIN at 15 folds (no fold-crosstalk exposure -- its target is the raw label):")
print(mo.round(4).to_string(index=False))

isoform fold_set  plain  mae_only  deadzone  total_gain_plain_minus_deadzone  loss_alone  clipping_on_top  loss_alone_share  clipping_share  deadzone_beats_plain
 CYP1A2  10-fold 0.8544    0.8250    0.7717                           0.0827      0.0294           0.0533            0.3554          0.6446                  True
 CYP1A2  15-fold 0.8546    0.8172    0.7718                           0.0828      0.0374           0.0453            0.4521          0.5479                  True
 CYP2C9  10-fold 0.6455    0.6456    0.5625                           0.0831     -0.0001           0.0831           -0.0007          1.0007                  True
 CYP2C9  15-fold 0.6576    0.6415    0.5676                           0.0900      0.0162           0.0738            0.1796          0.8204                  True
 CYP2D6  10-fold 1.0179    0.9199    0.8850                           0.1329      0.0980           0.0349            0.7372          0.2628                  True
 CYP2D6  15-fold 1.0111    0

### 4.2 The split question, at 15 folds

Each arm's Butina result set beside its random-partition result, all read read-only from
`outputs/05_cv_comparison/` (PLAIN and the six tree configs, 25 folds),
`outputs/29_cv_confirmation/` (AID, 25 folds) and `outputs/31_deadzone/` (DEADZONE **and
MAE_ONLY**, one fold x three seeds). A cell counts as a sign flip only where **both** partitions
resolve a sign — |mean difference| exceeding its own standard error — the correction notebook `34`
made after one apparent flip turned out to be notebook `05`'s own +0.000019 ST-RAE near-tie.

In [60]:
# ---- Random-partition reference differences vs. that source's own PLAIN/baseline arm.
def nb05_strae(config):
    rows = []
    for r in range(5):
        for f in range(5):
            p = NB05_OUT / "scores" / f"{config}__repeat{r}_fold{f}.csv"
            if not p.exists():
                continue
            d = pd.read_csv(p)
            g = d.groupby("Endpoint")["ST-RAE"].mean()
            for ep, v in g.items():
                if ep in REGRESSION_ENDPOINTS:
                    rows.append({"repeat": r, "fold": f, "isoform": EP_TO_ISO[ep], "ST-RAE": float(v)})
    return pd.DataFrame(rows)


ref_rows = []
nb05_plain = nb05_strae("chemprop_chemeleoninit")
assert len(nb05_plain) == 100, f"expected 100 (fold, isoform) cells from notebook 05, got {len(nb05_plain)}"
for cfg in TREE_ARMS:
    t = nb05_strae(cfg)
    if not len(t):
        print(f"  no notebook 05 scores for {cfg} -- skipped")
        continue
    for iso in ISOFORMS:
        a = nb05_plain[nb05_plain["isoform"] == iso].set_index(["repeat", "fold"])["ST-RAE"]
        b = t[t["isoform"] == iso].set_index(["repeat", "fold"])["ST-RAE"]
        shared = sorted(set(a.index) & set(b.index))
        dd = (b.loc[shared] - a.loc[shared])
        se_ = float(dd.std(ddof=1) / np.sqrt(len(shared))) if len(shared) > 1 else np.nan
        ref_rows.append({"arm": cfg, "isoform": iso, "source": "nb05 (25 folds)",
                         "random_plain": float(a.loc[shared].mean()),
                         "random_arm": float(b.loc[shared].mean()),
                         "random_diff": float(dd.mean()), "random_se": se_,
                         "random_sign_resolved": bool(abs(dd.mean()) > se_) if se_ == se_ else False,
                         "random_n": len(shared)})

nb29 = pd.read_csv(NB29_OUT / "run_scores.csv")
nb29["isoform"] = nb29["Endpoint"].map(EP_TO_ISO)
for iso in ISOFORMS:
    a = nb29[(nb29["arm"] == "baseline") & (nb29["isoform"] == iso)].set_index(["repeat", "fold"])["ST-RAE"]
    b = nb29[(nb29["arm"] == "aid") & (nb29["isoform"] == iso)].set_index(["repeat", "fold"])["ST-RAE"]
    shared = sorted(set(a.index) & set(b.index))
    dd = (b.loc[shared] - a.loc[shared])
    se_ = float(dd.std(ddof=1) / np.sqrt(len(shared)))
    ref_rows.append({"arm": "aid", "isoform": iso, "source": "nb29 (25 folds)",
                     "random_plain": float(a.loc[shared].mean()), "random_arm": float(b.loc[shared].mean()),
                     "random_diff": float(dd.mean()), "random_se": se_,
                     "random_sign_resolved": bool(abs(dd.mean()) > se_), "random_n": len(shared)})

nb31 = pd.read_csv(NB31_OUT / "run_scores.csv")
nb31["isoform"] = nb31["Endpoint"].map(EP_TO_ISO)
nb31f = nb31[nb31["Endpoint"].isin(REGRESSION_ENDPOINTS)].groupby(
    ["arm", "seed", "isoform"], as_index=False)["ST-RAE"].mean()
# Notebook 31 carries BOTH deadzone and mae_only against its own baseline -- the only
# random-partition reference that exists for either arm.
for nb31_arm in ["deadzone", "mae_only"]:
    if nb31_arm not in set(nb31f["arm"]):
        print(f"  notebook 31 has no {nb31_arm} arm -- no random reference for it")
        continue
    for iso in ISOFORMS:
        a = nb31f[(nb31f["arm"] == "baseline") & (nb31f["isoform"] == iso)].set_index("seed")["ST-RAE"]
        b = nb31f[(nb31f["arm"] == nb31_arm) & (nb31f["isoform"] == iso)].set_index("seed")["ST-RAE"]
        shared = sorted(set(a.index) & set(b.index))
        if not shared:
            continue
        dd = (b.loc[shared] - a.loc[shared])
        se_ = float(dd.std(ddof=1) / np.sqrt(len(shared))) if len(shared) > 1 else np.nan
        ref_rows.append({"arm": nb31_arm, "isoform": iso, "source": "nb31 (1 fold x 3 seeds)",
                         "random_plain": float(a.loc[shared].mean()), "random_arm": float(b.loc[shared].mean()),
                         "random_diff": float(dd.mean()), "random_se": se_,
                         "random_sign_resolved": bool(abs(dd.mean()) > se_) if se_ == se_ else False,
                         "random_n": len(shared)})

ref = pd.DataFrame(ref_rows)
print(f"random-partition reference differences assembled: {len(ref)} (arm, isoform) cells")
print(ref.groupby(["arm", "source"]).size().to_string())

random-partition reference differences assembled: 36 (arm, isoform) cells
arm                     source                 
aid                     nb29 (25 folds)            4
deadzone                nb31 (1 fold x 3 seeds)    4
ecfp4_narrow__lightgbm  nb05 (25 folds)            4
ecfp4_narrow__rf        nb05 (25 folds)            4
ecfp4_narrow__xgboost   nb05 (25 folds)            4
mae_only                nb31 (1 fold x 3 seeds)    4
mordred_pca__lightgbm   nb05 (25 folds)            4
mordred_pca__rf         nb05 (25 folds)            4
mordred_pca__xgboost    nb05 (25 folds)            4


In [61]:
# ---- Sign agreement and ordering agreement between the two partitions.
but = paired[paired["metric"] == "ST-RAE"][["isoform", "arm", "mean_plain", "mean_arm",
                                            "mean_diff", "se", "sign_resolved", "p_BH",
                                            "verdict", "n"]].rename(
    columns={"mean_plain": "butina_plain", "mean_arm": "butina_arm",
             "mean_diff": "butina_diff", "se": "butina_se",
             "sign_resolved": "butina_sign_resolved", "p_BH": "butina_p_BH",
             "verdict": "butina_verdict", "n": "butina_n"})
agree = but.merge(ref, on=["arm", "isoform"], how="left")

# A sign-agreement test is only meaningful where BOTH partitions actually resolved a sign.
# This is not a cosmetic guard: on CYP1A2, notebook 05's own 25-fold difference between
# ecfp4_narrow__rf and chemprop_chemeleoninit is +0.000019 ST-RAE -- a genuine near-tie,
# verified directly against its stored per-fold scores. Comparing signs there manufactures a
# "flip" out of arithmetic noise. "Resolved" means |mean difference| exceeds its own standard
# error across the folds it was measured on -- a data-derived bar, not an invented epsilon.
# .eq(True) rather than .fillna(False).astype(bool): identical result on an object column
# holding True/False/NA, without pandas' deprecated silent downcast.
agree["both_resolved"] = (agree["random_diff"].notna()
                          & agree["butina_sign_resolved"].eq(True)
                          & agree["random_sign_resolved"].eq(True))
_same = np.sign(agree["butina_diff"]) == np.sign(agree["random_diff"])
agree["sign_agrees"] = np.where(agree["both_resolved"], _same, pd.NA)
agree["flip"] = ["" if pd.isna(v) else ("" if v else "FLIP") for v in agree["sign_agrees"]]
agree["status"] = np.where(agree["random_diff"].isna(), "no random reference",
                  np.where(~agree["both_resolved"], "sign not resolved",
                  np.where(agree["sign_agrees"] == True, "agrees", "FLIP")))
agree.to_csv(OUT / "partition_agreement.csv", index=False)

print("=== SIGN of each arm's ST-RAE difference against PLAIN, under both partitions ===")
print("(negative = better than PLAIN. A cell counts as a FLIP only when BOTH partitions")
print(" resolved a sign, i.e. |difference| > its own standard error on each side.)")
print(agree[["isoform", "arm", "source", "butina_diff", "butina_se", "random_diff", "random_se",
             "status", "butina_verdict"]].round(4).to_string(index=False))

res = agree[agree["both_resolved"]]
unres = agree[(agree["random_diff"].notna()) & (~agree["both_resolved"])]
print("")
print(f"sign agreement, over the {len(res)} (arm, isoform) cells where both partitions resolved a")
print(f"sign: {int((res['sign_agrees'] == True).sum())} agree, "
      f"{int((res['sign_agrees'] == False).sum())} flip.")
print(f"{len(unres)} further comparable cells had an UNRESOLVED sign on one or both sides and are")
print("excluded rather than counted either way:")
if len(unres):
    print(unres[["isoform", "arm", "butina_diff", "butina_se", "random_diff", "random_se"]]
          .round(5).to_string(index=False))
missing_ref = sorted(set(agree.loc[agree['random_diff'].isna(), 'arm']))
if missing_ref:
    print(f"no random-partition reference exists for: {missing_ref} -- reported as unavailable, "
          f"not compared.")

print("")
print("--- the AID question specifically ---")
nb33_tests = pd.read_csv(NB33_OUT / "part3_paired_tests.csv")
aid_rows = agree[agree["arm"] == "aid"]
if not len(aid_rows):
    print("The AID arm has fewer than 3 completed (repeat, fold) units paired with PLAIN, so no")
    print("paired difference exists for it yet and this comparison cannot be made. Reported as")
    print("unavailable rather than as a count of zero flips.")
else:
    # Notebook 33's "flips sign on three of four isoforms" is reconciled here against BOTH
    # possible bases rather than one being declared wrong. Against its OWN 5-fold random slice
    # the count is 4 of 4; against notebook 29's full 25-fold differences it is 3 of 4. The one
    # cell that separates the two bases is CYP2C9, whose 25-fold random difference is not
    # sign-resolved in the first place -- so "3 of 4" is right on the 25-fold basis, and the
    # apparent discrepancy is a property of that unresolved cell, not an error in either source.
    rows33 = []
    for iso in ISOFORMS:
        b33 = float(nb33_tests[(nb33_tests["partition"] == "BUTINA") &
                               (nb33_tests["isoform"] == iso)]["mean_diff"].iloc[0])
        r33 = float(nb33_tests[(nb33_tests["partition"] == "RANDOM") &
                               (nb33_tests["isoform"] == iso)]["mean_diff"].iloc[0])
        ar = aid_rows[aid_rows["isoform"] == iso]
        r29 = float(ar["random_diff"].iloc[0]) if len(ar) else np.nan
        se29 = float(ar["random_se"].iloc[0]) if len(ar) else np.nan
        bhere = float(ar["butina_diff"].iloc[0]) if len(ar) else np.nan
        rows33.append({"isoform": iso, "nb33_butina_5f": b33, "nb33_random_5f": r33,
                       "nb29_random_25f": r29, "nb29_random_se": se29,
                       "nb29_sign_resolved": bool(abs(r29) > se29) if se29 == se29 else False,
                       "flip_vs_nb33_own_random": bool(np.sign(b33) != np.sign(r33)),
                       "flip_vs_nb29_25fold": bool(np.sign(b33) != np.sign(r29)),
                       "this_notebook_butina": bhere,
                       "this_notebook_status": ar["status"].iloc[0] if len(ar) else "n/a"})
    aid_recon = pd.DataFrame(rows33)
    aid_recon.to_csv(OUT / "aid_flip_reconciliation.csv", index=False)
    print(aid_recon.round(4).to_string(index=False))
    n_own = int(aid_recon["flip_vs_nb33_own_random"].sum())
    n_29 = int(aid_recon["flip_vs_nb29_25fold"].sum())
    print("")
    print(f"notebook 33's OWN Butina differences, recomputed from its saved part3_paired_tests.csv:")
    print(f"  {n_own} of 4 sign changes against its own 5-fold random slice;")
    print(f"  {n_29} of 4 against notebook 29's full 25-fold random differences.")
    print(f"  Its prose says 'three of four'. That is exactly right on the 25-fold basis. The cell")
    print(f"  that separates the two counts is CYP2C9, whose 25-fold random difference is not")
    print(f"  sign-resolved (|diff| < its own SE), so it never had a direction to reverse.")
    print("")
    print("THIS notebook's repeated design, per isoform:")
    for _, row in aid_recon.iterrows():
        print(f"  {row['isoform']}: nb29 random(25f) {row['nb29_random_25f']:+.4f} | "
              f"nb33 butina(5f) {row['nb33_butina_5f']:+.4f} | "
              f"this notebook butina {row['this_notebook_butina']:+.4f} -> {row['this_notebook_status']}")
    aid_flips_here = int((aid_rows["sign_agrees"] == False).sum())
    aid_resolved_here = int(aid_rows["both_resolved"].sum())
    print("")
    print(f"AID sign flips in THIS notebook's repeated design: {aid_flips_here} of "
          f"{aid_resolved_here} sign-resolved isoforms "
          f"({len(aid_rows) - aid_resolved_here} unresolved, excluded).")
    if aid_resolved_here == 0:
        print("  -> No isoform resolved a sign on both sides, so this design can neither confirm")
        print("     nor overturn notebook 33's flip. Reported as inconclusive, not as agreement.")
    elif aid_flips_here >= 2:
        print("  -> the repeated design CONFIRMS notebook 33's flip on the isoforms it can resolve.")
    elif aid_flips_here == 0:
        print("  -> the repeated design OVERTURNS notebook 33's flip on every isoform it resolves.")
    else:
        print("  -> the repeated design PARTLY reproduces notebook 33's flip.")
    print("  A flip at screening tier reversing again here is a legitimate outcome, not an error:")
    print("  notebook 33's five-fold, one-partition result sits at the same evidentiary tier as")
    print("  notebook 28's AID screen, which notebook 29 then overturned.")

=== SIGN of each arm's ST-RAE difference against PLAIN, under both partitions ===
(negative = better than PLAIN. A cell counts as a FLIP only when BOTH partitions
 resolved a sign, i.e. |difference| > its own standard error on each side.)
isoform                    arm                  source  butina_diff  butina_se  random_diff  random_se              status       butina_verdict
 CYP1A2       ecfp4_narrow__rf         nb05 (25 folds)       0.0143     0.0155       0.0000     0.0095   sign not resolved        NOT DIFFERENT
 CYP1A2  ecfp4_narrow__xgboost         nb05 (25 folds)       0.0519     0.0191       0.0483     0.0097              agrees  SIGNIFICANTLY WORSE
 CYP1A2 ecfp4_narrow__lightgbm         nb05 (25 folds)       0.0257     0.0142       0.0066     0.0104   sign not resolved        NOT DIFFERENT
 CYP1A2        mordred_pca__rf         nb05 (25 folds)       0.0861     0.0180       0.0723     0.0091              agrees  SIGNIFICANTLY WORSE
 CYP1A2   mordred_pca__xgboost         nb

In [62]:
# ---- Ordering agreement: do the two partitions rank the arms the same way?
order_rows = []
for iso in ISOFORMS:
    b = arm_summary[(arm_summary["isoform"] == iso)].set_index("arm")["ST-RAE_mean"]
    # Random-partition ORDER is only defined for arms with a reference; PLAIN is included via
    # each source's own baseline, and the tree arms + PLAIN all come from notebook 05, so they
    # are directly comparable on one scale. AID/DEADZONE come from other notebooks, so their
    # absolute level is not on nb05's scale -- they are placed by their own diff-vs-baseline
    # added to nb05's PLAIN level, stated explicitly as a construction rather than a measurement.
    r_level = {}
    nb05_plain_iso = float(nb05_plain[nb05_plain["isoform"] == iso]["ST-RAE"].mean())
    r_level["plain"] = nb05_plain_iso
    for _, row in ref[ref["isoform"] == iso].iterrows():
        if row["source"].startswith("nb05"):
            r_level[row["arm"]] = float(row["random_arm"])
        else:
            r_level[row["arm"]] = nb05_plain_iso + float(row["random_diff"])
    common = [a for a in b.index if a in r_level]
    if len(common) >= 3:
        rho, p = stats.spearmanr([b[a] for a in common], [r_level[a] for a in common])
    else:
        rho, p = np.nan, np.nan
    order_rows.append({"isoform": iso, "n_arms_compared": len(common), "spearman_rho": rho, "p": p,
                       "butina_best": b[common].idxmin() if common else None,
                       "random_best": min(common, key=lambda a: r_level[a]) if common else None})
ordering = pd.DataFrame(order_rows)
ordering.to_csv(OUT / "ordering_agreement.csv", index=False)
print("ORDERING agreement between partitions (Spearman rank correlation of arm mean ST-RAE):")
print(ordering.round(4).to_string(index=False))
print("\\nWeak evidence by construction: the rank correlation is computed over a handful of arms,")
print("and the AID/DEADZONE random-partition levels are constructed from their own")
print("diff-vs-baseline rather than measured on notebook 05's scale. Read the SIGN table above")
print("as the primary evidence and this as a summary.")

ORDERING agreement between partitions (Spearman rank correlation of arm mean ST-RAE):
isoform  n_arms_compared  spearman_rho      p butina_best random_best
 CYP1A2               10        0.9758 0.0000    deadzone    deadzone
 CYP2C9               10        0.9636 0.0000    deadzone    deadzone
 CYP2D6               10        0.8788 0.0008    deadzone    deadzone
 CYP3A4               10        0.9879 0.0000    deadzone    deadzone
\nWeak evidence by construction: the rank correlation is computed over a handful of arms,
and the AID/DEADZONE random-partition levels are constructed from their own
diff-vs-baseline rather than measured on notebook 05's scale. Read the SIGN table above
as the primary evidence and this as a summary.


### 4.3 Decorrelation — why TREES and SINGLE are in this run at all

Error correlation means correlation of **residuals** (prediction minus truth), not of raw
predictions — two models can track the same true values closely and still make usefully different
errors (Dietterich, 2000; this project's own rule). Computed per isoform, pooled across all
completed folds, **restricted to labelled rows** (notebook 31's version came back silently all-NaN
from sparse per-isoform labelling; the restriction is applied once at the source in 4.0).

**The floor** is PLAIN-against-PLAIN across repeats: the same compound's residual under two
different cluster-to-fold assignments and two different training seeds. With three repeats it now
rests on **3 repeat-pairs instead of notebook `34`'s single pair** — the main thing the extra
repeat buys this section. That is the correlation two
runs of the *same* recipe produce, so an arm at or above it is not decorrelated in any useful sense.

Reference bands: notebook 24 measured this project's existing 11-config pool at **0.792–0.908**;
notebook 29 measured AID against baseline at **0.850–0.933**. Both verified against their own saved
files below rather than quoted.

In [63]:
# ---- Verify the two quoted reference bands against their own saved files, not the prose.
# Notebook 24's documented band (0.792-0.908) is the range across isoforms of the MEAN
# off-diagonal pairwise error correlation -- not the range of individual pairs, which its own
# text separately reports as reaching 0.965. Recomputed here on that definition.
nb24_long = pd.read_csv(NB24_OUT / "error_correlation_long.csv")
off = nb24_long[nb24_long["config_1"] != nb24_long["config_2"]]
nb24_means = off.groupby("isoform")["error_correlation"].mean()
NB24_BAND_LO, NB24_BAND_HI = float(nb24_means.min()), float(nb24_means.max())
print("notebook 24, recomputed from its own error_correlation_long.csv:")
print(f"  mean off-diagonal error correlation per isoform: "
      + ", ".join(f"{i_}={v:.3f}" for i_, v in nb24_means.items()))
print(f"  band across isoforms: {NB24_BAND_LO:.3f}-{NB24_BAND_HI:.3f}   "
      f"(documented: 0.792-0.908)")
print(f"  individual-pair max (its own separately-reported figure): "
      f"{off['error_correlation'].max():.3f} (documented: 0.965)")
_ok24 = abs(NB24_BAND_LO - 0.792) < 0.005 and abs(NB24_BAND_HI - 0.908) < 0.005
print(f"  -> {'CONFIRMED' if _ok24 else 'DOES NOT MATCH -- the recomputed values are used, not the quoted ones'}")

nb29_corr = pd.read_csv(NB29_OUT / "error_correlation.csv")
NB29_LO = float(nb29_corr["aid_vs_baseline_mean"].min())
NB29_HI = float(nb29_corr["aid_vs_baseline_mean"].max())
NB29_FLOOR_LO = float(nb29_corr["baseline_fold_floor_mean"].min())
NB29_FLOOR_HI = float(nb29_corr["baseline_fold_floor_mean"].max())
print("\\nnotebook 29, recomputed from its own error_correlation.csv:")
print(f"  AID vs baseline: {NB29_LO:.3f}-{NB29_HI:.3f}   (documented: 0.850-0.933)")
print(f"  its own baseline fold-to-fold floor: {NB29_FLOOR_LO:.3f}-{NB29_FLOOR_HI:.3f} "
      f"(documented: 0.867-0.920)")
_ok29 = abs(NB29_LO - 0.850) < 0.005 and abs(NB29_HI - 0.933) < 0.005
print(f"  -> {'CONFIRMED' if _ok29 else 'PARTIAL -- the recomputed values are used, not the quoted ones'}")
print(nb29_corr.round(4).to_string(index=False))

notebook 24, recomputed from its own error_correlation_long.csv:
  mean off-diagonal error correlation per isoform: CYP1A2=0.884, CYP2C9=0.842, CYP2D6=0.908, CYP3A4=0.792
  band across isoforms: 0.792-0.908   (documented: 0.792-0.908)
  individual-pair max (its own separately-reported figure): 0.965 (documented: 0.965)
  -> CONFIRMED
\nnotebook 29, recomputed from its own error_correlation.csv:
  AID vs baseline: 0.850-0.933   (documented: 0.850-0.933)
  its own baseline fold-to-fold floor: 0.867-0.919 (documented: 0.867-0.920)
  -> CONFIRMED
isoform  n_folds  aid_vs_baseline_mean  aid_vs_baseline_std  baseline_fold_floor_mean  baseline_fold_floor_std  n_floor_pairs
 CYP1A2       25                0.9022               0.0228                    0.9044                   0.0095             10
 CYP2C9       25                0.8806               0.0238                    0.8913                   0.0081             10
 CYP2D6       25                0.9334               0.0244              

In [64]:
# ---- Pairwise arm error correlation, per isoform, pooled across folds.
DECORR_BAR = 0.85   # the brief's own stated bar: "if TREES come back below roughly 0.85"
corr_mats, floor_rows, decorr_rows = {}, [], []

for iso in ISOFORMS:
    w = OOF_WIDE[iso]
    arm_cols = [c for c in ARMS_PRESENT if c in w.columns]
    resid = w[["repeat", "fold", "inchikey"]].copy()
    for a in arm_cols:
        resid[a] = w[a] - w["y_true"]
    # Pairwise-complete Pearson. Every row here is labelled by construction; a NaN can still
    # appear if an arm has fewer completed folds than another, so pairs are intersected.
    M = pd.DataFrame(index=arm_cols, columns=arm_cols, dtype=float)
    for a in arm_cols:
        for b in arm_cols:
            sel = resid[a].notna() & resid[b].notna()
            M.loc[a, b] = (np.corrcoef(resid.loc[sel, a], resid.loc[sel, b])[0, 1]
                           if sel.sum() > 2 else np.nan)
    corr_mats[iso] = M
    M.to_csv(OUT / f"error_correlation_{iso}.csv")

    # FLOOR: PLAIN vs PLAIN across repeat pairs (same compound, different assignment + seed).
    pl = resid[["repeat", "inchikey", "plain"]].dropna()
    reps = sorted(pl["repeat"].unique())
    fl = []
    for r1, r2 in itertools.combinations(reps, 2):
        m = (pl[pl["repeat"] == r1][["inchikey", "plain"]]
             .merge(pl[pl["repeat"] == r2][["inchikey", "plain"]], on="inchikey",
                    suffixes=("_1", "_2")))
        if len(m) > 2:
            fl.append(float(np.corrcoef(m["plain_1"], m["plain_2"])[0, 1]))
    floor = float(np.mean(fl)) if fl else np.nan
    floor_rows.append({"isoform": iso, "n_repeat_pairs": len(fl), "plain_vs_plain_mean": floor,
                       "plain_vs_plain_min": float(np.min(fl)) if fl else np.nan,
                       "plain_vs_plain_max": float(np.max(fl)) if fl else np.nan})

    for a in arm_cols:
        if a == "plain":
            continue
        c = float(M.loc[a, "plain"])
        decorr_rows.append({"isoform": iso, "arm": a, "corr_vs_plain": c,
                            "plain_floor": floor,
                            "below_floor": bool(c < floor) if not np.isnan(floor) else None,
                            "below_bar_0.85": bool(c < DECORR_BAR),
                            "below_nb24_band": bool(c < NB24_BAND_LO),
                            "decorrelated": bool(c < DECORR_BAR and (np.isnan(floor) or c < floor))})

floors = pd.DataFrame(floor_rows)
decorr = pd.DataFrame(decorr_rows)
floors.to_csv(OUT / "plain_seed_floor.csv", index=False)
decorr.to_csv(OUT / "decorrelation.csv", index=False)

print("PLAIN-vs-PLAIN across-repeat residual correlation (the floor):")
print(floors.round(4).to_string(index=False))
print(f"\\nerror correlation against PLAIN, per arm per isoform (bar {DECORR_BAR}, "
      f"notebook 24 band {NB24_BAND_LO:.3f}-{NB24_BAND_HI:.3f}):")
print(decorr.pivot(index="arm", columns="isoform", values="corr_vs_plain").round(3).to_string())

PLAIN-vs-PLAIN across-repeat residual correlation (the floor):
isoform  n_repeat_pairs  plain_vs_plain_mean  plain_vs_plain_min  plain_vs_plain_max
 CYP1A2               3               0.9246              0.9144              0.9299
 CYP2C9               3               0.8998              0.8957              0.9041
 CYP2D6               3               0.9182              0.9098              0.9288
 CYP3A4               3               0.8842              0.8736              0.8953
\nerror correlation against PLAIN, per arm per isoform (bar 0.85, notebook 24 band 0.792-0.908):
isoform                 CYP1A2  CYP2C9  CYP2D6  CYP3A4
arm                                                   
aid                      0.889   0.865   0.921   0.837
deadzone                 0.944   0.933   0.941   0.915
ecfp4_narrow__lightgbm   0.869   0.792   0.887   0.770
ecfp4_narrow__rf         0.868   0.800   0.881   0.749
ecfp4_narrow__xgboost    0.845   0.785   0.867   0.746
mae_only                 0.944

In [65]:
# ---- The plain verdict, per arm, in words. Stated for every arm, not only the interesting ones.
print("=== DECORRELATION VERDICT, per arm ===")
verdict_lines = []
for arm in [a for a in ARMS_PRESENT if a != "plain"]:
    sub = decorr[decorr["arm"] == arm]
    if not len(sub):
        continue
    lo, hi = sub["corr_vs_plain"].min(), sub["corr_vs_plain"].max()
    n_below_bar = int(sub["below_bar_0.85"].sum())
    n_below_floor = int(sub["below_floor"].eq(True).sum())
    n_below_band = int(sub["below_nb24_band"].sum())
    # Two independent references, reported separately rather than collapsed into one label:
    # the fixed 0.85 bar the brief names, and PLAIN's own across-repeat re-seed floor (the
    # correlation two runs of the SAME recipe produce). A floor of NaN means fewer than two
    # repeats completed, in which case only the bar can be judged and that is said explicitly.
    floor_known = bool(sub["plain_floor"].notna().all())
    detail = (f"below the {DECORR_BAR} bar on {n_below_bar}/{len(sub)} isoforms; below PLAIN's own "
              f"re-seed floor on {n_below_floor}/{len(sub)}"
              if floor_known else
              f"below the {DECORR_BAR} bar on {n_below_bar}/{len(sub)} isoforms; PLAIN's re-seed "
              f"floor is unavailable (fewer than two repeats), so only the bar is judged")
    if n_below_bar == len(sub) and (not floor_known or n_below_floor == len(sub)):
        v = f"DECORRELATED from PLAIN on every isoform -- {detail}"
    elif n_below_bar == 0:
        v = f"NOT decorrelated, pool-typical or above on every isoform -- {detail}"
    else:
        v = f"PARTIALLY decorrelated -- {detail}"
    line = (f"  {arm:24s} corr vs PLAIN {lo:.3f}-{hi:.3f}  "
            f"below {DECORR_BAR}: {n_below_bar}/{len(sub)}  below own floor: {n_below_floor}/{len(sub)}  "
            f"below nb24 band: {n_below_band}/{len(sub)}\\n    -> {v}")
    print(line)
    verdict_lines.append({"arm": arm, "corr_min": float(lo), "corr_max": float(hi),
                          "n_below_bar": n_below_bar, "n_isoforms": len(sub),
                          "n_below_floor": n_below_floor, "n_below_nb24_band": n_below_band,
                          "verdict": v})
decorr_verdict = pd.DataFrame(verdict_lines)
decorr_verdict.to_csv(OUT / "decorrelation_verdict.csv", index=False)

MATERIALLY_DECORRELATED = sorted(decorr_verdict.loc[
    decorr_verdict["n_below_bar"] >= 1, "arm"].tolist()) if len(decorr_verdict) else []
TREE_DECORRELATED = [a for a in MATERIALLY_DECORRELATED if a in TREE_ARMS]
print(f"\\narms below the {DECORR_BAR} bar on at least one isoform: "
      f"{MATERIALLY_DECORRELATED if MATERIALLY_DECORRELATED else 'NONE'}")
if TREE_DECORRELATED:
    print(f"\\nTREES below the bar on at least one isoform ({TREE_DECORRELATED}) -- on the brief's")
    print("own stated reading, this project has its first genuine ensemble candidate. The blend in")
    print("4.4 therefore runs, with its nested honesty check.")
else:
    print("\\nNo tree arm falls below the bar on any isoform. On the brief's own stated reading this")
    print("is equally informative: the shared ignorance is a property of the COMPOUNDS rather than")
    print("of the architecture, and the ensemble question closes on evidence rather than on cost.")

=== DECORRELATION VERDICT, per arm ===
  ecfp4_narrow__rf         corr vs PLAIN 0.749-0.881  below 0.85: 2/4  below own floor: 4/4  below nb24 band: 1/4\n    -> PARTIALLY decorrelated -- below the 0.85 bar on 2/4 isoforms; below PLAIN's own re-seed floor on 4/4
  ecfp4_narrow__xgboost    corr vs PLAIN 0.746-0.867  below 0.85: 3/4  below own floor: 4/4  below nb24 band: 2/4\n    -> PARTIALLY decorrelated -- below the 0.85 bar on 3/4 isoforms; below PLAIN's own re-seed floor on 4/4
  ecfp4_narrow__lightgbm   corr vs PLAIN 0.770-0.887  below 0.85: 2/4  below own floor: 4/4  below nb24 band: 1/4\n    -> PARTIALLY decorrelated -- below the 0.85 bar on 2/4 isoforms; below PLAIN's own re-seed floor on 4/4
  mordred_pca__rf          corr vs PLAIN 0.721-0.887  below 0.85: 3/4  below own floor: 4/4  below nb24 band: 2/4\n    -> PARTIALLY decorrelated -- below the 0.85 bar on 3/4 isoforms; below PLAIN's own re-seed floor on 4/4
  mordred_pca__xgboost     corr vs PLAIN 0.696-0.845  below 0.85: 4/4

### 4.4 Blend — run only if at least one arm is materially decorrelated from PLAIN

Per-isoform weights by **greedy selection with replacement in z-space** (Caruana et al. 2004's
selection mechanic, applied here to this partition's own out-of-fold predictions). The in-sample
figure is reported **together with** a nested honesty check: per-fold greedy selection scored on the
held-out fold. Without the nested number a per-isoform blend is just another selection that cannot
be defended, so if the nested check cannot be computed the blend is reported as **unusable** rather
than reported at its in-sample figure alone.

In [66]:
def strae(y_true, y_pred, lo, hi):
    return float(rae_soft_threshold_absolute_error(np.asarray(y_true, float), np.asarray(y_pred, float),
                                                  y_true_upper=np.asarray(hi, float),
                                                  y_true_lower=np.asarray(lo, float)))


# TWO PARAMETERISATIONS, and the reason both are reported rather than one.
#
# The brief specifies weights fitted "in z-space". Taken literally -- standardise each arm,
# average the selected z columns, then map back onto y_true's own mean and standard deviation --
# that map-back forces the blend's spread to equal the LABEL spread. This project has already
# measured, twice and independently, that matching label spread is not ST-RAE-optimal:
# notebook 26's spread sweep found the ST-RAE-optimal multiplier strictly between rho and 1.0 on
# every isoform, and notebook 27 Section 1 found board ST-RAE associated with spread ratio at
# Spearman -0.822. So the z-space parameterisation bakes in a variance expansion that ST-RAE
# penalises, and its in-sample score can come out WORSE than the best single arm -- which a
# greedy search initialised at the best single arm cannot do on its own objective. That is a
# property of the parameterisation, not a bug in the search, and it was caught by exactly that
# inconsistency during a dry run of this analysis.
#
# `space="raw"` is therefore reported as the primary blend: greedy selection with replacement on
# a weighted mean of the raw predictions, which is on the same scale as every single-arm score
# and is guaranteed not to be worse than the best single arm in sample. `space="z"` is reported
# beside it, as specified, so the cost of the parameterisation is visible rather than hidden.
def greedy_blend(df, arm_cols, space="raw", max_iter=50):
    y = df["y_true"].to_numpy(float)
    lo, hi = df["conf_low"].to_numpy(float), df["conf_high"].to_numpy(float)
    ymu, ysd = y.mean(), y.std(ddof=0)
    if space == "raw":
        P = {a: df[a].to_numpy(float) for a in arm_cols}
        back = lambda v: v
    elif space == "z":
        P = {}
        for a in arm_cols:
            v = df[a].to_numpy(float)
            sd = v.std(ddof=0)
            P[a] = (v - v.mean()) / sd if sd > 0 else np.zeros_like(v)
        back = lambda z: z * ysd + ymu
    else:
        raise ValueError(f"unknown space: {space}")

    def score_of(counts):
        tot = sum(counts.values())
        v = sum(P[a] * n for a, n in counts.items()) / tot
        return strae(y, back(v), lo, hi)

    best_single = min(arm_cols, key=lambda a: score_of({a: 1}))
    counts = {best_single: 1}
    best = score_of(counts)
    for _ in range(max_iter - 1):
        cand_best, cand_score = None, best
        for a in arm_cols:
            trial = dict(counts)
            trial[a] = trial.get(a, 0) + 1
            sc = score_of(trial)
            if sc < cand_score - 1e-12:
                cand_best, cand_score = a, sc
        if cand_best is None:
            break
        counts[cand_best] = counts.get(cand_best, 0) + 1
        best = cand_score
    tot = sum(counts.values())
    return {a: n / tot for a, n in counts.items()}, best, best_single


def apply_blend(fit_df, pred_df, weights, space="raw"):
    # Every statistic used to transform the held-out fold comes from the FITTING set only.
    if space == "raw":
        out = np.zeros(len(pred_df))
        for a, wt in weights.items():
            out = out + wt * pred_df[a].to_numpy(float)
        return out
    yf = fit_df["y_true"].to_numpy(float)
    ymu, ysd = yf.mean(), yf.std(ddof=0)
    z = np.zeros(len(pred_df))
    for a, wt in weights.items():
        vf = fit_df[a].to_numpy(float)
        sd = vf.std(ddof=0)
        zz = (pred_df[a].to_numpy(float) - vf.mean()) / sd if sd > 0 else np.zeros(len(pred_df))
        z = z + wt * zz
    return z * ysd + ymu

In [67]:
BLEND_RAN = False
blend_rows = []
if not MATERIALLY_DECORRELATED:
    print("BLEND NOT RUN -- no arm is materially decorrelated from PLAIN on any isoform, which is")
    print("this section's own stated precondition. Fitting blend weights over a pool whose members")
    print("all make the same errors would produce a number with no mechanism behind it.")
else:
    BLEND_RAN = True
    print(f"BLEND RUNS -- materially decorrelated arms present: {MATERIALLY_DECORRELATED}")
    print("")
    for iso in ISOFORMS:
        w = OOF_WIDE[iso]
        arm_cols = [c for c in ARMS_PRESENT if c in w.columns and w[c].notna().all()]
        if len(arm_cols) < 2:
            print(f"{iso}: fewer than 2 arms with complete coverage -- skipped")
            continue
        blend_units = sorted(set(map(tuple, w[["repeat", "fold"]].values.tolist())))
        row = {"isoform": iso, "n_arms": len(arm_cols), "n_units": len(blend_units)}
        for space in ["raw", "z"]:
            weights, in_score, init_arm = greedy_blend(w, arm_cols, space=space)
            nested_pred, nested_true, nested_lo, nested_hi = [], [], [], []
            for (r, f) in blend_units:
                held = (w["repeat"] == r) & (w["fold"] == f)
                fit_df, pred_df = w[~held], w[held]
                if len(fit_df) < 50 or not len(pred_df):
                    continue
                wts, _, _ = greedy_blend(fit_df, arm_cols, space=space)
                nested_pred.append(apply_blend(fit_df, pred_df, wts, space=space))
                nested_true.append(pred_df["y_true"].to_numpy(float))
                nested_lo.append(pred_df["conf_low"].to_numpy(float))
                nested_hi.append(pred_df["conf_high"].to_numpy(float))
            if nested_pred:
                nested = strae(np.concatenate(nested_true), np.concatenate(nested_pred),
                               np.concatenate(nested_lo), np.concatenate(nested_hi))
                usable = True
            else:
                nested, usable = np.nan, False
            row[f"{space}_in_sample"] = in_score
            row[f"{space}_nested"] = nested
            row[f"{space}_usable"] = usable
            row[f"{space}_n_nested_folds"] = len(nested_pred)
            row[f"{space}_init_arm"] = init_arm
            row[f"{space}_weights"] = json.dumps({k: round(v, 4) for k, v in
                                                  sorted(weights.items(), key=lambda kv: -kv[1])})
        best_arm = min(arm_cols, key=lambda a: strae(w["y_true"], w[a], w["conf_low"], w["conf_high"]))
        row["best_single_arm"] = best_arm
        row["best_single_strae"] = strae(w["y_true"], w[best_arm], w["conf_low"], w["conf_high"])
        row["plain_strae"] = (strae(w["y_true"], w["plain"], w["conf_low"], w["conf_high"])
                              if "plain" in arm_cols else np.nan)
        blend_rows.append(row)
        print(f"{iso}  ({len(blend_units)} folds, {len(arm_cols)} arms)")
        print(f"   PLAIN {row['plain_strae']:.4f} | best single ({best_arm}) {row['best_single_strae']:.4f}")
        print(f"   raw-space: in-sample {row['raw_in_sample']:.4f} -> NESTED {row['raw_nested']:.4f}")
        print(f"     weights {row['raw_weights']}")
        print(f"   z-space:   in-sample {row['z_in_sample']:.4f} -> NESTED {row['z_nested']:.4f}")
        print(f"     weights {row['z_weights']}")

if blend_rows:
    blend = pd.DataFrame(blend_rows)
    blend.to_csv(OUT / "blend_results.csv", index=False)
    print("")
    print("The NESTED figure is the only defensible one. The in-sample figure sits beside it so the")
    print("gap between them -- the optimism a per-isoform selection buys itself -- stays visible.")
    print("")
    for _, r_ in blend.iterrows():
        # Sanity property of the search, asserted rather than assumed: greedy selection
        # initialised at the best single arm cannot be worse than it in sample, on its own
        # objective. Raw space shares ST-RAE's scale so this must hold there.
        if r_["raw_in_sample"] > r_["best_single_strae"] + 1e-9:
            print(f"  {r_['isoform']}: WARNING -- raw-space in-sample ({r_['raw_in_sample']:.4f}) "
                  f"exceeds the best single arm ({r_['best_single_strae']:.4f}); the search or the "
                  f"objective is inconsistent. Investigate before quoting this row.")
        for space in ["raw", "z"]:
            if not r_[f"{space}_usable"]:
                print(f"  {r_['isoform']} ({space}): nested check could not be computed -- UNUSABLE.")
                continue
            beats = r_[f"{space}_nested"] < r_["plain_strae"]
            print(f"  {r_['isoform']} ({space}): nested blend {'BEATS' if beats else 'does NOT beat'} "
                  f"PLAIN ({r_[f'{space}_nested']:.4f} vs {r_['plain_strae']:.4f}); optimism "
                  f"{r_[f'{space}_nested'] - r_[f'{space}_in_sample']:+.4f}")
    print("")
    z_worse = int((blend["z_in_sample"] > blend["best_single_strae"] + 1e-9).sum())
    print(f"z-space in-sample worse than the best single arm on {z_worse} of {len(blend)} isoforms.")
    if z_worse:
        print("This is the parameterisation cost described above, not a search failure: mapping the")
        print("blended z back onto the LABEL spread expands variance, and ST-RAE penalises that --")
        print("consistent with notebook 26's spread sweep and notebook 27 Section 1's own findings.")
        print("The raw-space blend is the one to read.")

BLEND RUNS -- materially decorrelated arms present: ['aid', 'ecfp4_narrow__lightgbm', 'ecfp4_narrow__rf', 'ecfp4_narrow__xgboost', 'mordred_pca__lightgbm', 'mordred_pca__rf', 'mordred_pca__xgboost']



CYP1A2  (15 folds, 11 arms)
   PLAIN 0.8494 | best single (deadzone) 0.7680
   raw-space: in-sample 0.7361 -> NESTED 0.7379
     weights {"deadzone": 0.5, "aid": 0.3333, "ecfp4_narrow__xgboost": 0.1667}
   z-space:   in-sample 0.8824 -> NESTED 0.8882
     weights {"aid": 0.3333, "deadzone": 0.3333, "ecfp4_narrow__xgboost": 0.1667, "mordred_pca__xgboost": 0.1667}


CYP2C9  (15 folds, 11 arms)
   PLAIN 0.6520 | best single (deadzone) 0.5644
   raw-space: in-sample 0.5479 -> NESTED 0.5485
     weights {"deadzone": 0.6667, "aid": 0.3333}
   z-space:   in-sample 0.6240 -> NESTED 0.6273
     weights {"deadzone": 0.3, "aid": 0.3, "ecfp4_narrow__rf": 0.1, "mordred_pca__xgboost": 0.1, "ecfp4_narrow__xgboost": 0.1, "mae_only": 0.1}


CYP2D6  (15 folds, 11 arms)
   PLAIN 1.0035 | best single (deadzone) 0.8743
   raw-space: in-sample 0.8743 -> NESTED 0.8740
     weights {"deadzone": 1.0}
   z-space:   in-sample 1.0799 -> NESTED 1.0849
     weights {"deadzone": 0.2857, "ecfp4_narrow__xgboost": 0.1429, "mordred_pca__lightgbm": 0.1429, "aid": 0.1429, "mordred_pca__xgboost": 0.1429, "ecfp4_narrow__lightgbm": 0.1429}


CYP3A4  (15 folds, 11 arms)
   PLAIN 0.5422 | best single (deadzone) 0.4798
   raw-space: in-sample 0.4583 -> NESTED 0.4594
     weights {"deadzone": 0.6, "aid": 0.2, "ecfp4_narrow__lightgbm": 0.2}
   z-space:   in-sample 0.5056 -> NESTED 0.5109
     weights {"deadzone": 0.25, "aid": 0.25, "ecfp4_narrow__rf": 0.25, "single": 0.25}

The NESTED figure is the only defensible one. The in-sample figure sits beside it so the
gap between them -- the optimism a per-isoform selection buys itself -- stays visible.

  CYP1A2 (raw): nested blend BEATS PLAIN (0.7379 vs 0.8494); optimism +0.0018
  CYP1A2 (z): nested blend does NOT beat PLAIN (0.8882 vs 0.8494); optimism +0.0058
  CYP2C9 (raw): nested blend BEATS PLAIN (0.5485 vs 0.6520); optimism +0.0006
  CYP2C9 (z): nested blend BEATS PLAIN (0.6273 vs 0.6520); optimism +0.0033
  CYP2D6 (raw): nested blend BEATS PLAIN (0.8740 vs 1.0035); optimism -0.0003
  CYP2D6 (z): nested blend does NOT beat PLAIN (1.0849 vs 1.0035); optimism +0.0050
  CYP3A4 (r

## Part 5 — Figures

House style, matching notebooks 21/24/25/26/27/29/31/33: `figsize=(10,6)`, `dpi=150`,
`bbox_inches="tight"`, base font size 10, top and right spines removed, legends without frames,
titles 11pt bold and left-aligned **stating the finding**, axis labels carrying units, `#8C8C8C`
grey for the reference series and `#B0413E` for the series carrying the argument. Each figure's
underlying table is written to CSV alongside it.

In [68]:
plt.rcParams.update({"font.size": 10, "figure.dpi": 110, "savefig.dpi": 150,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
                     "legend.frameon": False})
GREY, ACCENT = "#8C8C8C", "#B0413E"
MID = "#4C4C4C"


def finish(fig, path):
    fig.savefig(path, bbox_inches="tight", dpi=150)
    plt.close(fig)
    print(f"wrote {path}")

In [69]:
# ---- FIGURE 1: out-of-fold ST-RAE by arm and isoform, with across-fold spread.
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=False)
arm_plot_order = (arm_summary.groupby("arm")["ST-RAE_mean"].mean().sort_values(ascending=False).index.tolist())
for ax, iso in zip(axes.ravel(), ISOFORMS):
    sub = arm_summary[arm_summary["isoform"] == iso].set_index("arm").reindex(arm_plot_order).dropna(subset=["ST-RAE_mean"])
    y = np.arange(len(sub))
    colors = [ACCENT if a == "plain" else GREY for a in sub.index]
    ax.barh(y, sub["ST-RAE_mean"], xerr=sub["ST-RAE_std"].fillna(0), color=colors,
            error_kw={"ecolor": MID, "elinewidth": 1, "capsize": 2}, height=0.72)
    ax.set_yticks(y)
    ax.set_yticklabels(sub.index, fontsize=7.5)
    ax.axvline(1.0, color=MID, lw=0.8, ls=":")
    plain_v = arm_summary[(arm_summary["isoform"] == iso) & (arm_summary["arm"] == "plain")]["ST-RAE_mean"]
    if len(plain_v):
        ax.axvline(float(plain_v.iloc[0]), color=ACCENT, lw=0.9, ls="--", alpha=0.7)
    n = int(sub["n_folds"].max())
    ax.set_title(f"{iso} (n={n} folds)", fontsize=10)
    ax.set_xlabel("ST-RAE, lower is better")
fig.suptitle("Cluster-disjoint out-of-fold ST-RAE: no arm separates from PLAIN by more than its own fold spread"
             if (paired[(paired.metric == 'ST-RAE') & (paired.verdict != 'NOT DIFFERENT')].empty)
             else "Cluster-disjoint out-of-fold ST-RAE by arm (dashed red = PLAIN; dotted = naive-mean parity)",
             fontsize=11, fontweight="bold", x=0.01, ha="left")
fig.tight_layout(rect=[0, 0, 1, 0.94])
finish(fig, FIG_OUT / "strae_by_arm_isoform.png")
arm_summary.to_csv(OUT / "figure1_strae_by_arm_isoform.csv", index=False)

wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/35_butina_repeat3/figures/strae_by_arm_isoform.png


**Figure 1 — `strae_by_arm_isoform.png`.** Out-of-fold ST-RAE for every arm at **15 folds**, one panel per isoform, bars ordered worst-to-best by cross-isoform mean
and error bars showing the across-fold standard deviation. PLAIN, the reference every other arm is
compared against, is in red with its level marked as a dashed line; the dotted line at 1.0 is parity
with a constant mean predictor. Panel titles carry the number of completed folds behind each
estimate, so the reader can see directly how much the spread is worth. Read against Figure 2: this
panel says where each arm *sits*, while Figure 2 says whether that position survives a change of
split scheme.

In [70]:
# ---- FIGURE 2: each arm's difference against PLAIN under BOTH partitions, paired.
# The figure that answers 4.2 -- a sign flip is visible as a pair straddling zero.
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
plot_arms = [a for a in ALL_ARMS if a != "plain" and a in ARMS_PRESENT]
for ax, iso in zip(axes.ravel(), ISOFORMS):
    sub = agree[agree["isoform"] == iso].set_index("arm").reindex(plot_arms)
    y = np.arange(len(sub))
    for yi, (arm, row) in zip(y, sub.iterrows()):
        bd, rd, st = row["butina_diff"], row["random_diff"], row.get("status")
        if pd.notna(bd) and pd.notna(rd):
            flip = st == "FLIP"
            unres = st == "sign not resolved"
            col = ACCENT if flip else GREY
            ax.plot([rd, bd], [yi, yi], color=col, lw=1.6 if flip else 1.0,
                    alpha=0.9 if flip else (0.3 if unres else 0.6),
                    ls=":" if unres else "-", zorder=1)
            ax.scatter([rd], [yi], marker="o", s=26, facecolor="white", edgecolor=GREY,
                       linewidth=1.2, alpha=0.4 if unres else 1.0, zorder=3)
            ax.scatter([bd], [yi], marker="s", s=28, color=(ACCENT if flip else MID),
                       alpha=0.4 if unres else 1.0, zorder=3)
        elif pd.notna(bd):
            ax.scatter([bd], [yi], marker="s", s=28, facecolor="white", edgecolor=MID,
                       linewidth=1.0, zorder=3)
    ax.axvline(0.0, color="black", lw=0.9)
    ax.set_yticks(y)
    ax.set_yticklabels(sub.index, fontsize=7.5)
    ax.invert_yaxis()
    n_flip = int((sub["status"] == "FLIP").sum())
    n_res = int((sub["status"].isin(["FLIP", "agrees"])).sum())
    n_unres = int((sub["status"] == "sign not resolved").sum())
    ax.set_title(f"{iso} -- {n_flip} of {n_res} sign-resolved arms flip ({n_unres} unresolved)",
                 fontsize=9.5)
for ax in axes[1]:
    ax.set_xlabel("ST-RAE difference against PLAIN (negative = better than PLAIN)")
h = [plt.Line2D([], [], marker="o", ls="", markerfacecolor="white", markeredgecolor=GREY, label="random partition"),
     plt.Line2D([], [], marker="s", ls="", color=MID, label="cluster-disjoint (this notebook)"),
     plt.Line2D([], [], color=ACCENT, lw=1.6, label="sign flips between partitions"),
     plt.Line2D([], [], color=GREY, lw=1.0, ls=":", alpha=0.4, label="sign not resolved (|diff| < its SE)"),
     plt.Line2D([], [], marker="s", ls="", markerfacecolor="white", markeredgecolor=MID,
                label="no random-partition reference")]
fig.suptitle("Does the split scheme change the conclusion? Paired arm-vs-PLAIN differences under both partitions",
             fontsize=11, fontweight="bold", x=0.01, ha="left")
fig.tight_layout(rect=[0, 0.07, 1, 0.94])
# Legend below the panels -- inside any panel it overlapped a real data marker.
fig.legend(handles=h, loc="lower center", ncol=3, fontsize=7.5, bbox_to_anchor=(0.5, 0.0))
finish(fig, FIG_OUT / "partition_agreement.png")
agree.to_csv(OUT / "figure2_partition_agreement.csv", index=False)

wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/35_butina_repeat3/figures/partition_agreement.png


**Figure 2 — `partition_agreement.png`.** For each isoform, every arm's out-of-fold ST-RAE
difference against PLAIN measured twice: once on the frozen random partition (open grey circle,
read from notebooks 05/29/31) and once on this notebook's repeated cluster-disjoint partition
(filled square), the two joined so the reader sees the pair rather than two separate numbers. The
vertical line at zero is PLAIN; anything to its left beat PLAIN on that partition. **A pair that
straddles the zero line is a sign flip** — the two partitions disagree about whether the arm helps
at all — and those pairs are drawn in red with a heavier connector. SINGLE has no random-partition
counterpart anywhere in this project and is drawn as a single square with that stated on the figure
rather than silently compared against something else. This is the figure that answers Part 4.2.

In [71]:
# ---- FIGURE 3: pairwise arm error-correlation matrix per isoform, with notebook 24's band
# marked on the colourbar and the PLAIN across-repeat floor annotated in each panel.
# Colour range starts at 0.6, not 0: every value in this matrix lies between ~0.65 and 1.0, and
# a 0-1 scale renders the whole thing one shade of red, hiding exactly the differences the
# figure exists to show. Stated on the colourbar so the truncation is not silent.
fig, axes = plt.subplots(2, 2, figsize=(10, 6.6))
VMIN, VMAX = 0.6, 1.0
im = None
for k, (ax, iso) in enumerate(zip(axes.ravel(), ISOFORMS)):
    M = corr_mats[iso]
    cols = [a for a in ALL_ARMS if a in M.columns]
    Mo = M.loc[cols, cols].astype(float)
    im = ax.imshow(Mo.to_numpy(), vmin=VMIN, vmax=VMAX, cmap="RdYlBu_r", aspect="auto")
    ax.set_xticks(range(len(cols)))
    # Only the bottom row carries x tick labels -- on the top row they collided with the
    # bottom row's panel titles.
    if k >= 2:
        ax.set_xticklabels(cols, rotation=90, fontsize=5.5)
    else:
        ax.set_xticklabels([])
    ax.set_yticks(range(len(cols)))
    ax.set_yticklabels(cols, fontsize=5.5)
    fl = floors[floors["isoform"] == iso]
    flv = float(fl["plain_vs_plain_mean"].iloc[0]) if len(fl) else np.nan
    ax.set_title(f"{iso} -- PLAIN across-repeat floor {flv:.3f}" if not np.isnan(flv)
                 else f"{iso} -- floor unavailable (one repeat)", fontsize=9.5)
    if "plain" in cols:
        j = cols.index("plain")
        ax.axvline(j, color=ACCENT, lw=1.4, alpha=0.95)
        ax.axhline(j, color=ACCENT, lw=1.4, alpha=0.95)
fig.tight_layout(rect=[0, 0, 0.88, 0.93])
cax = fig.add_axes([0.90, 0.12, 0.022, 0.72])
cb = fig.colorbar(im, cax=cax)
# Band edges become real ticks rather than free-floating text over the colourbar label.
ticks = sorted({0.6, 0.7, 0.8, 0.9, 1.0, round(NB24_BAND_LO, 3), round(NB24_BAND_HI, 3)})
cb.set_ticks(ticks)
cb.set_ticklabels([(f"{t:.2f}  nb24" if abs(t - NB24_BAND_LO) < 1e-6 or abs(t - NB24_BAND_HI) < 1e-6
                    else f"{t:.2f}") for t in ticks])
cb.ax.tick_params(labelsize=7)
for v in [NB24_BAND_LO, NB24_BAND_HI]:
    cb.ax.axhline(v, color="black", lw=1.4)
cb.set_label("error correlation (residuals, labelled rows only); scale truncated at 0.6",
             fontsize=7.5, labelpad=6)
fig.suptitle("Error correlation between arms: the shared-ignorance question, with notebook 24's pool band marked",
             fontsize=11, fontweight="bold", x=0.01, ha="left")
finish(fig, FIG_OUT / "error_correlation_matrix.png")
pd.concat([corr_mats[iso].assign(isoform=iso) for iso in ISOFORMS]).to_csv(
    OUT / "figure3_error_correlation_matrix.csv")

wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/35_butina_repeat3/figures/error_correlation_matrix.png


**Figure 3 — `error_correlation_matrix.png`.** Pairwise correlation of **residuals** (prediction
minus truth, labelled rows only) between every pair of arms, one panel per isoform, pooled across
all completed folds. The two black lines on the shared colourbar mark notebook 24's measured band
for this project's existing 11-config pool (0.79–0.91), so a pair can be read immediately as
"pool-typical" or "unusually decorrelated". The red cross-hairs pick out PLAIN's row and column —
the comparison that decides whether an arm is an ensemble candidate — and each panel title carries
that isoform's PLAIN-against-PLAIN across-repeat floor, the correlation two runs of the *same*
recipe produce under different cluster-to-fold assignments and seeds. An arm at or above its own
panel's floor is not decorrelated in any useful sense, whatever its absolute value. This is the
figure that answers Part 4.3.

In [72]:
# ---- FIGURE 4: the same arm measured at 10 folds and at 15, paired. This is the figure that
# answers "did adding a repeat move anything?" -- if every pair is a short horizontal segment,
# the extra data tightened the estimate without changing the picture.
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=False)
plot_arms = [a for a in ALL_ARMS if a in set(cmp["arm"])]
for ax, iso in zip(axes.ravel(), ISOFORMS):
    sub = cmp[cmp["isoform"] == iso].set_index("arm").reindex(plot_arms)
    y = np.arange(len(sub))
    for yi, (arm, row) in zip(y, sub.iterrows()):
        m10, m15 = row["ST-RAE_mean_10"], row["ST-RAE_mean_15"]
        if pd.notna(m10) and pd.notna(m15):
            ax.plot([m10, m15], [yi, yi], color=GREY, lw=1.0, alpha=0.7, zorder=1)
            ax.scatter([m10], [yi], marker="o", s=26, facecolor="white", edgecolor=GREY,
                       linewidth=1.2, zorder=3)
            ax.scatter([m15], [yi], marker="s", s=28, color=ACCENT, zorder=3)
        elif pd.notna(m15):
            ax.scatter([m15], [yi], marker="s", s=28, color=ACCENT, zorder=3)
    ax.set_yticks(y)
    ax.set_yticklabels(sub.index, fontsize=7.5)
    ax.invert_yaxis()
    d = sub["delta"].abs().max()
    ax.set_title(f"{iso} -- largest shift {d:.4f} ST-RAE", fontsize=10)
for ax in axes[1]:
    ax.set_xlabel("out-of-fold ST-RAE, lower is better")
h4 = [plt.Line2D([], [], marker="o", ls="", markerfacecolor="white", markeredgecolor=GREY,
                 label="10 folds (notebook 34)"),
      plt.Line2D([], [], marker="s", ls="", color=ACCENT, label="15 folds (this notebook)")]
fig.suptitle("Adding a third repeat: every arm measured at 10 folds and at 15",
             fontsize=11, fontweight="bold", x=0.01, ha="left")
fig.tight_layout(rect=[0, 0.06, 1, 0.94])
fig.legend(handles=h4, loc="lower center", ncol=2, fontsize=8, bbox_to_anchor=(0.5, 0.0))
finish(fig, FIG_OUT / "folds_10_vs_15.png")
cmp.to_csv(OUT / "figure4_folds_10_vs_15.csv", index=False)

wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/35_butina_repeat3/figures/folds_10_vs_15.png


**Figure 4 — `folds_10_vs_15.png`.** Every arm's out-of-fold ST-RAE measured twice on the same
partition: once on notebook `34`'s two repeats (open grey circle) and once with the third repeat
added (filled red square), joined so the pair reads as one object. Panel titles give that isoform's
largest shift across all arms. This is the figure that answers whether adding a repeat was a
selection: short segments mean the extra 5 folds tightened the estimates without moving the
picture, and any arm that moved far enough to change its standing against PLAIN would show here as
a long segment crossing its neighbours. MAE_ONLY appears with both markers because it was run on
all three repeats, but its 10-fold point is this notebook's own data, not a reproduction of
notebook `34`.

## Part 6 — Verdict

In [73]:
print("=" * 78)
print("NOTEBOOK 35 -- VERDICT")
print("=" * 78)
_units_done = pd.read_csv(RUN_SCORES_PATH)[["repeat", "fold", "arm"]].drop_duplicates()
print(f"\nTRAINED HERE: {len(_units_done)} (repeat, fold, arm) units")
print(f"  repeat {TARGET_REPEAT} of the main design + MAE_ONLY on repeats {MAE_ONLY_REPEATS}")
print(f"REUSED READ-ONLY from notebook 34: {len(k34)} units (repeats {NB34_REPEATS})")
print(f"ANALYSED: {len(UNITS_15)} folds x {len(ARMS_PRESENT)} arms")
if ABORT["aborted"]:
    print(f"  *** RUN ABORTED at {ABORT['where']}: {ABORT['reason']}")
if DEADLINE_SKIPS:
    print(f"  *** {len(DEADLINE_SKIPS)} units skipped by the {WALL_DEADLINE_H}h wall guard")
print(f"  elapsed: {(time.time() - RUN_STARTED_AT)/3600:.2f} h since campaign start")

print("\n--- DID ADDING A REPEAT MOVE ANYTHING? ---")
# Count changes only over cells that HAVE both verdicts. An arm with no 10-fold row -- which
# happens if the wall guard skipped MAE_ONLY's repeats 0/1 -- would otherwise be scored as a
# "change" against a verdict that was never measured, making n_changed exceed n_comparable.
_both = vcmp["verdict_10"].notna() & vcmp["verdict_15"].notna()
n_changed = int((vcmp["verdict_changed"] & _both).sum())
n_comparable = int(_both.sum())
_no10 = sorted(set(vcmp.loc[~vcmp["verdict_10"].notna(), "arm"]))
if _no10:
    print(f"  arms with no 10-fold verdict (excluded from the change count): {_no10}")
print(f"  verdicts changed between 10 and 15 folds: {n_changed} of {n_comparable} comparable cells")
print(f"  largest shift in any arm's mean ST-RAE: {cmp['delta'].abs().max():.4f}")
if n_changed == 0:
    print("  -> Nothing moved. The third repeat tightened the estimates and left every verdict")
    print("     standing, which is what a uniform addition of data should do.")
else:
    print("  -> Changed cells are listed in verdict_10_vs_15.csv; each is reported with BOTH")
    print("     mean_diff columns so a power change is not mistaken for a sign change.")

print("\n--- 4.1 MODEL COMPARISON AT 15 FOLDS (ST-RAE, BH within isoform) ---")
for iso in ISOFORMS:
    sub = st15[st15["isoform"] == iso]
    better = sub[sub["verdict"] == "SIGNIFICANTLY BETTER"]["arm"].tolist()
    worse = sub[sub["verdict"] == "SIGNIFICANTLY WORSE"]["arm"].tolist()
    print(f"  {iso}: better than PLAIN {better if better else 'none'}; "
          f"worse {worse if worse else 'none'}")

print("\n--- 4.1b THE MAE_ONLY DECOMPOSITION ---")
for iso in ISOFORMS:
    r_ = decomp[(decomp["isoform"] == iso) & (decomp["fold_set"] == "15-fold")].iloc[0]
    if not r_["deadzone_beats_plain"]:
        print(f"  {iso}: DEADZONE does not beat PLAIN -- shares undefined")
        continue
    print(f"  {iso}: loss-alone {r_['loss_alone_share']:.0%} / clipping {r_['clipping_share']:.0%}")
print("  MAE_ONLY's own target is the literal label, so it carries none of DEADZONE's")
print("  fold-crosstalk exposure -- where the two agree, crosstalk is not what is driving it.")

print("\n--- 4.2 THE SPLIT QUESTION ---")
n_res_all = int(((agree["status"] == "agrees") | (agree["status"] == "FLIP")).sum())
print(f"  {int((agree['status'] == 'FLIP').sum())} of {n_res_all} sign-RESOLVED cells flip; "
      f"{int((agree['status'] == 'sign not resolved').sum())} unresolved; "
      f"{int((agree['status'] == 'no random reference').sum())} without a random reference.")
for _, r_ in ordering.iterrows():
    print(f"    {r_['isoform']}: ordering rho={r_['spearman_rho']:.3f} (p={r_['p']:.3f})")

print("\n--- 4.3 DECORRELATION (floor now on 3 repeat-pairs, not 1) ---")
print(floors[["isoform", "n_repeat_pairs", "plain_vs_plain_mean",
              "plain_vs_plain_min", "plain_vs_plain_max"]].round(4).to_string(index=False))
for _, r_ in decorr_verdict.iterrows():
    print(f"  {r_['arm']:24s} {r_['corr_min']:.3f}-{r_['corr_max']:.3f}  -> {r_['verdict']}")

print("\n--- 4.4 BLEND ---")
if not BLEND_RAN:
    print("  Not run -- its stated precondition (a materially decorrelated arm) was not met.")
elif blend_rows:
    for r_ in blend_rows:
        print(f"  {r_['isoform']}: PLAIN {r_['plain_strae']:.4f} | best single "
              f"({r_['best_single_arm']}) {r_['best_single_strae']:.4f} | "
              f"raw nested {r_['raw_nested']:.4f} | z nested {r_['z_nested']:.4f}")
    beats_raw = [r_["isoform"] for r_ in blend_rows
                 if r_["raw_usable"] and r_["raw_nested"] < r_["plain_strae"]]
    print(f"  Nested raw-space blend beats PLAIN on: {beats_raw if beats_raw else 'no isoform'}.")

print("\n--- COST ---")
_tm = pd.read_csv(TIMINGS_PATH)
_real = _tm[_tm["wall_min"] > 0.5]
if len(_real):
    print(_real.groupby("arm_kind")["wall_min"].agg(["mean", "min", "max", "size"]).round(2).to_string())
    print(f"  total real training wall time: {_real['wall_min'].sum()/60:.2f} h")
    _mpe = _real[_real["min_per_epoch"].notna()].groupby("arm_kind")["min_per_epoch"].agg(["min", "max"])
    if len(_mpe):
        _mpe["ratio"] = _mpe["max"] / _mpe["min"]
        print(f"  worst within-arm per-epoch ratio: {_mpe['ratio'].max():.2f}x "
              f"({'no degradation' if _mpe['ratio'].max() < 3 else 'DEGRADATION FLAGGED'})")

print("\n--- SCOPE ---")
print("  Notebook 34 not modified or re-executed; outputs/34_butina_5x5/ read-only.")
print("  No submission built, validated or sent. No full-data retrain. No adoption recommended.")
print("=" * 78)

NOTEBOOK 35 -- VERDICT

TRAINED HERE: 65 (repeat, fold, arm) units
  repeat 2 of the main design + MAE_ONLY on repeats [0, 1, 2]
REUSED READ-ONLY from notebook 34: 100 units (repeats [0, 1])
ANALYSED: 15 folds x 11 arms
  elapsed: 7.83 h since campaign start

--- DID ADDING A REPEAT MOVE ANYTHING? ---
  verdicts changed between 10 and 15 folds: 6 of 40 comparable cells
  largest shift in any arm's mean ST-RAE: 0.0126
  -> Changed cells are listed in verdict_10_vs_15.csv; each is reported with BOTH
     mean_diff columns so a power change is not mistaken for a sign change.

--- 4.1 MODEL COMPARISON AT 15 FOLDS (ST-RAE, BH within isoform) ---
  CYP1A2: better than PLAIN ['aid', 'deadzone', 'mae_only']; worse ['ecfp4_narrow__xgboost', 'mordred_pca__rf', 'mordred_pca__xgboost', 'mordred_pca__lightgbm']
  CYP2C9: better than PLAIN ['deadzone']; worse ['ecfp4_narrow__rf', 'ecfp4_narrow__xgboost', 'ecfp4_narrow__lightgbm', 'mordred_pca__rf', 'mordred_pca__xgboost', 'mordred_pca__lightgbm']
  

## Decisions taken without approval

This section exists whether or not it has content. Every entry is also written to
`outputs/35_butina_repeat3/autonomous_decisions.json`.

In [74]:
record_decision(
    decision="Ran a third cluster-disjoint repeat after notebook 34's timing gate had settled on two",
    reason=("Notebook 34's gate fired mechanically on AID's fold 0 (46.66 min), which proved that "
            "arm's slowest of ten against an eventual 34.82 min mean, projecting 31.69 h against a "
            "30 h threshold -- 5.6% over. The realised per-fold means give 26.77 h, which the same "
            "rule answers with three repeats. Adding a repeat is NOT re-deciding the gate: the "
            "fold assignment for repeat 2 was frozen in cv_folds_butina_5x5.csv before any result "
            "was seen (sha256 asserted identical here), and the extra folds are added UNIFORMLY to "
            "every arm, so no arm can be advantaged and there is no outcome being selected for. "
            "Re-running the gate itself on realised costs would be choosing a rule's input after "
            "seeing its output, which is the selection behaviour CLAUDE.md warns against and is "
            "deliberately not done. To make the distinction checkable rather than asserted, every "
            "analysis table reports the 10-fold and 15-fold results side by side, plus a dedicated "
            "figure and a verdict-change count."),
    alternatives=("Leave the design at 2 repeats (loses statistical power that was already paid "
                  "for and leaves the PLAIN re-seed floor resting on a single repeat-pair); or "
                  "re-run the gate on realised costs and let it authorise the third repeat "
                  "(indistinguishable in result, but is the selection behaviour being avoided)."),
    authority="explicit instruction in this notebook's task brief, with the reasoning recorded here as it requires")

record_decision(
    decision="Reused notebook 34's own assembled wide OOF tables for repeats 0/1 rather than re-deriving them from its per-run files",
    reason=("outputs/34_butina_5x5/oof_long/oof_long_{iso}.csv is tracked in git and is what "
            "notebook 34 actually reported from. Re-deriving the same table from its 100 per-run "
            "files could only reproduce it or silently differ; reusing it keeps this side of the "
            "15-fold join byte-identical to 34's published result. Repeat 2 and the MAE_ONLY "
            "columns are joined onto it, and the join is asserted not to change 34's row count."),
    alternatives="Re-derive everything from per-run files (more code, no benefit, and a new way to disagree with 34).",
    authority="autonomy rule: take the most conservative option and record it")

record_decision(
    decision=f"Set the wall-clock guard to {WALL_DEADLINE_H} h measured from this campaign's own start",
    reason=("Specified in the task brief. The guard measures from this notebook's own campaign.json, "
            "never notebook 34's -- 34's timestamp is from 26 Sep and inheriting it would put every "
            "unit past the deadline and skip the entire run. Ordering puts repeat 2's main design "
            "before MAE_ONLY so that, if the guard bites, it costs the decomposition arm rather "
            "than leaving the third repeat half-finished."),
    alternatives="No guard (an unlucky slowdown could leave the analysis sections never reached).",
    authority="task brief, plus the campaign-freshness constraint it states")

with open(OUT / "autonomous_decisions.json", "w") as fh:
    json.dump(AUTONOMOUS_DECISIONS, fh, indent=2)

print(f"{len(AUTONOMOUS_DECISIONS)} decision(s) recorded:\n")
for i, d in enumerate(AUTONOMOUS_DECISIONS, 1):
    print(f"{i}. {d['decision']}")
    print(f"   WHY: {d['reason']}")
    print(f"   ALTERNATIVES: {d['alternatives']}")
    print(f"   AUTHORITY: {d['authority']}\n")

3 decision(s) recorded:

1. Ran a third cluster-disjoint repeat after notebook 34's timing gate had settled on two
   WHY: Notebook 34's gate fired mechanically on AID's fold 0 (46.66 min), which proved that arm's slowest of ten against an eventual 34.82 min mean, projecting 31.69 h against a 30 h threshold -- 5.6% over. The realised per-fold means give 26.77 h, which the same rule answers with three repeats. Adding a repeat is NOT re-deciding the gate: the fold assignment for repeat 2 was frozen in cv_folds_butina_5x5.csv before any result was seen (sha256 asserted identical here), and the extra folds are added UNIFORMLY to every arm, so no arm can be advantaged and there is no outcome being selected for. Re-running the gate itself on realised costs would be choosing a rule's input after seeing its output, which is the selection behaviour CLAUDE.md warns against and is deliberately not done. To make the distinction checkable rather than asserted, every analysis table reports the 10-fo

## Scope check — fails loudly

Two independent tests, following notebook 30/34's fixed version rather than notebook 29's, whose
substring path matching without reading `git status` codes raised a false violation. **TEST 1**
looks only at *tracked* content changes under a protected path; **TEST 2** checks that nothing
inside a protected tree was *written* during this campaign, using this notebook's own
first-execution timestamp. `outputs/34_butina_5x5/` and `notebooks/34_butina_5x5_repeated_cv.ipynb`
are on the protected list: this notebook reads both and must not touch either. `src/` is protected
too — nothing here needed to modify it.

In [75]:
st_out = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                        cwd=REPO_ROOT).stdout
print("git status --porcelain:")
print(st_out if st_out.strip() else "(clean)")

PROTECTED = [
    "data/folds/cv_folds.csv", "data/folds/cv_folds_butina.csv",
    "data/folds/cv_folds_butina_5x5.csv",
    "data/processed/", "data/raw/", "src/",
    "notebooks/34_butina_5x5_repeated_cv.ipynb", "outputs/34_butina_5x5/",
    "outputs/05_cv_comparison/", "outputs/11_caruana_prep/", "outputs/11b_caruana_selection/",
    "outputs/24_nonnegative_stacking/", "outputs/27_calibration/", "outputs/27b_aid_cyp2d6_corrected/",
    "outputs/28_external_data/", "outputs/29_cv_confirmation/", "outputs/30_aid_full_retrain/",
    "outputs/31_deadzone/", "outputs/32_deadzone_aid_retrain/", "outputs/33_butina_split/",
    "outputs/board_solved_calibration/", "outputs/19_cyp2c9_revert/", "outputs/10c_control_submission/",
    "outputs/12_caruana_retrain_predict/",
]

tracked_violations = []
for line in st_out.splitlines():
    code_, path = line[:2], line[3:].strip()
    if any(x in path for x in PROTECTED) and code_.strip() != "??":
        tracked_violations.append(line)

written_during_run = []
for prot in PROTECTED:
    q = REPO_ROOT / prot
    if not q.exists():
        continue
    candidates = [q] if q.is_file() else [x for x in q.rglob("*") if x.is_file()]
    for fp in candidates:
        try:
            if fp.stat().st_mtime > RUN_STARTED_AT:
                written_during_run.append(str(fp.relative_to(REPO_ROOT)))
        except OSError:
            pass

print(f"\nTEST 1 -- tracked modifications under a protected path: {len(tracked_violations)}")
for t in tracked_violations:
    print(f"    VIOLATION: {t}")
print(f"TEST 2 -- files written inside a protected tree during this campaign: {len(written_during_run)}")
for t in written_during_run[:25]:
    print(f"    VIOLATION: {t}")

if tracked_violations or written_during_run:
    raise ValueError("a protected path was modified -- investigate before trusting anything above")
print("\nconfirmed: no protected path modified, by either test.")
print(f"  notebook 34's own outputs and notebook file are intact.")
print(f"\noutputs written by this notebook: {sorted(q.name for q in OUT.iterdir())}")
print(f"figures: {sorted(q.name for q in FIG_OUT.iterdir())}")

git status --porcelain:
?? notebooks/35_butina_repeat3.ipynb
?? outputs/35_butina_repeat3/


TEST 1 -- tracked modifications under a protected path: 0
TEST 2 -- files written inside a protected tree during this campaign: 0

confirmed: no protected path modified, by either test.
  notebook 34's own outputs and notebook file are intact.

outputs written by this notebook: ['aid_flip_reconciliation.csv', 'arm_summary_10_vs_15.csv', 'arm_summary_10fold.csv', 'arm_summary_15fold.csv', 'autonomous_decisions.json', 'blend_results.csv', 'campaign.json', 'chemprop_runs', 'curated_deadzone_target_r2.csv', 'deadzone_target_checks.csv', 'decorrelation.csv', 'decorrelation_verdict.csv', 'design.json', 'error_correlation_CYP1A2.csv', 'error_correlation_CYP2C9.csv', 'error_correlation_CYP2D6.csv', 'error_correlation_CYP3A4.csv', 'figure1_strae_by_arm_isoform.csv', 'figure2_partition_agreement.csv', 'figure3_error_correlation_matrix.csv', 'figure4_folds_10_vs_15.csv', 'figures', 'fold_isoform_label_cou